# NSE · Weekly Liquidity-Sweep Screener
### Swing entries where a *weekly* sweep of a swing low is **complete**

**Read this first — 60 seconds.**

| | |
|---|---|
| **What it finds** | NSE stocks whose latest **completed weekly candle** swept (raided) a swing low and **closed back above it with a proper rejection wick**. |
| **Whose lows** | Old *and* new. Confirmed fractal swing lows in the lookback, the rolling window low, and the 52-week low — a sweep candle may itself print the new low. |
| **Universe** | Every NSE equity with usable history, ranked, **top 1000 screened** (default 2 315 fetched first — nothing is silently dropped). |
| **Data** | Yahoo `v8/finance/chart`, **exactly 1 HTTP call per symbol**, shared global rate limiter, cache-first, automatic slow second pass for anything throttled. This is the part that normally breaks screeners — here it is engineered, see §10. |
| **Runtime** | First run ≈ 6–8 min (download 2 300 symbols). Every later run ≈ **5 seconds** from cache. |
| **Frequency** | Weekly. This is a *weekly* structure tool; run it after the Saturday close (or Friday ~15:45 IST). |

> ⚠️ **Not investment advice.** A screening tool that finds a liquidity-raid-and-reclaim pattern. It has no opinion on fundamentals, news, sector rotation or market regime. You own the risk.

---
## 0 · Run me first
Everything below is optional.

In [ ]:
# @title 0 · Setup (deps + self-contained engine)
import os, sys, json, math, time, warnings, importlib, importlib.util, datetime as _dt
warnings.filterwarnings("ignore")
os.chdir("/content" if os.path.isdir("/content") else os.getcwd())
PKG_DIR  = os.path.join(os.getcwd(), "wlsweep")
OUT_DIR  = os.path.join(os.getcwd(), "out")
os.makedirs(PKG_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
if PKG_DIR not in sys.path:
    sys.path.insert(0, PKG_DIR)

_PKGS = ["pandas", "numpy", "requests", "tqdm", "plotly", "matplotlib", "openpyxl"]

def _missing(pkgs):
    return [q for q in pkgs if importlib.util.find_spec(q) is None]

try:                                            # Colab: install quietly, ignore if offline
    import google.colab                          # noqa: F401
    from IPython import get_ipython
    _pk = "-q install -U " + " ".join(_PKGS + ["curl-cffi", "yfinance"])
    get_ipython().run_line_magic("pip", _pk)
except Exception as _exc:
    print("auto-install skipped ({0}) - if an import below fails, run:".format(type(_exc).__name__))
    print("   !pip install " + " ".join(_PKGS))

_miss = _missing(_PKGS + ["curl_cffi", "yfinance"])
if _miss:                                        # non-Colab fallback path
    import subprocess
    print("installing:", _miss)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _miss, check=False)

import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
def display_if(obj):
    try:
        display(obj)
    except NameError:
        print(obj)
print("Setup OK —", os.getcwd())


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.3/149.3 kB 12.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sou

## 1 · The engine (embedded, auditable, editable)

Four cells below are the complete source of the project — the resilient data layer, the
sweep detector, the orchestration/presentation layer and the self-test suite. They are
written to `wlsweep/*.py` at runtime and imported normally, so you can read and patch the
actual code that runs. Nothing is hidden in a wheel or a gist.

<details><summary><b>Why 1 HTTP call per symbol matters</b></summary>

Yahoo throttles by **request rate**, not by symbol. The naive design (chart call → parse →
second chart call to grab the meta block → weekly fallback → lookup for renames) makes
3–4 calls per symbol; on a 2 300-symbol universe that is 8–9 k requests and you spend the
next ten minutes inside 429s. This project fetches bars **and** the free `meta` block in a
single response, keeps a process-wide minimum interval between calls (not per-thread),
never fires an optional call while a 429 is still hot, and defers throttled symbols to one
quiet second pass on a different transport. Measured here: **1.00 call/symbol, 0×429**.

</details>

In [ ]:
# Data layer — resilient Yahoo + NSE fetching   ·   embedded verbatim 2026-08-29, module starts at the next line
_SRC_NSE_DATA = r'''
"""
nse_data.py — resilient market-data layer for the NSE Weekly Liquidity Sweep Screener.

Design goals (this file exists because "data fetching" is the failure mode):
  * Multi-source universe resolution (NSE archive CSV -> local cache -> pasted list).
  * Multi-source OHLCV: Yahoo chart API (primary, plain requests) -> Yahoo weekly
    fallback -> local CSV cache -> NSE archived daily bhavcopy (deep fallback).
  * Politely rate-limited, retrying with exponential backoff + jitter, host rotation,
    adaptive backpressure when the API pushes 429/5xx, and per-symbol failure
    classification so the screener can *prove* data quality instead of hoping.
  * On-disk cache (gzip CSV) so a Colab re-run costs ~0 network calls.

No paid APIs, no key, no scraping of the anti-bot NSE HTML pages.
"""

from __future__ import annotations

import gzip
import io
import json
import math
import os
import random
import re
import threading
import time
from collections import deque
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from typing import Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

try:  # optional, nicer progress bar in Colab
    from tqdm.auto import tqdm as _tqdm
except Exception:  # pragma: no cover
    _tqdm = None


# --------------------------------------------------------------------------------------
# Constants
# --------------------------------------------------------------------------------------
IST = timezone(timedelta(hours=5, minutes=30))

OHLCV_COLS = ["Open", "High", "Low", "Close", "Volume"]

YAHOO_HOSTS = ("query1.finance.yahoo.com", "query2.finance.yahoo.com")

_UAS = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) Gecko/20100101 Firefox/125.0",
)

# Series that are unusable / not tradeable as normal swing candidates
_EXCLUDE_SERIES = {"BE", "BL", "B1", "BC", "Z", "Z9", "8P", "G", "GC", "GW", "WQ", "V", "V1"}
_INCLUDE_SERIES_DEFAULT = {"EQ", "A", "AQ", "AQ1000", "BE", "SM", "SS", "MT", "M1", "MZ"}

CSV_SECTIONS = ["Company Name", "Symbol", "Series", "ISIN", "Industry"]

# Symbols Yahoo no longer serves under the NSE ticker (demergers/renames). The screener
# reports them explicitly instead of silently dropping them; extend this if you hit more.
YAHOO_ALIASES: Dict[str, str] = {
    "TATAMOTORS": "TATAMOTORS.NS",      # delisted from Yahoo post demerger -> reported, not guessed
}


# --------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------
@dataclass
class FetchConfig:
    """All knobs that control *how* data is fetched (not what is screened)."""

    cache_dir: str = "cache_nse"
    history_years: int = 6            # daily history requested per symbol
    min_weekly_bars: int = 104        # 2 years of weekly bars, else "insufficient history"
    max_fresh_lag_days: int = 10      # last bar older than this => data_quality flag
    request_sleep: float = 0.10       # min gap between two HTTP calls (per worker)
    transport: str = "auto"           # auto | curl_cffi | requests | yfinance
    throttle_pause: float = 45.0      # park time after repeated 429s
    final_pass: bool = True           # retry throttled symbols once, slowly, on another transport
    quote_chunk: int = 100            # symbols per batch-quote call (auto-shrinks on pushback)
    max_retries: int = 5
    backoff_base: float = 1.7
    backoff_cap: float = 40.0
    timeout: float = 25.0
    max_workers: int = 3
    cache_format: str = "csv"         # 'csv' (gzip); parquet also works if pyarrow is present
    use_quotes: bool = False          # optional: try Yahoo batch-quotes for true market cap
    quote_max_calls: int = 6          # ...but never hammer: Yahoo throttles this endpoint hard
    alias_lookup: bool = True         # use the YAHOO_ALIASES map for delisted/renamed tickers
    fetch_reuse_cache_off: bool = False  # True = ignore local cache and refetch everything
    bhavcopy_fallback: bool = False   # deep rescue: pull NSE bhavcopy zips (one file per DAY)
    bhavcopy_max_days: int = 40       # cap: each zip is the whole market, so keep this small


# --------------------------------------------------------------------------------------
# Shared backoff state (one place decides how polite we are, and it adapts)
# --------------------------------------------------------------------------------------
class BackoffState:
    """
    Process-wide throttle state shared by every worker thread.

    * enforces a minimum gap between consecutive HTTP calls
    * doubles the gap after a 429 (and decays it back after successes)
    * pauses everyone when a retry window is pending
    """

    def __init__(self, base_interval: float = 0.25, max_interval: float = 12.0):
        self.base = float(base_interval)
        self.max = float(max_interval)
        self.mult = 1.0
        self.cooldown_until = 0.0
        self._lock = threading.Lock()

    @property
    def interval(self) -> float:
        return min(self.max, self.base * self.mult)

    def schedule(self, secs: float) -> None:
        with self._lock:
            self.cooldown_until = max(self.cooldown_until, time.monotonic() + float(secs))

    def bump(self, factor: float = 2.0) -> None:
        with self._lock:
            self.mult = min(self.max / max(self.base, 1e-6), max(1.0, self.mult * factor))

    def relax(self) -> None:
        with self._lock:
            self.mult = max(1.0, self.mult * 0.7)

    def acquire(self) -> None:
        """Block until it is polite to issue the next request."""
        while True:
            with self._lock:
                now = time.monotonic()
                wait_for = max(0.0, self.cooldown_until - now)
                gap = self.interval
            if wait_for > 0:
                time.sleep(min(wait_for, 10.0))
                continue
            with self._lock:
                now = time.monotonic()
                if now - getattr(self, "_last", 0.0) >= self.interval:
                    self._last = now
                    return
            time.sleep(min(gap, 0.05))


# --------------------------------------------------------------------------------------
# Session / HTTP plumbing
# --------------------------------------------------------------------------------------
class YahooClient:
    """Thin, polite, self-healing wrapper around the public Yahoo endpoints."""

    RETRYABLE = (429, 500, 502, 503, 504, 520, 521, 522)

    def __init__(self, cfg: FetchConfig, log: Optional[Callable[[str], None]] = None,
                 transport: Optional[str] = None):
        self.cfg = cfg
        self.log = log or (lambda m: None)
        self.transport = transport or (cfg.transport or "auto")
        if self.transport == "auto":
            self.transport = "curl_cffi" if _have_curl_cffi() else "requests"
        if self.transport not in ("curl_cffi", "requests", "yfinance"):
            self.transport = "requests"
        self.state = BackoffState(base_interval=max(0.02, cfg.request_sleep))
        self.stats: Dict[str, float] = {
            "http_calls": 0, "retries": 0, "rate_limited_429": 0, "server_errors": 0,
            "not_found": 0, "parse_errors": 0, "network_errors": 0, "backoff_seconds": 0.0,
        }
        self._local = threading.local()
        self._crumb_lock = threading.Lock()
        self._crumb: Optional[str] = None
        self._crumb_attempted = False

    # -- transport --------------------------------------------------------------------
    def _session(self):
        sess = getattr(self._local, "sess", None)
        if sess is None:
            if self.transport == "curl_cffi":
                try:
                    from curl_cffi import requests as cr          # type: ignore
                    sess = cr.Session(impersonate="chrome")
                    self._local.sess = sess
                    return sess
                except Exception:
                    self.transport = "requests"
            import requests
            sess = requests.Session()
            sess.headers.update({
                "User-Agent": random.choice(_UAS),
                "Accept": "application/json, text/plain, */*",
                "Accept-Language": "en-US,en;q=0.9",
                "Referer": "https://finance.yahoo.com/",
                "Connection": "keep-alive",
            })
            self._local.sess = sess
        return sess

    def _raw_get(self, url: str, params: dict):
        """One HTTP GET through the active transport, returning a response-like object."""
        if self.transport == "curl_cffi":
            return self._session().get(url, params=params, timeout=self.cfg.timeout)
        if self.transport == "yfinance":
            return _YfResponse(self._yf_fetch(url, params))
        return self._session().get(url, params=params, timeout=self.cfg.timeout,
                                   headers={"Accept-Encoding": "gzip, deflate"})

    def _yf_fetch(self, url: str, params: dict):
        """Last-resort transport: let yfinance do the request (it has its own cookie/crumb logic)."""
        import yfinance as yf  # type: ignore
        sym = url.split("/chart/")[-1].split("?")[0]
        rng = str(params.get("range") or "max")
        ivl = str(params.get("interval") or "1d")
        if "period1" in params:
            h = yf.Ticker(sym).history(start=str(pd.to_datetime(int(params["period1"]), unit="s").date()),
                                       end=str(pd.to_datetime(int(params["period2"]), unit="s").date()),
                                       interval=ivl, auto_adjust=False, actions=False)
        else:
            h = yf.Ticker(sym).history(period=rng, interval=ivl, auto_adjust=False, actions=False)
        if h is None or h.empty:
            return None
        idx = pd.to_datetime(h.index)
        idx = idx.tz_convert("Asia/Kolkata").tz_localize(None) if getattr(idx, "tz", None) else idx
        return {"chart": {"result": [{
            "timestamp": [int(t.timestamp()) for t in idx],
            "meta": {"symbol": sym, "longName": sym},
            "indicators": {"quote": [{
                "open": [float(v) if pd.notna(v) else None for v in h["Open"]],
                "high": [float(v) if pd.notna(v) else None for v in h["High"]],
                "low": [float(v) if pd.notna(v) else None for v in h["Low"]],
                "close": [float(v) if pd.notna(v) else None for v in h["Close"]],
                "volume": [float(v) if pd.notna(v) else 0.0 for v in h.get("Volume", [])]
                if "Volume" in h.columns else [0.0] * len(h)}]}}]}}

    def throttled_recently(self, within: float = 20.0) -> bool:
        """True if a 429 landed in the last `within` seconds (used to skip optional calls)."""
        return (time.monotonic() - getattr(self, "_last_429_ts", 0.0)) < within

    def switch_transport(self, name: str) -> None:
        self.transport = name
        self._local = threading.local()
        self.log(f"    transport -> {name}")

    def _record(self, key: str, n: float = 1) -> None:
        with self._crumb_lock:
            self.stats[key] = self.stats.get(key, 0) + n

    # -- raw GET with retry / backoff / host rotation ----------------------------------
    def _get_json(self, path: str, params: dict, *, expect_crumb: bool = False,
                  max_attempts: Optional[int] = None) -> Optional[dict]:
        attempts = int(max_attempts if max_attempts is not None else self.cfg.max_retries)
        last_err = "unknown"
        for n in range(attempts):
            host = YAHOO_HOSTS[n % len(YAHOO_HOSTS)]
            url = f"https://{host}{path}"
            p = dict(params)
            crumb_used = False
            if expect_crumb:
                crumb = self.get_crumb()
                if crumb:
                    p["crumb"] = crumb
                    crumb_used = True
            self.state.acquire()
            self._record("http_calls")
            try:
                r = self._raw_get(url, p)
            except Exception as exc:
                last_err = f"{type(exc).__name__}: {exc}"
                self._record("network_errors")
                self.state.bump(1.5)
                delay = min(self.cfg.backoff_cap, self.cfg.backoff_base ** n)
                delay *= (0.6 + 0.8 * random.random())
                self.state.schedule(delay)
                self._record("backoff_seconds", delay)
                continue

            code = getattr(r, "status_code", 200)
            if code == 200:
                self._429_run = 0
                self.state.relax()
                try:
                    return r.json()
                except Exception as exc:
                    last_err = f"json decode: {exc}"
                    self._record("parse_errors")
                    continue
            if code == 404:
                self._record("not_found")
                return None
            if code == 429:
                # the only thing that works here is to slow down, never to hammer harder
                self._record("rate_limited_429")
                self._last_429_ts = time.monotonic()
                self._429_run = getattr(self, "_429_run", 0) + 1
                last_err = "HTTP 429 (rate limited)"
                try:
                    wait = float(r.headers.get("Retry-After") or 0.0)
                except Exception:
                    wait = 0.0
                delay = max(wait, min(self.cfg.backoff_cap,
                                      self.cfg.backoff_base ** (n + 1)) * 1.5)
                self.state.bump(2.0)
                self.state.schedule(delay)
                self._record("backoff_seconds", delay)
                if self._429_run >= 6:                # everybody is being throttled
                    self.state.schedule(max(delay, self.cfg.throttle_pause))
                    self._429_run = 0
                continue
            if code in (401, 403) and expect_crumb and not crumb_used:
                with self._crumb_lock:
                    self._crumb, self._crumb_attempted = None, False
                last_err = f"HTTP {code} (crumb refresh requested)"
                continue
            if code in self.RETRYABLE:
                self._record("server_errors")
                last_err = f"HTTP {code}"
                delay = min(self.cfg.backoff_cap, self.cfg.backoff_base ** n)
                self.state.bump(1.3)
                self.state.schedule(delay)
                self._record("backoff_seconds", delay)
                continue
            last_err = f"HTTP {code}: {r.text[:120]}"
            self._record("parse_errors")
            return None
        self.log(f"    ! giving up on {path}: {last_err}")
        return None

    # -- crumb (only the batch quote endpoint needs it) --------------------------------
    def get_crumb(self) -> Optional[str]:
        with self._crumb_lock:
            if self._crumb_attempted:
                return self._crumb
            self._crumb_attempted = True
        sess = self._session()
        crumb = None
        try:
            try:
                sess.get("https://finance.yahoo.com/markets/stocks/", timeout=15)
            except Exception:
                pass
            self.state.acquire()
            r = sess.get(f"https://{YAHOO_HOSTS[0]}/v1/test/getcrumb", timeout=15,
                         headers={"Accept": "text/plain, */*"})
            if r.status_code == 200:
                c = r.text.strip()
                crumb = c if _looks_like_crumb(c) else None
        except Exception:
            crumb = None
        with self._crumb_lock:
            self._crumb = crumb
        return crumb

    # -- chart data --------------------------------------------------------------------
    def chart(self, yahoo_symbol: str, period: str = "max", interval: str = "1d",
              start: Optional[str] = None, end: Optional[str] = None) -> pd.DataFrame:
        """Daily/weekly bars only (one HTTP call)."""
        df, _ = self.chart_with_meta(yahoo_symbol, period=period, interval=interval,
                                      start=start, end=end)
        return df

    def chart_with_meta(self, yahoo_symbol: str, period: str = "max",
                        interval: str = "1d", start: Optional[str] = None,
                        end: Optional[str] = None) -> Tuple[pd.DataFrame, dict]:
        """
        ONE HTTP call, both the bars and the little meta block Yahoo sends for free
        (name, last price, 52w range). Keeping this at exactly one request per symbol is
        the whole reason the bulk fetch survives Yahoo's throttling.
        """
        params: dict = {"interval": interval, "includePrePost": "false",
                        "events": "div,split"}
        if start and end:
            params["period1"] = _to_epoch(start)
            params["period2"] = _to_epoch(end)
        else:
            params["range"] = period
        js = self._get_json(f"/v8/finance/chart/{yahoo_symbol}", params,
                            max_attempts=max(3, int(self.cfg.max_retries)))
        meta: dict = {}
        try:
            meta = (((js or {}).get("chart") or {}).get("result") or [{}])[0].get("meta", {}) or {}
        except Exception:
            meta = {}
        return parse_chart_json(js), meta

    def quote_batch(self, yahoo_symbols: Sequence[str]) -> pd.DataFrame:
        """
        Market cap / price / average volume for many symbols. Uses the batch quote endpoint
        (needs cookie+crumb) and *shrinks the chunk* whenever Yahoo pushes back, so a rate
        limit degrades throughput instead of destroying the ranking.
        """
        fields = ("symbol,marketCap,regularMarketPrice,sharesOutstanding,fiftyTwoWeekLow,"
                  "fiftyTwoWeekHigh,twoHundredDayAverage,averageDailyVolume3Month,"
                  "averageDailyVolume10Day,longName,shortName,regularMarketVolume,currency,"
                  "exchange,priceToBook,trailingPE,firstTradeDateMilliseconds")
        syms = [x if x.endswith(".NS") else f"{x}.NS" for x in yahoo_symbols]
        rows: List[dict] = []
        chunk = max(25, int(self.cfg.quote_chunk))
        # this endpoint is throttled hard by Yahoo, so: big chunk, few calls, quit early
        n_chunks = -(-len(syms) // chunk)
        allowed = int(self.cfg.quote_max_calls)
        if n_chunks > allowed:                       # spread the budget over bigger chunks
            chunk = max(120, -(-len(syms) // allowed))
            n_chunks = -(-len(syms) // chunk)
            if n_chunks > allowed:
                syms = syms[: chunk * allowed]        # and simply rank the rest by turnover
                n_chunks = allowed
        i = 0
        fails = 0
        while i < len(syms):
            batch = syms[i:i + chunk]
            js = self._get_json("/v7/finance/quote",
                                {"symbols": ",".join(batch), "fields": fields,
                                 "formatted": "false"},
                                expect_crumb=True, max_attempts=2)
            res = ((js or {}).get("quoteResponse") or {}).get("result")
            if res is None:
                fails += 1
                if fails >= 2:
                    self.log("    quote endpoint is throttling us -> falling back to "
                             "turnover-based ranking (no market cap). This does not affect "
                             "the screen, only the 'top 1000' ordering.")
                    break
                i += chunk                            # skip this slice rather than loop forever
                continue
            fails = 0
            rows.extend(res)
            i += chunk
        if not rows:
            return pd.DataFrame(columns=["Symbol", "Name", "MarketCap", "Close", "AvgVolume3M"])
        q = pd.DataFrame(rows).drop_duplicates(subset=["symbol"], keep="first")
        q["Symbol"] = q["symbol"].astype(str).str.upper().str.replace(".NS", "", regex=False)
        for src, dst in (("marketCap", "MarketCap"), ("regularMarketPrice", "Close"),
                         ("averageDailyVolume3Month", "AvgVolume3M"),
                         ("sharesOutstanding", "SharesOutstanding"),
                         ("fiftyTwoWeekLow", "Low52W"), ("fiftyTwoWeekHigh", "High52W"),
                         ("twoHundredDayAverage", "SMA200D"), ("priceToBook", "Pb"),
                         ("trailingPE", "PE")):
            q[dst] = pd.to_numeric(q.get(src), errors="coerce") if src in q.columns else np.nan
        if "longName" in q.columns or "shortName" in q.columns:
            ln = q.get("longName")
            sn = q.get("shortName")
            q["Name"] = ln if ln is None else ln.fillna(sn)
        else:
            q["Name"] = ""
        cols = ["Symbol", "Name", "MarketCap", "Close", "AvgVolume3M", "SharesOutstanding",
                "Low52W", "High52W", "SMA200D", "Pb", "PE"]
        return q.reindex(columns=cols)

    def lookup(self, query: str, limit: int = 10) -> pd.DataFrame:
        js = self._get_json("/v1/finance/lookup",
                            {"query": query, "type": "equity", "count": limit},
                            max_attempts=2)
        res = ((js or {}).get("finance") or {}).get("result") or []
        rows = []
        for block in res:
            for qq in block.get("quotes", []) or []:
                rows.append({"yahoo_symbol": qq.get("symbol"),
                             "description": qq.get("shortname") or qq.get("longname"),
                             "exch": qq.get("exchDisp") or qq.get("exch")})
        return pd.DataFrame(rows)


class _YfResponse:
    """Minimal requests.Response stand-in for the yfinance transport."""

    def __init__(self, payload):
        self._payload = payload
        self.status_code = 200 if payload else 404
        self.headers = {}
        self.text = "" if payload else "no data"

    def json(self):
        if self._payload is None:
            raise ValueError("empty payload")
        return self._payload


def _have_curl_cffi() -> bool:
    try:
        import curl_cffi  # noqa: F401
        return True
    except Exception:
        return False


def _looks_like_crumb(c: str) -> bool:
    return bool(c) and len(c) < 32 and re.fullmatch(r"[A-Za-z0-9;/=\-\._%+]+", c) is not None


def _to_epoch(date_str: str) -> str:
    d = pd.to_datetime(date_str).tz_localize(IST)
    return str(int(d.timestamp()))


# --------------------------------------------------------------------------------------
# Yahoo JSON parsing
# --------------------------------------------------------------------------------------
def parse_chart_json(js: Optional[dict]) -> pd.DataFrame:
    """chart JSON -> DataFrame[Open,High,Low,Close,Volume] indexed by IST date (daily)."""
    empty = pd.DataFrame(columns=OHLCV_COLS)
    if not js:
        return empty
    chart = js.get("chart") or {}
    result = chart.get("result")
    if not result:
        return empty
    r = result[0]
    ts = r.get("timestamp")
    if not ts:
        return empty
    quote = (r.get("indicators") or {}).get("quote") or [{}]
    q = quote[0] or {}
    adj = (r.get("indicators") or {}).get("adjclose") or []
    df = pd.DataFrame({
        "Open": q.get("open"), "High": q.get("high"), "Low": q.get("low"),
        "Close": q.get("close"), "Volume": q.get("volume"),
    }, index=pd.to_datetime(ts, unit="s", utc=True))
    if adj and adj[0].get("adjclose") is not None:
        # keep raw close (screener trades price, not adjusted returns) but store ratio
        df["AdjClose"] = pd.Series(adj[0]["adjclose"], index=df.index)
    if df.index.tz is not None:
        df.index = df.index.tz_convert(IST)
    df.index = pd.to_datetime(df.index).normalize()
    df = df[~df.index.duplicated(keep="last")].sort_index()
    for c in ["Open", "High", "Low", "Close"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["Volume"] = pd.to_numeric(df["Volume"], errors="coerce").fillna(0)
    # A bar is only usable if OHLC are all finite and High >= Low > 0.
    ok = df[["Open", "High", "Low", "Close"]].notna().all(axis=1)
    ok &= df["High"] >= df["Low"]
    ok &= df["Low"] > 0
    df = df[ok].astype(float)
    return df.dropna(subset=["Close"])


# --------------------------------------------------------------------------------------
# Universe resolution
# --------------------------------------------------------------------------------------
def _read_csv_bytes(raw: bytes) -> pd.DataFrame:
    return pd.read_csv(io.BytesIO(raw), dtype=str, skipinitialspace=True)


def load_universe_full_list(cache_dir: str, client: Optional[YahooClient] = None,
                            allow_network: bool = True, log: Optional[Callable[[str], None]] = None) -> pd.DataFrame:
    """
    All NSE-listed equities with Series / ISIN / face value.
    Sources tried in order: archives.nseindia.com, nseindia.com mirror, cached file,
    bundled sample. Returns columns: Symbol, Company, Series, ISIN, FaceValue, DateOfListing.
    """
    log = log or (lambda m: None)
    try:
        os.makedirs(cache_dir, exist_ok=True)     # read-only / disconnected Drive must not
    except Exception as exc:                      # turn "use the cache" into a hard crash
        log(f"  ! cannot create {cache_dir} ({type(exc).__name__}) - trying read-only use")
    cache_fp = os.path.join(cache_dir, "universe_equity_l.csv")
    urls = [
        "https://archives.nseindia.com/content/equities/EQUITY_L.csv",
        "https://www.nsearchives.co.in/content/equities/EQUITY_L.csv",
        "https://nsearchives.nseindia.com/content/equities/EQUITY_L.csv",
    ]
    raw = None
    if allow_network:
        for u in urls:
            try:
                sess = (client or YahooClient(FetchConfig()))._session()
                r = sess.get(u, timeout=25)
                if r.status_code == 200 and len(r.content) > 500:
                    raw = r.content
                    with open(cache_fp, "wb") as f:
                        f.write(raw)
                    log(f"  universe: downloaded {u.split('/')[2]} ({len(raw)//1024} KB)")
                    break
            except Exception as exc:
                log(f"  universe: {u.split('/')[2]} failed ({type(exc).__name__})")
    if raw is None and os.path.exists(cache_fp):
        log(f"  universe: using cached copy {cache_fp}")
        raw = open(cache_fp, "rb").read()
    if raw is None:
        raise RuntimeError(
            "Could not obtain the NSE equity list (network blocked and no cache). "
            "Download EQUITY_L.csv from archives.nseindia.com/content/equities/EQUITY_L.csv "
            "on your laptop, upload it to the Colab as 'cache_nse/universe_equity_l.csv', "
            "or paste your own symbol list (UNIVERSE['pasted_symbols'])."
        )
    df = _read_csv_bytes(raw)
    ren = {}
    for c in df.columns:
        cl = c.strip().lower()
        if cl == "symbol":
            ren[c] = "Symbol"
        elif "name" in cl:
            ren[c] = "Company"
        elif cl == "series":
            ren[c] = "Series"
        elif "isin" in cl:
            ren[c] = "ISIN"
        elif "face" in cl:
            ren[c] = "FaceValue"
        elif "listing" in cl:
            ren[c] = "DateOfListing"
    df = df.rename(columns=ren)
    if "Symbol" not in df.columns:
        raise RuntimeError(f"Unexpected EQUITY_L.csv header: {list(df.columns)[:8]}")
    df["Symbol"] = df["Symbol"].astype(str).str.strip().str.upper()
    for c in ("Series", "Company", "ISIN", "FaceValue", "DateOfListing"):
        if c not in df.columns:
            df[c] = np.nan
    df = df.dropna(subset=["Symbol"]).drop_duplicates(subset=["Symbol"])
    if "Series" in df.columns:
        df["Series"] = df["Series"].fillna("").astype(str).str.strip()
        df = df[~df["Series"].str.upper().isin(_EXCLUDE_SERIES)]
    return df[["Symbol", "Company", "Series", "ISIN", "FaceValue", "DateOfListing"]].reset_index(drop=True)


def load_industry_map(cache_dir: str, allow_network: bool = True,
                      log: Optional[Callable[[str], None]] = None) -> pd.Series:
    """Symbol -> Industry, from the Nifty 500 constituent file (best-effort, non fatal)."""
    log = log or (lambda m: None)
    os.makedirs(cache_dir, exist_ok=True)
    fp = os.path.join(cache_dir, "universe_nifty500_industry.csv")
    urls = [
        "https://archives.nseindia.com/content/indices/ind_nifty500list.csv",
        "https://nsearchives.nseindia.com/content/indices/ind_nifty500list.csv",
    ]
    raw = None
    if allow_network:
        for u in urls:
            try:
                import requests
                r = requests.get(u, timeout=25, headers={"User-Agent": _UAS[0]})
                if r.status_code == 200 and len(r.content) > 200:
                    raw = r.content
                    with open(fp, "wb") as f:
                        f.write(raw)
                    break
            except Exception:
                pass
    if raw is None and os.path.exists(fp):
        raw = open(fp, "rb").read()
    if raw is None:
        log("  industry map unavailable (offline) — continuing without sector column")
        return pd.Series(dtype=object)
    try:
        df = _read_csv_bytes(raw).rename(columns=lambda c: c.strip())
        if "Symbol" in df.columns and "Industry" in df.columns:
            df["Symbol"] = df["Symbol"].astype(str).str.upper().str.strip()
            return df.drop_duplicates("Symbol").set_index("Symbol")["Industry"]
    except Exception:
        pass
    return pd.Series(dtype=object)


# --------------------------------------------------------------------------------------
# Cache
# --------------------------------------------------------------------------------------
def _cache_path(cache_dir: str, symbol: str, interval: str, fmt: str) -> str:
    # NSE tickers really do contain "&" (M&M, S&P...) which is not filename-safe. Map it to a
    # token that cannot collide with a *different* real symbol, so two tickers never share one
    # cache file (read and write go through here, so they can never disagree).
    sym = re.sub(r"\s+", "", str(symbol).upper())
    safe = sym.replace("&", "_AMP_")
    safe = re.sub(r"[^A-Z0-9_.-]", "_", safe)
    return os.path.join(cache_dir, f"{safe}.{interval}.{fmt}")


def save_daily_cache(df: pd.DataFrame, cache_dir: str, symbol: str, fmt: str = "csv") -> None:
    if df is None or df.empty:
        return
    os.makedirs(cache_dir, exist_ok=True)
    fp = _cache_path(cache_dir, symbol, "1d", fmt)
    try:
        if fmt == "parquet":
            df.to_parquet(fp)
        else:
            with gzip.open(fp + ".gz", "wt") as f:
                df.to_csv(f)
    except Exception:
        pass


def load_daily_cache(cache_dir: str, symbol: str, fmt: str = "csv") -> Optional[pd.DataFrame]:
    fp = _cache_path(cache_dir, symbol, "1d", fmt)
    try:
        if fmt == "parquet" and os.path.exists(fp):
            return pd.read_parquet(fp)
        if os.path.exists(fp + ".gz"):
            with gzip.open(fp + ".gz", "rt") as f:
                df = pd.read_csv(f, index_col=0, parse_dates=True)
            for c in OHLCV_COLS:
                if c in df.columns:
                    df[c] = pd.to_numeric(df[c], errors="coerce")
            return df
    except Exception:
        return None
    return None


def save_meta(meta: dict, cache_dir: str, symbol: str) -> None:
    """Persist the free per-symbol meta (name, price, 52w range) next to the bars."""
    if not meta:
        return
    try:
        d = os.path.join(cache_dir, "meta")
        os.makedirs(d, exist_ok=True)
        pd.DataFrame([{
            "Symbol": str(meta.get("symbol", symbol)).replace(".NS", "").upper(),
            "Name": meta.get("longName") or meta.get("shortName") or "",
            "Close": meta.get("regularMarketPrice"),
            "High52W": meta.get("fiftyTwoWeekHigh"),
            "Low52W": meta.get("fiftyTwoWeekLow"),
            "PrevClose": meta.get("chartPreviousClose"),
            "FirstTrade": pd.to_datetime(meta.get("firstTradeDate"), unit="s")
            if meta.get("firstTradeDate") else pd.NaT,
            "LastQuote": pd.to_datetime(meta.get("regularMarketTime"), unit="s", utc=True)
            if meta.get("regularMarketTime") else pd.NaT,
        }]).to_csv(os.path.join(d, f"{symbol.upper()}.meta.csv"), index=False)
    except Exception:
        pass


def build_quotes_from_meta(cache_dir: str) -> pd.DataFrame:
    """
    Rebuild a quotes table from the meta files written during the bulk fetch. Zero extra
    network calls, and it gives the notebook company names + 52-week context for free.
    """
    d = os.path.join(cache_dir, "meta")
    if not os.path.isdir(d):
        return pd.DataFrame(columns=["Symbol", "Name", "Close", "High52W", "Low52W", "MarketCap"])
    frames = []
    for f in os.scandir(d):
        if f.name.endswith(".meta.csv"):
            try:
                frames.append(pd.read_csv(f.path))
            except Exception:
                continue
    if not frames:
        return pd.DataFrame(columns=["Symbol", "Name", "Close", "High52W", "Low52W", "MarketCap"])
    q = pd.concat(frames, ignore_index=True).drop_duplicates("Symbol")
    if "MarketCap" not in q.columns:
        q["MarketCap"] = np.nan
    return q


# --------------------------------------------------------------------------------------
# Index membership (used to sanity-check the 'top 1000' ordering)
# --------------------------------------------------------------------------------------
INDEX_FILES = {
    "Nifty50": "https://archives.nseindia.com/content/indices/ind_nifty50list.csv",
    "Nifty100": "https://archives.nseindia.com/content/indices/ind_nifty100list.csv",
    "Nifty200": "https://archives.nseindia.com/content/indices/ind_nifty200list.csv",
    "Nifty500": "https://archives.nseindia.com/content/indices/ind_nifty500list.csv",
    "NiftyMidcap150": "https://archives.nseindia.com/content/indices/ind_niftymidcap150list.csv",
    "NiftySmallcap250": "https://archives.nseindia.com/content/indices/ind_niftysmallcap250list.csv",
}


def load_index_membership(cache_dir: str, allow_network: bool = True,
                          log: Optional[Callable[[str], None]] = None) -> pd.DataFrame:
    """
    Symbol -> best index tier found in the NSE archive (all free, no cookies, no bot wall).
    Returns columns [Symbol, IndexTier, Industry]. Never raises: a missing file is not fatal.
    """
    log = log or (lambda m: None)
    try:
        os.makedirs(cache_dir, exist_ok=True)
    except Exception:
        pass
    rank = {k: n for n, k in enumerate(INDEX_FILES)}
    best: Dict[str, Tuple[int, str, str]] = {}
    import requests
    for name, url in INDEX_FILES.items():
        fp = os.path.join(cache_dir, f"index_{name}.csv")
        raw = None
        if allow_network:
            for attempt in range(2):
                try:
                    r = requests.get(url, timeout=25, headers={"User-Agent": _UAS[attempt % len(_UAS)]})
                    if r.status_code == 200 and len(r.content) > 150:
                        raw = r.content
                        with open(fp, "wb") as f:
                            f.write(raw)
                        break
                    if r.status_code == 404:
                        break
                except Exception:
                    time.sleep(1.0)
        if raw is None and os.path.exists(fp):
            raw = open(fp, "rb").read()
        if raw is None:
            continue
        try:
            df = _read_csv_bytes(raw).rename(columns=lambda c: str(c).strip())
            if "Symbol" not in df.columns:
                continue
            ind = df["Industry"] if "Industry" in df.columns else pd.Series([""] * len(df))
            for sym, industry in zip(df["Symbol"].astype(str), ind.astype(str)):
                sym = sym.strip().upper()
                if not sym:
                    continue
                cur = best.get(sym)
                if cur is None or rank[name] < cur[0]:
                    best[sym] = (rank[name], name, industry)
        except Exception as exc:
            log(f"  index {name}: parse failed ({type(exc).__name__})")
    if not best:
        log("  index membership unavailable (offline?) - ranking on turnover only")
        return pd.DataFrame(columns=["Symbol", "IndexTier", "Industry"])
    out = pd.DataFrame([{"Symbol": k, "IndexTier": v[1], "Industry": v[2], "_r": v[0]}
                        for k, v in best.items()]).sort_values("_r").drop(columns="_r")
    return out.reset_index(drop=True)


# --------------------------------------------------------------------------------------
# Data-quality gate
# --------------------------------------------------------------------------------------
def assess_daily(df: pd.DataFrame, *, min_weekly_bars: int, max_fresh_lag_days: int,
                  today: Optional[pd.Timestamp] = None) -> Tuple[pd.DataFrame, List[str], dict]:
    """Clean a daily frame + return (clean_df, list_of_issues, metrics)."""
    issues: List[str] = []
    metrics: dict = {}
    if df is None or df.empty:
        return pd.DataFrame(columns=OHLCV_COLS), ["no_data"], metrics
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        try:
            df.index = pd.to_datetime(df.index)
        except Exception:
            return pd.DataFrame(columns=OHLCV_COLS), ["bad_index"], metrics
    if getattr(df.index, "tz", None) is not None:
        df.index = df.index.tz_convert(IST).tz_localize(None)
    else:
        df.index = pd.to_datetime(df.index)
    df = df[~df.index.duplicated(keep="last")].sort_index()
    keep = [c for c in OHLCV_COLS if c in df.columns]
    df = df[keep]

    # ---- repair, do not delete -------------------------------------------------------
    # Yahoo intermittently ships bars whose High/Low ignore Open/Close (seen on NORBTEAEXP:
    # Open 6.35, High 6.30). Dropping them loses real history and can hide a sweep; taking the
    # max/min of the four fields recovers a bar that is true to the tape.
    four = df[["Open", "High", "Low", "Close"]]
    h_true = four.max(axis=1)
    l_true = four[["Open", "Low", "Close"]].min(axis=1)
    viol = (df["High"] < h_true - 1e-9) | (df["Low"] > l_true + 1e-9) | (df["High"] < df["Low"])
    n_repaired = int(viol.sum())
    if n_repaired:
        df.loc[viol, "High"] = h_true[viol]
        df.loc[viol, "Low"] = l_true[viol]
        issues.append(f"repaired {n_repaired} inconsistent OHLC bar(s)")
    metrics["ohlc_repaired"] = n_repaired
    # only genuinely unusable rows get dropped
    bad = (df[["Open", "High", "Low", "Close"]].isna().any(axis=1)
           | (df[["Open", "High", "Low", "Close"]] <= 0).any(axis=1))
    if bad.any():
        df = df[~bad.to_numpy()]
        issues.append(f"dropped {int(bad.sum())} corrupt bar(s)")
        if len(df) < 2:
            return pd.DataFrame(columns=OHLCV_COLS), ["no_data"], metrics

    # Yahoo returns a row for every NSE *holiday* too: zero volume, close carried forward.
    # Those are not "sessions" - left in, they poison the volume ratio, the turnover
    # estimate, the week's Day count and the no-trade detector. Drop the pure carry-forward
    # rows; keep a zero-volume day if the price actually moved (a real halt is informative).
    vol = pd.to_numeric(df["Volume"], errors="coerce").fillna(0.0)
    prev_close = df["Close"].shift(1)
    carry = (vol <= 0) & ((df["Close"] - prev_close).abs() <= 1e-9)
    zero_vol = int((vol <= 0).sum())
    metrics["zero_volume_days"] = zero_vol
    metrics["holiday_rows_dropped"] = int(carry.sum())
    if carry.any():
        df = df[~carry.to_numpy()]
        if len(df) < 2:
            return pd.DataFrame(columns=OHLCV_COLS), ["no_data"], metrics
    if len(df) >= 60:
        frac = zero_vol / len(df)
        if frac > 0.25:
            issues.append(f"low liquidity: {frac:.0%} zero-volume days")

    # ---- corporate-action / bad-feed artefacts -------------------------------------
    # Splits ARE pre-adjusted by Yahoo (verified: WIPRO 2:1 on 2024-12-03 shows no price
    # jump), but a demerger or a bad feed can leave a huge one-day step that invents or
    # destroys a swing low. Flag it so the user can judge the affected symbol.
    r = df["Close"].pct_change()
    worst = float(r.abs().max()) if len(r) else 0.0
    big = r[r.abs() > 0.35]
    metrics["max_1d_move"] = round(float(worst) * 100, 2) if np.isfinite(worst) else None
    metrics["gap_days"] = int(len(big))
    if len(big) >= 3:
        issues.append(f"{len(big)} unexplained >35% one-day moves (demerger/bonus or bad feed) "
                      f"max {metrics['max_1d_move']}%")
    elif len(big):
        issues.append(f"{len(big)} >35% one-day move(s) - check for corporate action")

    n_weeks = max(0, int(math.floor(len(df) / 5.0)))
    metrics["daily_rows"] = int(len(df))
    metrics["first_day"] = str(df.index[0].date()) if len(df) else None
    metrics["last_day"] = str(df.index[-1].date()) if len(df) else None
    if n_weeks < min_weekly_bars:
        issues.append(f"only ~{n_weeks} weekly bars (need {min_weekly_bars})")

    ref = pd.Timestamp(today) if today is not None else pd.Timestamp(datetime.now(IST).date())
    lag = int((ref - df.index[-1]).days)
    metrics["staleness_days"] = lag
    if lag > max(20, max_fresh_lag_days * 3):
        issues.append(f"very stale data ({lag} days)")
    elif lag > max_fresh_lag_days:
        issues.append(f"stale by {lag} days")

    if "Close" in df.columns and len(df) > 30:
        metrics["price"] = float(df["Close"].iloc[-1])
        if metrics["price"] <= 0:
            issues.append("non-positive close")
    return df, issues, metrics


def bhavcopy_available() -> bool:
    """Cheap check that the NSE daily-archive bucket is reachable (no scraping, no cookies)."""
    import requests
    d = (datetime.now(IST) - timedelta(days=4)).strftime("%Y%m%d")
    u = f"https://archives.nseindia.com/historical/contract%20notes/SME_SqRdBhav_{d}.zip"
    try:
        r = requests.head(u, timeout=10, headers={"User-Agent": _UAS[0]}, allow_redirects=True)
        return r.status_code in (200, 404)  # 404 = bucket alive, wrong date is fine
    except Exception:
        return False


def fetch_bhavcopy_days(start: pd.Timestamp, end: pd.Timestamp, max_days: int = 60) -> pd.DataFrame:
    """
    Deep fallback: concatenated daily bhavcopies from the NSE archive bucket.
    Returns columns: Symbol, date, Open/High/Low/Close/Volume. Never raises.
    """
    import requests
    rows = []
    d = pd.Timestamp(start).date()
    stop = pd.Timestamp(end).date()
    if (stop - d).days > max_days:
        stop = d + timedelta(days=max_days)
    while d <= stop:
        if d.weekday() < 5:  # Mon-Fri only (holidays just 404)
            for kind in ("SqRdBhav", "Sq8Bhav"):
                url = f"https://archives.nseindia.com/historical/contract%20notes/{kind}_{d.strftime('%Y%m%d')}.zip"
                try:
                    r = requests.get(url, timeout=25, headers={"User-Agent": _UAS[0]})
                    if r.status_code != 200:
                        continue
                    import zipfile
                    zf = zipfile.ZipFile(io.BytesIO(r.content))
                    name = [n for n in zf.namelist() if n.lower().endswith(".csv")]
                    if not name:
                        continue
                    txt = zf.read(name[0]).decode("latin1")
                    first = txt.splitlines()[0] if txt else ""
                    if first.strip().upper().startswith("symbol"):
                        df = pd.read_csv(io.StringIO(txt))
                        cols = {c.strip().lower(): c for c in df.columns}
                        need = {"open": "Open", "high": "High", "low": "Low", "close": "Close", "price": "Close"}
                        out = pd.DataFrame()
                        out["Symbol"] = df[cols["symbol"]].astype(str).str.upper().str.strip()
                        for src, dst in need.items():
                            if src in cols:
                                out[dst] = pd.to_numeric(df[cols[src]], errors="coerce")
                        if "avgprice" in cols or "volume" in cols:
                            out["Volume"] = pd.to_numeric(df[cols.get("volume", cols.get("avgprice"))], errors="coerce")
                        out["date"] = pd.Timestamp(d)
                        rows.append(out)
                    break
                except Exception:
                    continue
        d = d + timedelta(days=1)
    if not rows:
        return pd.DataFrame()
    allb = pd.concat(rows, ignore_index=True)
    allb = allb.dropna(subset=["Open", "High", "Low", "Close"]).drop_duplicates(["Symbol", "date"])
    allb["date"] = pd.to_datetime(allb["date"])
    return allb.set_index(["Symbol", "date"]).sort_index()


def bhavcopy_to_daily(allb: pd.DataFrame, symbol: str) -> pd.DataFrame:
    """Slice one symbol out of a MultiIndex(Symbol, date) bhavcopy frame."""
    if allb is None or allb.empty or not isinstance(allb.index, pd.MultiIndex):
        return pd.DataFrame(columns=OHLCV_COLS)
    try:
        if symbol not in allb.index.get_level_values(0):
            return pd.DataFrame(columns=OHLCV_COLS)
        sub = allb.xs(symbol, level=0).copy()
        sub.index = pd.to_datetime(sub.index)
        cols = [c for c in OHLCV_COLS if c in sub.columns]
        return sub[cols].apply(pd.to_numeric, errors="coerce").dropna(subset=["Open", "High", "Low", "Close"])
    except Exception:
        return pd.DataFrame(columns=OHLCV_COLS)


# --------------------------------------------------------------------------------------
# The big fetcher
# --------------------------------------------------------------------------------------
def fetch_daily_bulk(symbols: Sequence[str], cfg: FetchConfig, client: YahooClient,
                     progress: bool = True,
                     on_progress: Optional[Callable[[dict], None]] = None
                     ) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    """
    Fetch + validate daily OHLCV for `symbols`, with a local cache in front of the network.

    Order of attempts per symbol:
        1. local cache file            (free, reproducible, survives runtime restarts)
        2. Yahoo daily bars            (primary)
        3. Yahoo weekly bars           (works when daily history is embargoed/short)
        4. Yahoo lookup -> renamed ticker retry (delistings/spin-offs, e.g. TATAMOTORS)

    Returns
    -------
    data   : dict symbol -> cleaned daily DataFrame (symbols that passed the quality gate)
    report : per-symbol status / row count / first / last / source / issue
    """
    data: Dict[str, pd.DataFrame] = {}
    report_rows: Dict[str, dict] = {}
    lock = threading.Lock()
    todo = list(dict.fromkeys(str(x).upper() for x in symbols if str(x).strip()))
    period = f"{max(2, int(cfg.history_years))}y"

    def _yahoo_symbol(sym: str) -> str:
        return sym if sym.endswith(".NS") else f"{sym}.NS"

    def _load_meta(sym: str) -> Optional[dict]:
        fp = os.path.join(cfg.cache_dir, "meta", f"{sym}.meta.csv")
        if os.path.exists(fp):
            try:
                return pd.read_csv(fp).iloc[-1].to_dict()
            except Exception:
                return None
        return None

    def _one(sym: str) -> dict:
        t0 = time.time()
        rec = {"Symbol": sym, "status": "ok", "rows": 0, "source": "-", "issues": "",
               "elapsed": 0.0, "first": "", "last": "", "alias": ""}
        df, used, meta = None, "cache", {}
        if not cfg.fetch_reuse_cache_off:
            df = load_daily_cache(cfg.cache_dir, sym, cfg.cache_format)
        if df is None or df.empty:
            # --- exactly one network call in the normal path ---
            df, meta = client.chart_with_meta(_yahoo_symbol(sym), period=period, interval="1d")
            used = "yahoo"
            if meta:
                save_meta(meta, cfg.cache_dir, sym)
        if (df is None or df.empty) and not client.throttled_recently():
            # only when Yahoo *really* has no daily bars (young listing / embargoed), never
            # after a throttle - a second call on a 429 just digs the hole deeper
            w, wmeta = client.chart_with_meta(_yahoo_symbol(sym), period=period, interval="1wk")
            if not w.empty:
                df, used = w, "yahoo_weekly"
                if wmeta:
                    save_meta(wmeta, cfg.cache_dir, sym)
        if (df is None or df.empty) and cfg.alias_lookup:
            alias = YAHOO_ALIASES.get(sym)
            if alias and alias != _yahoo_symbol(sym):
                rec["alias"] = alias
                df, _m = client.chart_with_meta(alias, period=period, interval="1d")
                if not df.empty:
                    used = "yahoo:alias"
        if df is None or df.empty:
            blocked = client.throttled_recently()
            rec.update(status=("throttled" if blocked else "no_data"),
                       issues=(("HTTP 429 (rate limited) | transport=" + client.transport)
                               if blocked else
                               "no bars from Yahoo (delisted, renamed, or newly listed)"))
            rec["elapsed"] = time.time() - t0
            return rec
        clean, issues, m = assess_daily(df, min_weekly_bars=cfg.min_weekly_bars,
                                       max_fresh_lag_days=cfg.max_fresh_lag_days)
        rec["rows"] = int(len(clean))
        rec["first"] = m.get("first_day", "") or ""
        rec["last"] = m.get("last_day", "") or ""
        rec["max_1d_move_pct"] = m.get("max_1d_move")
        rec["gap_days"] = m.get("gap_days")
        rec["ohlc_repaired"] = m.get("ohlc_repaired", 0)
        hard = [i for i in issues if i.startswith(("no_data", "bad_index", "only ~", "very stale",
                                                   "non-positive", "low liquidity"))]
        if hard:
            rec.update(status=("insufficient_history" if any(i.startswith("only ~") for i in hard)
                               else ("illiquid_history"
                                     if any(i.startswith("low liquidity") for i in hard)
                                     else "stale_data")),
                       issues="; ".join(issues))
            rec["elapsed"] = time.time() - t0
            return rec
        rec["issues"] = "; ".join(issues)
        rec["source"] = used
        # cache the raw bars even when the quality gate rejects them: the gate can be
        # relaxed later (params change) and re-downloading 1000 files is what gets you blocked
        if used.startswith("yahoo"):
            try:
                save_daily_cache(clean if not hard else df, cfg.cache_dir, sym, cfg.cache_format)
            except Exception:
                pass
        rec["elapsed"] = time.time() - t0
        with lock:
            data[sym] = clean
            report_rows[sym] = rec
        return rec

    workers = max(1, int(cfg.max_workers))
    bar = _tqdm(total=len(todo), desc="Fetching bars", unit="stock", ncols=100) \
        if (progress and _tqdm is not None) else None
    done_ct = 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        inflight = {}
        nxt = 0

        def submit_next():
            nonlocal nxt
            if nxt < len(todo):
                s_ = todo[nxt]
                nxt += 1
                inflight[ex.submit(_one, s_)] = s_

        for _ in range(min(workers * 3, len(todo))):
            submit_next()
        while inflight:
            finished, _ = wait(list(inflight.keys()), return_when=FIRST_COMPLETED)
            for fut in finished:
                sym = inflight.pop(fut)
                try:
                    rec = fut.result()
                except Exception as exc:
                    rec = {"Symbol": sym, "status": "exception", "rows": 0, "source": "-",
                           "issues": f"{type(exc).__name__}: {exc}", "elapsed": 0.0,
                           "first": "", "last": "", "alias": ""}
                    report_rows[sym] = rec
                report_rows.setdefault(sym, rec)
                done_ct += 1
                if bar is not None:
                    bar.update(1)
                if on_progress is not None and (done_ct % 25 == 0 or done_ct == len(todo)):
                    on_progress({"done": done_ct, "total": len(todo), "ok": len(data),
                                 "failed": done_ct - len(data)})
                submit_next()
    if bar is not None:
        bar.close()

    # optional: true market caps from the batch-quote endpoint (best effort only)
    quotes = None
    if cfg.use_quotes and todo:
        try:
            quotes = client.quote_batch(todo)
        except Exception as exc:
            client.log(f"  market-cap enrichment skipped ({type(exc).__name__})")
    if quotes is not None and not quotes.empty:
        quotes.to_csv(os.path.join(cfg.cache_dir, "quotes_optional.csv"), index=False)

    # ---- deferred pass: symbols lost to rate limiting, retried slowly on another transport
    if cfg.final_pass:
        retry = [s for s, r in report_rows.items()
                 if r.get("rows", 0) == 0 and "429" in str(r.get("issues", ""))
                 or (r.get("status") in ("exception", "not_run"))]
        if retry:
            client.log(f"· {len(retry)} symbol(s) were throttled; slow second pass in "
                       f"{max(5, int(cfg.throttle_pause))}s…")
            client.state.schedule(cfg.throttle_pause)
            order = {"curl_cffi": "requests", "requests": "curl_cffi", "yfinance": "requests"}
            client.switch_transport(order.get(client.transport, "yfinance"))
            saved_sleep, saved_workers = cfg.request_sleep, cfg.max_workers
            cfg.request_sleep = max(0.35, saved_sleep * 3)
            cfg.max_workers = 1
            client.state = BackoffState(base_interval=cfg.request_sleep)
            for sym in retry:
                rec = _one(sym)
                report_rows[sym] = rec
            cfg.request_sleep, cfg.max_workers = saved_sleep, saved_workers
            client.log(f"  → second pass recovered "
                       f"{sum(1 for s in retry if report_rows[s].get('rows', 0) > 0)}/{len(retry)}")

    rep = pd.DataFrame([report_rows.get(s, {"Symbol": s, "status": "not_run", "rows": 0,
                                           "source": "-", "issues": "", "elapsed": 0.0,
                                           "first": "", "last": "", "alias": ""}) for s in todo])
    if not rep.empty:
        rep["status"] = rep["status"].fillna("unknown")
    return data, rep


# --------------------------------------------------------------------------------------
# Symbol alias resolution for "renamed/delisted" misses
# --------------------------------------------------------------------------------------
def resolve_missing(missing: Sequence[str], client: YahooClient,
                    universe: Optional[pd.DataFrame] = None, max_tries: int = 40) -> Dict[str, str]:
    """
    For symbols Yahoo could not serve, ask the lookup endpoint and keep any *.NS match
    whose leading token is a prefix of the original symbol (e.g. TATAMOTORS -> TATAMOTOLERS).
    Returns {old_symbol: new_yahoo_symbol} and logs the mapping.
    """
    out: Dict[str, str] = {}
    if getattr(client, "transport", "") == "yfinance":
        client.log("  (alias lookup skipped: lookup endpoint is not proxied by yfinance)")
        return out
    for sym in list(missing)[:max_tries]:
        if client.throttled_recently():
            client.log("  (alias lookup aborted: Yahoo is throttling this IP right now)")
            break
        try:
            lk = client.lookup(sym, limit=8)
        except Exception:
            continue
        if lk is None or lk.empty:
            continue
        base = sym.upper()
        for cand in lk["yahoo_symbol"].dropna().astype(str):
            if not cand.endswith(".NS"):
                continue
            head = cand[:-3].upper()
            if head == base:
                continue
            # same first 5 chars => likely a rename of the same company
            if head[:5] == base[:5] or base[:5] in head:
                out[sym] = cand
                break
    return out


# --------------------------------------------------------------------------------------
# Health report
# --------------------------------------------------------------------------------------
def data_health(report: pd.DataFrame, cfg: FetchConfig) -> str:
    lines: List[str] = []
    if report is None or report.empty:
        return "No data fetched."
    # "not_run" rows are placeholders for symbols whose worker never finished (or that were
    # supplied twice) - count them separately so 100% coverage is provable, not implied
    if "status" in report.columns:
        n = int(report.loc[report["status"] != "not_run", "status"].size)
    else:
        n = len(report)
    notrun = int((report["status"] == "not_run").sum()) if "status" in report.columns else 0
    ok = int((report["status"] == "ok").sum())
    lines.append(f"Symbols attempted : {n}" + (f"  (+{notrun} placeholder rows)" if notrun else ""))
    lines.append(f"Usable            : {ok}  ({ok/n:.1%})")
    for st, c in report["status"].value_counts().items():
        lines.append(f"   - {st}: {c}")
    thr = int((report["status"] == "throttled").sum())
    if thr:
        lines.append(f"   ! {thr} symbol(s) were rate-limited (429), NOT unavailable. "
                     f"Raise cfg.throttle_pause / lower cfg.max_workers and re-run; "
                     f"already-cached symbols are reused, so the retry is cheap.")
    if "source" in report.columns:
        lines.append("Sources: " + ", ".join(f"{k}={v}" for k, v in report["source"].value_counts().items()))
    rows = pd.to_numeric(report.get("rows", pd.Series(dtype=float)), errors="coerce")
    if rows.notna().any():
        lines.append(f"Rows/stock: median {rows.median():.0f}, min {rows.min():.0f}, max {rows.max():.0f}")
    stale = pd.to_numeric(report.get("issues", pd.Series(dtype=object)).astype(str).str.contains("stale"), errors="coerce")
    if stale.notna().any():
        lines.append(f"Staleness warnings: {int(stale.sum())}")
    lines.append(f"Minimum weekly bars required: {cfg.min_weekly_bars} (~{cfg.min_weekly_bars*5} trading days)")
    return "\n".join(lines)


def save_load_report(report: pd.DataFrame, data: Dict[str, pd.DataFrame], out_dir: str) -> None:
    os.makedirs(out_dir, exist_ok=True)
    report.to_csv(os.path.join(out_dir, "data_fetch_report.csv"), index=False)
    summary = {s: int(len(d)) for s, d in data.items()}
    pd.DataFrame({"Symbol": list(summary), "daily_rows": list(summary.values())}).to_csv(
        os.path.join(out_dir, "data_rowcounts.csv"), index=False)
'''
with open(os.path.join(PKG_DIR, 'nse_data.py'), 'w', encoding='utf-8') as _f:
    _f.write(_SRC_NSE_DATA)
print('nse_data.py           ', os.path.getsize(os.path.join(PKG_DIR, 'nse_data.py')), 'bytes  ·  Data layer — resilient Yahoo + NSE fetching')

nse_data.py            59541 bytes  ·  Data layer — resilient Yahoo + NSE fetching


In [ ]:
# Detection engine — the weekly sweep rule   ·   embedded verbatim 2026-08-29, module starts at the next line
_SRC_SWEEP_ENGINE = r'''
"""
sweep_engine.py — Weekly liquidity-sweep detection for NSE swing trading (buy side).

A "sweep" (a.k.a. stop hunt / turtle soup / liquidity grab) on the weekly timeframe:

   prior weekly swing low = a low lower than the N lows on its left and N on its right
                            (N = swing_strength), OR a fresh low of the lookback window
                            / the 52-week low — both are allowed by design.
   sweep candle           = a COMPLETED weekly candle whose LOW trades below that swing
                            low AND whose CLOSE prints back above that swing low.
   rejection evidence     = a genuine lower wick: close in the upper part of the range,
                            wick dominant vs. body, bar not absurdly wide.

Only candles satisfying all three are flagged, then ranked by a composite Setup Score so
the output is tradable rather than merely "technically true". Nothing here repaints:
the in-progress week is dropped before any signal is computed.
"""

from __future__ import annotations

from dataclasses import dataclass, field, asdict
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

EPS = 1e-9


# --------------------------------------------------------------------------------------
# Parameters
# --------------------------------------------------------------------------------------
@dataclass
class Params:
    # --- structure ---
    swing_strength: int = 2                  # fractal half-width on weekly bars (2 => 5-bar fractal)
    swing_lookback: int = 26                 # weeks to search for swing lows to be swept
    min_swing_age: int = 1                   # swing must be >= this many weeks before the sweep
    max_sweep_bars_ago: int = 1              # 1 => sweep must be on the last completed weekly candle
    include_sweep_on_swing_bar: bool = False  # allow the sweep bar to BE the (unconfirmed) swing bar
    allow_fresh_low: bool = True             # sweeping a fresh low / 52w low is a valid sweep
    require_old_swing: bool = False          # stricter: only confirmed swing lows, no fresh-low sweeps

    # --- sweep geometry ---
    min_close_above_pct: float = 0.0         # close must be >= swept_level * (1 + x)
    min_wick_ratio: float = 0.34             # lower_wick / weekly range
    min_wick_body_mult: float = 1.15         # lower_wick >= k * |body|
    min_close_in_range: float = 0.50         # (close - low) / range
    max_range_of_close: float = 0.12         # reject monster bars (gaps/ex-rights/buybacks)
    min_depth_pct: float = 0.004             # penetration >= 0.4% of level (tick noise filter)
    max_depth_pct: float = 0.16              # but not a violent collapse through the level
    max_depth_atr_mult: float = 3.0          # and not more than this many weekly ATRs

    # --- confirmation ---
    require_green_close: bool = False        # extra strictness: bullish weekly candle
    min_vol_ratio_soft: float = 0.55         # below this, the sweep is penalised, not rejected

    # --- filters (liquidity / tradability) ---
    min_price: float = 5.0
    min_turnover_lakh: float = 75.0          # median daily turnover (Rs lakh) over last 60 sessions
    hard_illiquid_mult: float = 0.35         # < this * min_turnover_lakh => rejected outright
    min_weekly_bars: int = 104
    exclude_locked_weeks: bool = True        # weeks containing a limit/circuit-locked day
    max_locked_weeks_26: int = 1

    # --- score weights (sum ~ 100) ---
    weights: Dict[str, float] = field(default_factory=lambda: {
        "wick": 22.0, "close_position": 16.0, "depth": 12.0, "volume": 10.0,
        "trend": 12.0, "proximity": 10.0, "structure": 10.0, "recency": 8.0,
    })
    min_score: float = 0.0

    def describe(self) -> Dict[str, object]:
        return {k: v for k, v in asdict(self).items() if k != "weights"}


# --------------------------------------------------------------------------------------
# Weekly resampling
# --------------------------------------------------------------------------------------
def _to_naive_ist(df: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, format="mixed")
    if getattr(df.index, "tz", None) is not None:
        df.index = df.index.tz_convert("Asia/Kolkata").tz_localize(None)
    df.index = pd.to_datetime(df.index).normalize()
    return df


CIRCUIT_LO, CIRCUIT_HI = 0.095, 0.115      # the 10% band; 20% band derived from it


def resample_weekly(daily: pd.DataFrame, keep_partial: bool = False,
                    today: Optional[pd.Timestamp] = None) -> pd.DataFrame:
    """
    Daily bars (DatetimeIndex) -> weekly OHLCV stamped with that week's Monday.

    The unfinished week is dropped unless keep_partial=True, so a signal can never appear
    and disappear mid-candle. Also derives LockedDays (limit/circuit/no-trade days).
    """
    cols = ["Open", "High", "Low", "Close", "Volume", "Days", "Partial", "LockedDays"]
    if daily is None or len(daily) == 0:
        return pd.DataFrame(columns=cols)
    df = daily.copy()
    df = _to_naive_ist(df)
    for c in ("Open", "High", "Low", "Close"):
        if c not in df.columns:
            return pd.DataFrame(columns=cols)
        df[c] = pd.to_numeric(df[c], errors="coerce")
    if "Volume" not in df.columns:
        df["Volume"] = 0.0
    df["Volume"] = pd.to_numeric(df["Volume"], errors="coerce").fillna(0.0)
    df = df.dropna(subset=["Open", "High", "Low", "Close"]).sort_index()
    if df.empty:
        return pd.DataFrame(columns=cols)

    dow = df.index.dayofweek                       # Mon=0 .. Sun=6
    week_start = df.index - pd.to_timedelta(dow, unit="D")
    g = df.groupby(week_start, sort=True)
    wk = pd.DataFrame({
        "Open": g["Open"].first(),
        "High": g["High"].max(),
        "Low": g["Low"].min(),
        "Close": g["Close"].last(),
        "Volume": g["Volume"].sum(),
        "Days": g["Open"].size(),
    })
    wk.index = pd.to_datetime(wk.index)
    wk.index.name = "WeekStart"

    last_daily = pd.Timestamp(df.index[-1]).normalize()
    # A week is finished once its Friday session exists (that holds for a half-week ending
    # Thursday too, when Friday was a holiday). If the newest bar is Mon-Thu and "now" is
    # already past it, the week is still forming -> keep it out of the signal tape so the
    # screener cannot repaint mid-candle.
    n = len(wk)
    if n:
        last_days = float(wk["Days"].iloc[-1])
        has_friday = bool((df.index[-1].dayofweek >= 4) and (wk.index[-1] + pd.Timedelta(days=4))
                          in set(pd.DatetimeIndex(df.index).normalize()))
        far_past = (today is not None) and (pd.Timestamp(today).normalize() > last_daily
                                           + pd.Timedelta(days=6))
        forming = (not has_friday) and (last_days < 4.0) and (not far_past)
        wk["Partial"] = forming & (wk.index == wk.index[-1])
    else:
        wk["Partial"] = []
    if not keep_partial:
        wk = wk[~wk["Partial"].to_numpy(dtype=bool)]

    # A week is "locked" only if a day actually *could not trade*: a flat, zero-volume
    # session (halted / no-trade), or a day that pinned at an NSE circuit limit (10%/20%)
    # with a frozen price. A merely large move is NOT a lock - NSE small caps move 5% daily
    # all the time, and treating that as untradeable rejects the whole universe.
    tol = 1e-6
    rng = (df["High"] - df["Low"]).to_numpy(dtype=float)
    close = df["Close"].to_numpy(dtype=float)
    prev = np.concatenate([[np.nan], close[:-1]])
    vol = df["Volume"].to_numpy(dtype=float)
    # a session that traded nobody at all AND did not move (real halt / no-trade band)
    flat = (rng <= np.maximum(tol, 1e-4 * np.abs(close))) & (vol <= 0)
    with np.errstate(divide="ignore", invalid="ignore"):
        ret = np.nan_to_num(np.abs(close / prev - 1.0), nan=0.0, posinf=0.0)
    pinned = rng <= np.maximum(tol, 5e-4 * np.abs(close))          # open=high=low=close
    ret = np.nan_to_num(ret, nan=0.0)
    at_limit = (((ret >= CIRCUIT_LO) & (ret <= CIRCUIT_HI)) |
                ((ret >= 2 * CIRCUIT_LO) & (ret <= 2 * CIRCUIT_HI + 0.005)))
    circ = np.asarray(at_limit & pinned, dtype=bool)
    circ = np.nan_to_num(circ.astype(float)).astype(bool)
    lock_daily = pd.Series(np.asarray(flat | circ, dtype=float),
                           index=pd.DatetimeIndex(week_start))
    wk["LockedDays"] = (lock_daily.groupby(level=0).sum().reindex(wk.index).fillna(0.0).to_numpy())
    out = wk.reindex(columns=cols)
    out["LockedDays"] = out["LockedDays"].fillna(0.0)
    out["Volume"] = out["Volume"].fillna(0.0)
    return out


# --------------------------------------------------------------------------------------
# Indicators
# --------------------------------------------------------------------------------------
def atr(df: pd.DataFrame, n: int = 14) -> pd.Series:
    h, l, c = df["High"], df["Low"], df["Close"]
    pc = c.shift(1)
    tr = pd.concat([(h - l).abs(), (h - pc).abs(), (l - pc).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1.0 / n, adjust=False, min_periods=1).mean()


def swing_lows(df: pd.DataFrame, strength: int = 2) -> np.ndarray:
    """Boolean mask: True where LOW is a confirmed fractal swing low.

    Rule: low[i] <= min(low[i-N:i]) and low[i] < min(low[i+1:i+1+N]).
    Strict on the right (so a plateau of equal lows is booked once, on its first bar),
    tolerant on the left. Bars inside the last N bars cannot be confirmed yet — a *fresh
    low* being taken out is handled separately as the "new swing low" case.
    """
    n = int(strength)
    low = pd.to_numeric(df["Low"], errors="coerce").to_numpy(dtype=float)
    N = len(low)
    out = np.zeros(N, dtype=bool)
    if n < 1 or N < 2 * n + 1:
        return out
    for i in range(n, N - n):
        cur = low[i]
        if not np.isfinite(cur):
            continue
        left, right = low[i - n:i], low[i + 1:i + 1 + n]
        if not (np.isfinite(left).all() and np.isfinite(right).all()):
            continue
        if cur <= left.min() and cur < right.min():
            out[i] = True
    return out


# --------------------------------------------------------------------------------------
# Scoring helpers
# --------------------------------------------------------------------------------------
def _ramp(x: float, lo: float, hi: float) -> float:
    if x is None or not np.isfinite(x):
        return 0.0
    return float(np.clip((x - lo) / max(hi - lo, EPS), 0.0, 1.0))


def _decay(x: float, half: float) -> float:
    if x is None or not np.isfinite(x):
        return 0.0
    return float(max(0.0, 1.0 - x / max(half, EPS)))


def _band(x: float, a: float, b: float, c: float, d: float) -> float:
    """Trapezoid membership: 0 below a, ramps to 1 at b, holds to c, decays to 0 at d."""
    if x is None or not np.isfinite(x):
        return 0.0
    if x <= a:
        return 0.0
    if x < b:
        return float((x - a) / max(b - a, EPS))
    if x <= c:
        return 1.0
    if x >= d:
        return 0.0
    return float((d - x) / max(d - c, EPS))


# --------------------------------------------------------------------------------------
# Core detector
# --------------------------------------------------------------------------------------
@dataclass
class SweepHit:
    sweep_idx: int
    swing_idx: int
    level: float
    date_sweep: pd.Timestamp
    date_swing: pd.Timestamp
    open_: float
    high: float
    low: float
    close: float
    volume: float
    wick_ratio: float
    lower_wick: float
    body: float
    range_: float
    close_in_range: float
    depth_abs: float
    depth_pct: float
    close_above_pct: float
    atr_at_sweep: float
    fresh_low: bool
    is_low_52w: bool
    swing_age_weeks: int
    n_pools: int
    volume_ratio: float
    green: bool
    sma20: float
    sma50: float
    bars_since_sweep: int
    score: float
    drivers: Dict[str, float] = field(default_factory=dict)


def _levels_to_sweep(wk: pd.DataFrame, i: int, p: Params) -> List[Tuple[float, int, bool]]:
    """Candidate liquidity pools for bar i: (level, swing_bar_index, is_fresh_low)."""
    low = wk["Low"].to_numpy(dtype=float)
    j0 = max(0, i - p.swing_lookback)
    out: List[Tuple[float, int, bool]] = []

    if p.swing_strength >= 1:
        mask = swing_lows(wk, p.swing_strength)
        for j in np.flatnonzero(mask):
            j = int(j)
            if j0 <= j <= i - p.min_swing_age:
                out.append((float(low[j]), j, False))
        if p.include_sweep_on_swing_bar:
            # opt-in only: allow the sweep bar to BE the bar that prints the new low
            if np.isfinite(low[i]) and low[i] > 0:
                out.append((float(low[i]), int(i), True))

    def _add(level: float, bar: int, fresh: bool) -> None:
        """Register a candidate pool. A level created BY the test bar itself (i.e. the bar
        simply printing a new low) is only considered when the user opts in — otherwise every
        new-low candle would trivially 'sweep' its own low and the screen would be all noise."""
        if bar == i and not p.include_sweep_on_swing_bar:
            return
        if np.isfinite(level) and level > 0:
            out.append((float(level), int(bar), bool(fresh)))

    # fresh low of the lookback window (a NEW swing low being taken out)
    win = low[j0:i]
    if p.allow_fresh_low and len(win) and np.isfinite(win).all():
        _add(win.min(), int(np.argmin(win)) + j0, True)

    # 52-week low — the most obvious resting liquidity on a weekly chart. Same "new low"
    # family as the running window low, so it is governed by allow_fresh_low, and it is
    # capped by swing_lookback so far-away pools are not silently traded.
    if p.allow_fresh_low:
        s52 = max(0, i - min(52, max(p.swing_lookback, p.min_swing_age + 1)))
        w52 = low[s52:i]
        if len(w52) and np.isfinite(w52).all():
            _add(w52.min(), int(np.argmin(w52)) + s52, True)

    # de-duplicate, keep the earliest swing bar per level
    uniq: Dict[float, Tuple[float, int, bool]] = {}
    for lev, j, fresh in out:
        if not np.isfinite(lev) or lev <= 0:
            continue
        key = round(lev, 6)
        if key not in uniq:
            uniq[key] = (lev, j, fresh)
        else:
            old_lev, old_j, old_fresh = uniq[key]
            # same price: keep the confirmed-swing flag and the oldest bar (one pool, one entry)
            uniq[key] = (old_lev, min(old_j, j), bool(old_fresh) and bool(fresh))
    return list(uniq.values())


def evaluate_bar(wk: pd.DataFrame, i: int, p: Params) -> Optional[SweepHit]:
    """Test one weekly bar for a completed, rejected sweep of a prior swing low."""
    if i < 1 or i >= len(wk):
        return None
    lo = float(wk["Low"].iloc[i]); hi = float(wk["High"].iloc[i])
    cl = float(wk["Close"].iloc[i]); op = float(wk["Open"].iloc[i])
    if not all(np.isfinite(x) for x in (lo, hi, cl, op)) or cl <= 0 or lo <= 0:
        return None
    rng = hi - lo
    if rng <= 0:
        return None
    body = abs(cl - op)
    lower_wick = min(op, cl) - lo
    wick_ratio = lower_wick / rng
    close_in_range = (cl - lo) / rng

    # --- rejection evidence -------------------------------------------------------------
    if wick_ratio < p.min_wick_ratio:
        return None
    if lower_wick < p.min_wick_body_mult * body:
        return None
    if close_in_range < p.min_close_in_range:
        return None
    if rng / cl > p.max_range_of_close:
        return None
    if p.require_green_close and cl <= op:
        return None

    # --- which pool did it take? -------------------------------------------------------
    a = float(atr(wk, 14).iloc[i])
    if not np.isfinite(a) or a <= 0:
        a = rng
    best: Optional[Tuple[float, int, bool, float]] = None
    n_pools = 0
    for lev, j, fresh in _levels_to_sweep(wk, i, p):
        depth = lev - lo
        if depth <= 0:
            continue                                   # level not actually swept
        if depth < p.min_depth_pct * lev:              # tick-noise floor
            continue                                   # tick noise, not a sweep
        if depth > p.max_depth_pct * lev:
            continue                                   # collapsed through the level
        if depth > p.max_depth_atr_mult * a:
            continue
        if cl <= lev * (1.0 + p.min_close_above_pct):
            continue                                   # must CLOSE back above the level
        if fresh and not p.allow_fresh_low:
            continue
        if p.require_old_swing and fresh:
            continue
        n_pools += 1
        # the deepest pool taken out is the one that matters (one candle clearing several
        # stacked lows = a much bigger liquidity grab than nudging the nearest one)
        if best is None or lev < best[0]:
            best = (lev, j, fresh, depth)
    if best is None:
        return None
    lev, j, fresh, depth = best

    s52w = max(0, i - 52)
    w52 = wk["Low"].to_numpy(dtype=float)[s52w:i]
    is52 = bool(len(w52) and np.isfinite(w52).all() and lev <= float(w52.min()) * (1 + 1e-6))

    vol = float(wk["Volume"].iloc[i]) if "Volume" in wk.columns else np.nan
    vseq = pd.to_numeric(wk["Volume"], errors="coerce").to_numpy(dtype=float)
    vma = np.nanmean(vseq[max(0, i - 20):i]) if i > 0 else np.nan
    vol_ratio = float(vol / vma) if np.isfinite(vma) and vma > 0 and np.isfinite(vol) else 1.0

    sma20 = float(wk["Close"].rolling(20, min_periods=8).mean().iloc[i])
    sma50 = float(wk["Close"].rolling(50, min_periods=25).mean().iloc[i])
    drivers = _score_drivers(wk, i, dict(
        low=lo, close=cl, level=lev, depth=depth, atr=a, wick_ratio=wick_ratio,
        close_in_range=close_in_range, green=(cl > op), vol_ratio=vol_ratio,
        sma20=sma20, sma50=sma50, bars_since_sweep=int(len(wk) - 1 - i),
        fresh=fresh, is52=is52, swing_age=int(i - j),
    ), p)
    w = p.weights
    score = float(np.clip(sum(w.get(k, 0.0) * v for k, v in drivers.items()), 0.0, 100.0))

    return SweepHit(
        sweep_idx=int(i), swing_idx=int(j), level=float(lev),
        date_sweep=pd.Timestamp(wk.index[i]), date_swing=pd.Timestamp(wk.index[j]),
        open_=op, high=hi, low=lo, close=cl, volume=vol if np.isfinite(vol) else np.nan,
        wick_ratio=float(wick_ratio), lower_wick=float(lower_wick), body=float(body),
        range_=float(rng), close_in_range=float(close_in_range),
        depth_abs=float(depth), depth_pct=float(depth / lev),
        close_above_pct=float(cl / lev - 1.0), atr_at_sweep=float(a),
        fresh_low=bool(fresh), is_low_52w=is52, swing_age_weeks=int(i - j), n_pools=int(n_pools),
        volume_ratio=float(vol_ratio), green=bool(cl > op),
        sma20=sma20 if np.isfinite(sma20) else np.nan,
        sma50=sma50 if np.isfinite(sma50) else np.nan,
        bars_since_sweep=int(len(wk) - 1 - i), score=score,
        drivers={k: round(float(v), 3) for k, v in drivers.items()},
    )


def _score_drivers(wk: pd.DataFrame, i: int, f: dict, p: Params) -> Dict[str, float]:
    d: Dict[str, float] = {}
    d["wick"] = _ramp(f["wick_ratio"], p.min_wick_ratio, 0.75)
    d["close_position"] = _ramp(f["close_in_range"], p.min_close_in_range, 0.95) * (1.15 if f["green"] else 0.85)
    depth_pct = f["depth"] / max(f["level"], EPS)
    d["depth"] = _band(depth_pct, 0.002, 0.012, 0.075, 0.16)
    vr = f["vol_ratio"]
    if vr >= 1.0:
        d["volume"] = _ramp(vr, 0.9, 2.6)
    else:
        d["volume"] = max(0.0, _ramp(vr, p.min_vol_ratio_soft, 1.0) * 0.7)
    c, s20, s50 = f["close"], f["sma20"], f["sma50"]
    tr = 0.0
    if np.isfinite(s20) and s20 > 0:
        tr += 0.6 if c >= s20 else (0.3 if c >= s20 * 0.985 else 0.0)
    if np.isfinite(s50) and s50 > 0:
        tr += 0.4 if s20 >= s50 * 0.98 else 0.0
    d["trend"] = min(1.0, tr)
    risk_pct = (c - f["low"]) / max(c, EPS)
    d["proximity"] = _band(risk_pct, 0.005, 0.02, 0.10, 0.17)
    st = 0.90 if f["is52"] else (0.72 if not f["fresh"] else 0.62)
    lo52 = float(np.nanmin(wk["Low"].to_numpy(dtype=float)[max(0, i - 52):i + 1])) if i > 0 else np.nan
    if np.isfinite(lo52) and lo52 > 0:
        st *= 1.0 - 0.55 * _ramp(c / lo52 - 1.0, 0.35, 1.10)      # don't chase far above the base
    d["structure"] = float(np.clip(st, 0.0, 1.0))
    d["recency"] = 1.0 if f["bars_since_sweep"] == 0 else max(0.0, 1.0 - 0.28 * f["bars_since_sweep"])
    return d


def find_sweeps(wk: pd.DataFrame, p: Params) -> List[SweepHit]:
    """Scan the most recent `max_sweep_bars_ago` completed weekly bars; return hits best-first."""
    n = len(wk)
    if n < max(p.min_weekly_bars, 2 * p.swing_strength + 3):
        return []
    last_i = n - 1
    span = max(1, int(p.max_sweep_bars_ago))
    first_i = max(p.swing_strength + 1, last_i - span + 1)
    hits = [h for h in (evaluate_bar(wk, i, p) for i in range(first_i, last_i + 1)) if h is not None]
    if not hits:
        return []
    hits = [h for h in hits if h.score >= p.min_score]
    # newest first for ties: a 3-week-old sweep must never outrank today's
    return sorted(hits, key=lambda h: (-round(h.score, 4), h.bars_since_sweep))


def sweep_in_window(wk: pd.DataFrame, p: Params) -> Tuple[Optional[SweepHit], Optional[dict]]:
    """
    Primary entry point of the screener. Returns (hit, near_miss_dict).

    Honours `max_sweep_bars_ago`: 1 = "the sweep must be on the newest closed week" (the
    strict, tradable-now reading); N = "a sweep completed within the last N weeks".
    A hit anywhere in the window suppresses near-miss reporting for that same window, so the
    watchlist never duplicates a row that already qualified.
    """
    n = len(wk)
    if n == 0:
        return None, None
    hits = find_sweeps(wk, p)
    if hits:
        return hits[0], None                       # already newest-first, best-score-first
    span = max(1, int(p.max_sweep_bars_ago))
    for i in range(n - 1, max(n - 1 - span, p.swing_strength) - 1, -1):
        nm = near_miss(wk, p, i)
        if nm is not None:
            return None, nm
    return None, None


# kept for the tests / older calls: window of exactly one bar == "the last bar"
def sweep_on_last_bar(wk: pd.DataFrame, p: Params, window: int = 1
                      ) -> Tuple[Optional[SweepHit], Optional[dict]]:
    if window != max(1, int(p.max_sweep_bars_ago)):
        p = Params(**{**p.__dict__, "max_sweep_bars_ago": int(window)})
    return sweep_in_window(wk, p)


def near_miss(wk: pd.DataFrame, p: Params, i: Optional[int] = None) -> Optional[dict]:
    """
    Bars that DID sweep a level and closed above it but failed a quality gate.
    Reported as a watchlist, and as proof the detector is not blind.
    """
    n = len(wk)
    i = (n - 1) if i is None else i
    if n < 3 or i < 1:
        return None
    lo = float(wk["Low"].iloc[i]); hi = float(wk["High"].iloc[i])
    cl = float(wk["Close"].iloc[i]); op = float(wk["Open"].iloc[i])
    if not all(np.isfinite(x) for x in (lo, hi, cl, op)) or cl <= 0:
        return None
    rng = hi - lo
    if rng <= 0:
        return None
    swept = [lev for lev, j, fresh in _levels_to_sweep(wk, i, p)
             if lo < lev and cl > lev * (1.0 + p.min_close_above_pct)]
    if not swept:
        return None
    level = min(swept)
    lower_wick = min(op, cl) - lo
    reasons = []
    if lower_wick / rng < p.min_wick_ratio:
        reasons.append("wick too small")
    if lower_wick < p.min_wick_body_mult * abs(cl - op):
        reasons.append("body larger than wick")
    if (cl - lo) / rng < p.min_close_in_range:
        reasons.append("close weak / not in upper half")
    if rng / cl > p.max_range_of_close:
        reasons.append("weekly range too wide")
    if p.require_green_close and cl <= op:
        reasons.append("red candle")
    if not reasons:
        return None
    return {
        "Week": str(pd.Timestamp(wk.index[i]).date()), "Close": round(cl, 2),
        "SweptLevel": round(level, 2), "SweepLow": round(lo, 2),
        "WickRatio%": round(lower_wick / rng * 100, 1),
        "CloseInRange%": round((cl - lo) / rng * 100, 1),
        "FailReason": ", ".join(reasons),
    }


# --------------------------------------------------------------------------------------
# Universe filters + master screen
# --------------------------------------------------------------------------------------
def turnover_lakh(df: pd.DataFrame, n: int = 60) -> float:
    """Median daily rupee turnover over the last `n` sessions, in lakh (1 lakh = 1e5)."""
    if df is None or len(df) == 0:
        return np.nan
    v = pd.to_numeric(df["Volume"], errors="coerce").to_numpy(dtype=float)
    c = pd.to_numeric(df["Close"], errors="coerce").to_numpy(dtype=float)
    m = np.nanmedian(v[-n:] * c[-n:]) / 1e5
    return float(m) if np.isfinite(m) else np.nan


def locked_weeks(wk: pd.DataFrame, n: int = 26) -> int:
    if wk is None or "LockedDays" not in wk.columns or len(wk) == 0:
        return 0
    arr = pd.to_numeric(wk["LockedDays"], errors="coerce").fillna(0).to_numpy(dtype=float)
    return int((arr[-n:] >= 1).sum())


def screen_all(data: Dict[str, pd.DataFrame], p: Params,
                universe: Optional[pd.DataFrame] = None,
                keep_near_miss: bool = True,
                progress_cb: Optional[Callable[[int, int], None]] = None,
                ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Screen every cached symbol.

    Returns
    -------
    hits       : confirmed setups (sorted by score, newest first)
    near_miss  : candles that swept + closed above but failed a quality gate
    diagnostics: one row per screened symbol. status is
                 hit | near_miss | no_setup | insufficient_weekly_bars | price_below_min
                 | illiquid | circuit_locked | error:*, i.e. nothing is left unexplained
    """
    hit_rows, miss_rows, diag_rows = [], [], []
    uni = None
    if universe is not None and len(universe) and "Symbol" in universe.columns:
        uni = universe.drop_duplicates("Symbol").set_index("Symbol")

    for k, (sym, daily) in enumerate(data.items()):
        if progress_cb is not None and k % 50 == 0:
            progress_cb(k, len(data))
        # "ok" here read as "passed every filter" in the status table; "hit" is unambiguous
        rec: Dict[str, object] = {"Symbol": sym, "status": "hit"}
        try:
            wk = resample_weekly(daily)
        except Exception as exc:  # defensive: never let one symbol kill the run
            diag_rows.append({**rec, "status": f"error:{type(exc).__name__}"})
            continue
        rec["WeeklyBars"] = int(len(wk))
        if len(wk) < p.min_weekly_bars:
            diag_rows.append({**rec, "status": "insufficient_weekly_bars"})
            continue
        cl = float(wk["Close"].iloc[-1])
        rec["Close"] = round(cl, 2)
        rec["LastWeek"] = str(pd.Timestamp(wk.index[-1]).date())
        rec["TurnoverLakh"] = round(turnover_lakh(daily), 2) if np.isfinite(turnover_lakh(daily)) else np.nan
        rec["LockedWeeks26"] = locked_weeks(wk)

        if cl < p.min_price:
            diag_rows.append({**rec, "status": "price_below_min"}); continue
        to = rec["TurnoverLakh"]
        if np.isfinite(to) and to < p.min_turnover_lakh * p.hard_illiquid_mult:
            diag_rows.append({**rec, "status": "illiquid"}); continue
        if p.exclude_locked_weeks and rec["LockedWeeks26"] > p.max_locked_weeks_26:
            diag_rows.append({**rec, "status": "circuit_locked"}); continue

        hit, miss = sweep_in_window(wk, p)
        if hit is not None:
            hit_rows.append(_row_from_hit(sym, hit, wk, uni, rec))
        elif keep_near_miss and miss is not None:
            miss_rows.append({"Symbol": sym, **miss})
        if hit is None and miss is not None:
            rec["status"] = "near_miss"
        elif hit is None:
            rec["status"] = "no_setup"
        else:
            rec["status"] = "hit"
        diag_rows.append(rec)

    hits = pd.DataFrame(hit_rows)
    if not hits.empty:
        hits = hits.sort_values(["Score", "BarsSinceSweep"], ascending=[False, True]).reset_index(drop=True)
        hits.insert(0, "Rank", np.arange(1, len(hits) + 1))
    return hits, pd.DataFrame(miss_rows), pd.DataFrame(diag_rows)


def _row_from_hit(sym: str, h: SweepHit, wk: pd.DataFrame, uni: Optional[pd.DataFrame],
                  rec: Dict[str, object]) -> dict:
    i = h.sweep_idx
    stop = h.low * 0.995
    risk = h.close - stop
    seg = pd.to_numeric(wk["High"].iloc[max(0, i - 26): i + 1], errors="coerce").to_numpy(dtype=float)
    seg = seg[np.isfinite(seg)]
    resist = float(seg.max()) if len(seg) else np.nan
    row = {
        "Symbol": sym,
        "Company": (str(uni.at[sym, "Company"]) if uni is not None and "Company" in uni.columns
                    and sym in uni.index else ""),
        "Close": round(h.close, 2),
        "SweepWeek": str(pd.Timestamp(h.date_sweep).date()),
        "SweptSwingLow": round(h.level, 2),
        "SweepLow": round(h.low, 2),
        "SwingLowMade": str(pd.Timestamp(h.date_swing).date()),
        "SwingAgeW": int(h.swing_age_weeks),
        "PoolsTaken": int(h.n_pools),
        "ClosedAboveBy%": round(h.close_above_pct * 100, 2),
        "SweepDepth%": round(h.depth_pct * 100, 2),
        "WickRatio%": round(h.wick_ratio * 100, 1),
        "CloseInRange%": round(h.close_in_range * 100, 1),
        "Green": bool(h.green),
        "SweepType": ("52-week-low sweep" if h.is_low_52w
                      else "new-low sweep" if h.fresh_low else "prior swing-low sweep"),
        "LowPoolRef": (f"lowest of {h.n_pools} pools taken" if h.n_pools > 1 else ""),
        "Is52wLow": bool(h.is_low_52w),
        "VolRatio": round(h.volume_ratio, 2),
        "ATR(14w)": round(h.atr_at_sweep, 2),
        "SMA20w": round(h.sma20, 2) if np.isfinite(h.sma20) else np.nan,
        "vsSMA20w%": round((h.close / h.sma20 - 1) * 100, 2) if np.isfinite(h.sma20) and h.sma20 else np.nan,
        "BarsSinceSweep": int(len(wk) - 1 - i),
        "Risk%": round(risk / h.close * 100, 2),
        "Stop": round(stop, 2),
        "Target2R": round(h.close + 2 * risk, 2),
        "Target3R": round(h.close + 3 * risk, 2),
        "Resistance26w": round(resist, 2) if np.isfinite(resist) else np.nan,
        "UpsideToRes%": round((resist / h.close - 1) * 100, 2) if np.isfinite(resist) else np.nan,
        "TurnoverLakh": rec.get("TurnoverLakh", np.nan),
        "Score": round(h.score, 1),
    }
    for k, v in h.drivers.items():
        row[f"S_{k}"] = v
    return row


# --------------------------------------------------------------------------------------
# Top-N ranking of the universe
# --------------------------------------------------------------------------------------
def rank_top(data: Dict[str, pd.DataFrame], quotes: Optional[pd.DataFrame] = None,
             top_n: int = 1000, index_tiers: Optional[pd.DataFrame] = None,
             prefer_index_tier: bool = False) -> pd.DataFrame:
    """
    Decide the "top N" of the universe.

    Ranking key, in order of preference:
      1. real market cap      - only when the optional quote endpoint cooperated
      2. index tier, turnover - when prefer_index_tier=True (mirrors the official Nifty
                                50/100/200/500 ordering)
      3. turnover             - default; always available, and it is the honest ordering
                                for a *liquidity* screener
    Stocks outside every index still get a fair shot at the top 1000 in modes 1 and 3.

    `index_tiers` : DataFrame[Symbol, IndexTier, Industry] from nse_data.load_index_membership
    Returns columns: Rank, Symbol, Close, TurnoverLakh, IndexTier, Industry, rank_basis
    """
    rows = []
    for sym, df in data.items():
        c = pd.to_numeric(df.get("Close"), errors="coerce").to_numpy(dtype=float)
        px = float(c[-1]) if len(c) and np.isfinite(c[-1]) else np.nan
        rows.append({"Symbol": sym, "Close": px, "TurnoverLakh": turnover_lakh(df)})
    tab = pd.DataFrame(rows)
    if tab.empty:
        return tab
    tab["RankLakh"] = tab["TurnoverLakh"].rank(ascending=False, method="min")

    if index_tiers is not None and len(index_tiers):
        it = index_tiers.drop_duplicates("Symbol").set_index("Symbol")
        tab["IndexTier"] = tab["Symbol"].map(it["IndexTier"]) if "IndexTier" in it.columns else None
        tab["Industry"] = tab["Symbol"].map(it["Industry"]) if "Industry" in it.columns else None
        order = {k: n for n, k in enumerate(
            ["Nifty50", "Nifty100", "Nifty200", "Nifty500", "NiftyMidcap150", "NiftySmallcap250"])}
        tab["_tier"] = tab["IndexTier"].map(lambda x: order.get(x, 99)).fillna(99).astype(int)
    else:
        tab["IndexTier"] = None
        tab["Industry"] = None
        tab["_tier"] = 99

    basis = "index_tier+turnover"
    if quotes is not None and len(quotes) and "MarketCap" in quotes.columns:
        tab = tab.merge(quotes[["Symbol", "MarketCap"]].drop_duplicates("Symbol"),
                        on="Symbol", how="left")
        if float(tab["MarketCap"].notna().mean() or 0) > 0.6:
            basis = "market_cap"
    else:
        tab["MarketCap"] = np.nan

    if basis == "market_cap":
        keys, asc = ["_mcap", "_tier", "Symbol"], [False, True, True]
    elif prefer_index_tier:
        keys, asc = ["_tier", "RankLakh", "Symbol"], [True, True, True]
    else:
        keys, asc = ["RankLakh", "_tier", "Symbol"], [True, True, True]
    tab["_mcap"] = tab["MarketCap"]
    tab["RankLakh"] = tab["RankLakh"].fillna(1e12)
    tab = tab.sort_values(keys, ascending=asc, na_position="last")
    tab = tab.head(int(top_n)).drop(columns=["_mcap", "_tier", "RankLakh"],
                                    errors="ignore").reset_index(drop=True)
    tab.insert(0, "Rank", np.arange(1, len(tab) + 1))
    tab["rank_basis"] = basis
    return tab


# --------------------------------------------------------------------------------------
# Backtest of the same rule (parameter sanity, not a promise)
# --------------------------------------------------------------------------------------
def backtest(data: Dict[str, pd.DataFrame], p: Params, horizon_weeks: int = 8,
             r_mult: float = 2.0, cost_pct: float = 0.15,
             max_symbols: Optional[int] = None) -> pd.DataFrame:
    """
    Same geometry rule, replayed on the whole history of each symbol:
    entry at the open of the week AFTER the sweep, stop = sweep low -0.5%,
    target = entry + r_mult*risk, otherwise exit at the close of week `horizon_weeks`.
    """
    rows = []
    items = list(data.items())
    if max_symbols:
        items = items[:max_symbols]
    for sym, daily in items:
        wk = resample_weekly(daily)
        if len(wk) < p.min_weekly_bars:
            continue
        lo_a = wk["Low"].to_numpy(float); hi_a = wk["High"].to_numpy(float)
        op_a = wk["Open"].to_numpy(float); cl_a = wk["Close"].to_numpy(float)
        atr_a = atr(wk, 14).to_numpy(float)
        sw = np.flatnonzero(swing_lows(wk, p.swing_strength))
        for i in range(p.swing_strength + 1, len(wk) - 1):
            rng = hi_a[i] - lo_a[i]
            if rng <= 0 or not np.isfinite(cl_a[i]) or cl_a[i] <= 0:
                continue
            lw = min(op_a[i], cl_a[i]) - lo_a[i]
            if lw / rng < p.min_wick_ratio or lw < p.min_wick_body_mult * abs(cl_a[i] - op_a[i]):
                continue
            if (cl_a[i] - lo_a[i]) / rng < p.min_close_in_range or rng / cl_a[i] > p.max_range_of_close:
                continue
            j0 = max(0, i - p.swing_lookback)
            cands = [float(wk["Low"].iloc[j]) for j in sw[(sw >= j0) & (sw <= i - p.min_swing_age)]]
            win = lo_a[j0:i]
            if len(win) and np.isfinite(win).all() and p.allow_fresh_low:
                cands.append(float(win.min()))
            a = atr_a[i] if np.isfinite(atr_a[i]) and atr_a[i] > 0 else rng
            ok = []
            for lev in cands:
                if not np.isfinite(lev) or lev <= 0:
                    continue
                depth = lev - lo_a[i]
                if depth <= 0 or cl_a[i] <= lev * (1 + p.min_close_above_pct):
                    continue
                if depth < p.min_depth_pct * lev or depth > p.max_depth_pct * lev:
                    continue
                if depth > p.max_depth_atr_mult * a:
                    continue
                ok.append(lev)
            if not ok:
                continue
            level = max(ok)
            entry = op_a[i + 1]
            stop = lo_a[i] * 0.995
            risk = entry - stop
            if not np.isfinite(entry) or entry <= 0 or risk <= 0:
                continue
            target = entry + r_mult * risk
            j_end = min(len(wk) - 1, i + 1 + max(1, int(horizon_weeks)))
            out, brk = cl_a[j_end], "time"
            for j in range(i + 1, j_end + 1):
                if lo_a[j] <= stop:
                    out, brk = stop, "stop"; break
                if hi_a[j] >= target:
                    out, brk = target, "target"; break
            ret = (out / entry - 1) * 100 - cost_pct
            rows.append({"Symbol": sym, "SweepWeek": str(pd.Timestamp(wk.index[i]).date()),
                         "SweptLevel": round(level, 2), "Entry": round(entry, 2),
                         "Stop": round(stop, 2), "Exit": round(out, 2), "Outcome": brk,
                         "Return%": round(ret, 2), "R": round((out - entry) / risk, 2),
                         "BarsToExit": int(j_end - i - 1)})
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["Symbol", "SweepWeek"]).reset_index(drop=True)
    return df


def summarize_backtest(trades: pd.DataFrame) -> dict:
    if trades is None or trades.empty:
        return {"trades": 0}
    r = trades["Return%"].to_numpy(float)
    r = r[np.isfinite(r)]
    if not len(r):
        return {"trades": 0}
    pos, neg = r[r > 0], r[r < 0]
    return {
        "trades": int(len(r)),
        "win_rate_%": round(100 * float((r > 0).mean()), 1),
        "avg_return_%": round(float(np.mean(r)), 2),
        "median_return_%": round(float(np.median(r)), 2),
        "avg_R": round(float(trades["R"].mean()), 2),
        "hit_target_%": round(100 * float((trades["Outcome"] == "target").mean()), 1),
        "hit_stop_%": round(100 * float((trades["Outcome"] == "stop").mean()), 1),
        "expired_%": round(100 * float((trades["Outcome"] == "time").mean()), 1),
        "profit_factor": round(float(min(99.0, np.sum(pos) / max(EPS, abs(np.sum(neg))))), 2),
        "p5_%": round(float(np.percentile(r, 5)), 2),
        "p95_%": round(float(np.percentile(r, 95)), 2),
    }


# --------------------------------------------------------------------------------------
# Chart payload (consumed by the notebook's plotting cell)
# --------------------------------------------------------------------------------------
def chart_frames(daily: pd.DataFrame, p: Params, weeks_to_show: int = 60
                 ) -> Tuple[pd.DataFrame, dict]:
    """Weekly tail for candlestick plotting + the swing/sweep annotations to overlay."""
    wk = resample_weekly(daily)
    if wk.empty:
        return wk, {"hit": None, "swings": [], "first_index": 0}
    hits = find_sweeps(wk, Params(**{**asdict(p)}))
    start = max(0, len(wk) - weeks_to_show)
    tail = wk.iloc[start:].copy()
    info: dict = {"hit": None, "swings": [], "first_index": int(start), "levels": []}
    mask = swing_lows(wk, p.swing_strength)
    for j in np.flatnonzero(mask):
        if j >= start:
            info["swings"].append({"pos": int(j - start), "date": str(pd.Timestamp(wk.index[j]).date()),
                                   "low": float(wk["Low"].iloc[j])})
    # the level(s) currently worth sweeping
    i = len(wk) - 1
    for lev, j, fresh in _levels_to_sweep(wk, i, p):
        info["levels"].append({"level": round(lev, 2), "date": str(pd.Timestamp(wk.index[j]).date()),
                               "fresh": bool(fresh)})
    cand = [h for h in hits if h.sweep_idx == i] or hits
    if cand:
        h = cand[0]
        d = asdict(h)
        d["sweep_pos"] = int(h.sweep_idx - start)
        d.pop("date_sweep", None); d.pop("date_swing", None)
        info["hit"] = d
    return tail, info
'''
with open(os.path.join(PKG_DIR, 'sweep_engine.py'), 'w', encoding='utf-8') as _f:
    _f.write(_SRC_SWEEP_ENGINE)
print('sweep_engine.py       ', os.path.getsize(os.path.join(PKG_DIR, 'sweep_engine.py')), 'bytes  ·  Detection engine — the weekly sweep rule')

sweep_engine.py        40467 bytes  ·  Detection engine — the weekly sweep rule


In [ ]:
# Orchestration — universe, ranking, exports, charts, diagnostics   ·   embedded verbatim 2026-08-29, module starts at the next line
_SRC_SCREENER = r'''
"""
screener.py — orchestration, presentation and exports for the NSE weekly sweep screener.

Everything network/CSV/plot-specific lives here so the two "engine" modules stay pure:
    nse_data.py     -> universe + bars (fetching is the hard part, so it is isolated here)
    sweep_engine.py -> weekly resampling, swing/sweep detection, scoring, backtest
    screener.py     -> the pipeline, tables, charts, Excel/CSV export, alerts  (this file)
"""

from __future__ import annotations

import json
import os
import re
import shutil
import sys
import time
import traceback
import zipfile
from datetime import datetime, timedelta, timezone
from typing import Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import nse_data as ND
import sweep_engine as SE
from nse_data import FetchConfig, YahooClient
from sweep_engine import Params

IST = timezone(timedelta(hours=5, minutes=30))


def now_ist() -> datetime:
    return datetime.now(IST)


def today_ts() -> pd.Timestamp:
    return pd.Timestamp(now_ist().date())


# ======================================================================================
# 1. Setup helpers (Colab-friendly)
# ======================================================================================
def ensure_packages(packages: Sequence[str] = ("pandas", "numpy", "requests", "tqdm",
                                               "openpyxl", "plotly")) -> List[str]:
    """
    Return the list of packages that are missing (the notebook pip-installs those and then
    asks for one runtime restart, which is the #1 cause of 'my colab notebook broke').
    """
    import importlib
    missing = []
    alias = {"plotly": "plotly", "openpyxl": "openpyxl", "yfinance": "yfinance"}
    for pkg in packages:
        mod = alias.get(pkg, pkg)
        try:
            importlib.import_module(mod)
        except Exception:
            missing.append(pkg)
    return missing


def mount_drive_if_requested(enabled: bool, mount_path: str = "/content/drive"):
    if not enabled:
        return None
    try:
        from google.colab import drive  # type: ignore
        drive.mount(mount_path)
        return mount_path
    except Exception as exc:
        print(f"Drive mount skipped ({type(exc).__name__}: {exc}) - using local disk.")
        return None


def snapshot_workspace(workdir: str, dest_dir: str) -> Optional[str]:
    """Zip the notebook workspace (code + outputs, no raw bars) so a Colab disconnect is not fatal."""
    if not os.path.isdir(workdir):
        return None
    os.makedirs(os.path.dirname(dest_dir) or ".", exist_ok=True)
    keep = {".py", ".csv", ".md", ".txt", ".html", ".xlsx", ".json"}
    tmp = dest_dir + ".tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(workdir):
            dirs[:] = [d for d in dirs if d not in {"cache_nse", "__pycache__", ".ipynb_checkpoints"}]
            for f in files:
                if os.path.splitext(f)[1].lower() in keep:
                    fp = os.path.join(root, f)
                    zf.write(fp, os.path.relpath(fp, workdir))
    shutil.move(tmp, dest_dir)
    return dest_dir


# ======================================================================================
# 2. Universe
# ======================================================================================
def build_universe(cfg: FetchConfig, client: YahooClient, log: Callable[[str], None] = print
                   ) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """NSE equity universe + index tier/industry map. Fails loudly but usefully."""
    log("· NSE full equity list (archives.nseindia.com, cached to disk)…")
    try:
        uni = ND.load_universe_full_list(cfg.cache_dir, client, log=log)
    except Exception as exc:
        log(f"  ! full list unavailable ({exc})")
        uni = pd.DataFrame(columns=["Symbol", "Company", "Series", "ISIN", "FaceValue",
                                    "DateOfListing"])
    log(f"  → {len(uni)} tradeable equities after excluding weird series")
    log("· Index membership (Nifty 50/100/200/500/Midcap150/Smallcap250)…")
    idx = ND.load_index_membership(cfg.cache_dir, log=log)
    if len(idx):
        log(f"  → {idx['Symbol'].nunique()} symbols carry an index tier "
            f"({idx['IndexTier'].value_counts().to_dict()})")
    if len(uni) and len(idx):
        uni = uni.merge(idx[["Symbol", "Industry", "IndexTier"]], on="Symbol", how="left")
    else:
        uni["Industry"] = np.nan
        uni["IndexTier"] = np.nan
    return uni, idx


# ======================================================================================
# 3. The main pipeline
# ======================================================================================
class Screener:
    """
    One object that owns config, cache, the fetched data and every output artefact, so you
    can re-screen with new parameters in seconds (no re-download) and re-export anything.
    """

    def __init__(self, params: Params, fetch_cfg: FetchConfig, out_dir: str = "out",
                 universe_size: int = 1000, prefer_index_tier: bool = False,
                 use_quotes: bool = False, verbose: bool = True):
        self.p = params
        self.cfg = fetch_cfg
        self.cfg.use_quotes = bool(use_quotes)
        self.out_dir = out_dir
        self.universe_size = int(universe_size)
        self.prefer_index_tier = bool(prefer_index_tier)
        self.verbose = verbose
        os.makedirs(out_dir, exist_ok=True)
        self.log_lines: List[str] = []
        self.client = YahooClient(self.cfg, log=self._log)
        self.universe: Optional[pd.DataFrame] = None
        self.index_tiers: Optional[pd.DataFrame] = None
        self.data: Dict[str, pd.DataFrame] = {}
        self.report: Optional[pd.DataFrame] = None
        self.ranking: Optional[pd.DataFrame] = None
        self.hits: Optional[pd.DataFrame] = None
        self.near: Optional[pd.DataFrame] = None
        self.diag: Optional[pd.DataFrame] = None
        self.finished_at: Optional[str] = None

    # -- logging -----------------------------------------------------------------------
    def _log(self, msg: str) -> None:
        line = str(msg)
        self.log_lines.append(line)
        if self.verbose:
            print(line, flush=True)

    # -- steps -------------------------------------------------------------------------
    def prepare_universe(self) -> "Screener":
        self._log("=" * 78)
        self._log(f"NSE WEEKLY LIQUIDITY-SWEEP SCREENER   {now_ist():%Y-%m-%d %H:%M} IST")
        self._log("=" * 78)
        self.universe, self.index_tiers = build_universe(self.cfg, self.client, self._log)
        return self

    def fetch(self, symbols: Optional[Sequence[str]] = None, progress: bool = True) -> "Screener":
        todo = list(symbols) if symbols is not None else list(self.universe["Symbol"])
        self._log(f"· Fetching {len(todo)} symbols, {self.cfg.history_years}y of daily bars "
                  f"(cache first, {self.cfg.max_workers} workers, "
                  f">={self.cfg.request_sleep}s between calls)…")
        t0 = time.time()
        self.data, self.report = ND.fetch_daily_bulk(todo, self.cfg, self.client,
                                                      progress=progress and self.verbose)
        got = len(self.data)
        attempted = len(self.report) if self.report is not None else 0
        self._log(f"  → usable {got} / {len(todo)} in {time.time()-t0:.0f}s "
                  f"(each symbol accounted for: {attempted == len(todo)})")
        if attempted != len(todo):
            self._log(f"  ! coverage gap: {len(todo)-attempted} symbol(s) have no report row")
        self.coverage = {"asked": len(todo), "reported": attempted, "usable": got}
        self._log("\n" + ND.data_health(self.report, self.cfg))
        ND.save_load_report(self.report, self.data, self.out_dir)
        return self

    def rank_universe(self) -> "Screener":
        """Pick the liquid top-N from everything that has usable history."""
        quotes = None
        if self.cfg.use_quotes:
            try:
                quotes = self.client.quote_batch(list(self.data))
            except Exception as exc:
                self._log(f"  market-cap enrichment skipped ({type(exc).__name__})")
        if quotes is None or len(quotes) == 0:
            quotes = ND.build_quotes_from_meta(self.cfg.cache_dir)
        self.ranking = SE.rank_top(self.data, quotes, top_n=self.universe_size,
                                   index_tiers=self.index_tiers,
                                   prefer_index_tier=self.prefer_index_tier)
        self._log(f"· Ranked universe: {len(self.ranking)} symbols kept "
                  f"(basis={self.ranking['rank_basis'].iloc[0] if len(self.ranking) else '-'})")
        return self

    def screen(self, subset: Optional[Sequence[str]] = None) -> "Screener":
        """Run the sweep detector. subset=None → screen the ranked top-N."""
        if subset is None:
            if self.ranking is None:
                self.rank_universe()
            syms = list(self.ranking["Symbol"]) if (self.ranking is not None and "Symbol" in self.ranking.columns) else []
            self._log(f"· Screening {len(syms)} symbols for completed weekly sweeps "
                      f"(swing_strength={self.p.swing_strength}, lookback={self.p.swing_lookback}w, "
                      f"wick≥{self.p.min_wick_ratio:.0%}, close in upper "
                      f"{self.p.min_close_in_range:.0%}+)…")
        else:
            syms = list(subset)
            self._log(f"· Screening {len(syms)} supplied symbols…")
        data = {s: self.data[s] for s in syms if s in self.data}
        uni = self.universe if self.universe is not None else None
        t0 = time.time()
        self.hits, self.near, self.diag = SE.screen_all(data, self.p, universe=uni)
        self._log(f"  → {len(self.hits)} setups, {len(self.near)} near-misses in "
                  f"{time.time()-t0:.1f}s")
        self.finished_at = now_ist().strftime("%Y-%m-%d %H:%M IST")
        return self

    def screen_all_cached(self) -> "Screener":
        """Ignore the top-N cut and screen literally every cached symbol."""
        return self.screen(subset=list(self.data))

    def set_params(self, **kw) -> "Screener":
        bad = {k: v for k, v in kw.items() if not hasattr(self.p, k)}
        if bad:
            raise KeyError(f"Unknown parameter(s): {list(bad)} — valid ones: "
                           f"{sorted(f for f in self.p.__dataclass_fields__)}")
        self.p = Params(**{**self.p.__dict__, **kw})
        return self

    # -- outputs -----------------------------------------------------------------------
    def summary_frame(self, top: Optional[int] = None) -> pd.DataFrame:
        if self.hits is None or self.hits.empty:
            return pd.DataFrame()
        cols = [c for c in ["Rank", "Symbol", "Company", "Close", "SweepWeek", "BarsSinceSweep",
                            "SweepType", "SweptSwingLow", "SweepLow", "SwingLowMade", "SwingAgeW",
                            "PoolsTaken", "LowPoolRef", "ClosedAboveBy%", "SweepDepth%",
                            "WickRatio%", "CloseInRange%", "Green", "Is52wLow", "VolRatio",
                            "ATR(14w)", "SMA20w", "vsSMA20w%", "Risk%", "Stop", "Target2R",
                            "Target3R", "Resistance26w", "UpsideToRes%", "TurnoverLakh",
                            "Industry", "IndexTier", "Score"] if c in self.hits.columns]
        out = self.hits[cols].copy()
        return out.head(top) if top else out

    def export(self, prefix: str = "sweep") -> Dict[str, str]:
        """Write results.csv / results.xlsx / near_misses.csv / diagnostics.csv + params.json."""
        stamp = now_ist().strftime("%Y-%m-%d")
        paths: Dict[str, str] = {}
        os.makedirs(self.out_dir, exist_ok=True)
        hits = self.summary_frame()
        if not hits.empty:                      # NaN -> empty cell in the exports
            hits = hits.astype(object).where(pd.notna(hits), "")
        p_csv = os.path.join(self.out_dir, f"{prefix}_results_{stamp}.csv")
        (hits if not hits.empty else pd.DataFrame(columns=["Symbol"])).to_csv(p_csv, index=False)
        paths["results_csv"] = p_csv

        p_xlsx = os.path.join(self.out_dir, f"{prefix}_results_{stamp}.xlsx")
        try:
            with pd.ExcelWriter(p_xlsx, engine="openpyxl") as xw:
                (hits if not hits.empty else pd.DataFrame({"note": ["no setups today"]})
                 ).to_excel(xw, sheet_name="Setups", index=False)
                (self.near if self.near is not None and not self.near.empty
                 else pd.DataFrame(columns=["Symbol"])).to_excel(xw, sheet_name="NearMisses",
                                                                  index=False)
                (self.diag if self.diag is not None else pd.DataFrame()).to_excel(
                    xw, sheet_name="ScreenedUniverse", index=False)
                (self.report if self.report is not None else pd.DataFrame()).to_excel(
                    xw, sheet_name="DataQuality", index=False)
                self.params_frame().to_excel(xw, sheet_name="Params", index=False)
                if self.ranking is not None:
                    self.ranking.to_excel(xw, sheet_name="UniverseRanking", index=False)
            paths["results_xlsx"] = p_xlsx
        except Exception as exc:
            self._log(f"  ! xlsx export skipped ({type(exc).__name__}: {exc})")

        if self.near is not None and len(self.near):
            self.near = self.near.rename(columns={"Week": "SweepWeek"})
        if self.near is not None:
            p = os.path.join(self.out_dir, f"{prefix}_near_misses_{stamp}.csv")
            self.near.to_csv(p, index=False)
            paths["near_miss_csv"] = p
        if self.diag is not None:
            p = os.path.join(self.out_dir, f"{prefix}_diagnostics_{stamp}.csv")
            self.diag.to_csv(p, index=False)
            paths["diagnostics_csv"] = p
        p = os.path.join(self.out_dir, f"{prefix}_params_{stamp}.json")
        with open(p, "w") as f:
            json.dump({"generated_ist": self.finished_at, "params": self.p.describe(),
                       "weights": self.p.weights, "fetch": {
                           k: v for k, v in self.cfg.__dict__.items()},
                       "counts": {"universe": int(len(self.universe) if self.universe is not None else 0),
                                  "fetched": len(self.data),
                                  "screened": int(len(self.diag) if self.diag is not None else 0),
                                  "hits": int(len(self.hits) if self.hits is not None else 0)}},
                      f, indent=2, default=str)
        paths["params_json"] = p
        self._log("· Wrote " + ", ".join(os.path.basename(v) for v in paths.values()))
        return paths

    def params_frame(self) -> pd.DataFrame:
        d = {"# of bars since last sweep": self.p.max_sweep_bars_ago}
        d.update(self.p.describe())
        rows = [{"parameter": k, "value": v} for k, v in d.items()]
        rows += [{"parameter": f"weight:{k}", "value": v} for k, v in self.p.weights.items()]
        return pd.DataFrame(rows)

    def telegram_alert(self, bot_token: str, chat_id: str, top_n: int = 10) -> str:
        """Optional push. Keep the token out of the saved notebook: paste at run time."""
        if not bot_token or not chat_id:
            return "skipped (no token/chat id)"
        text = format_message(self.summary_frame(top_n), self.finished_at or "")
        try:
            import requests
            r = requests.post(f"https://api.telegram.org/bot{bot_token}/sendMessage",
                              data={"chat_id": chat_id, "text": text,
                                    "disable_web_page_preview": "true"}, timeout=20)
            return f"telegram {r.status_code}"
        except Exception as exc:
            return f"telegram failed: {type(exc).__name__}"

    # -- analysis ---------------------------------------------------------------------
    def selftest(self) -> pd.DataFrame:
        """
        Detector sanity check on synthetic tapes (no network). Proves the rule works before
        you trust the live run, and shows which filters a real ticker failed.
        """
        import tests_fixtures as TF
        import types
        mod = types.ModuleType("m")
        rows = []
        for name in sorted(d for d in dir(TF) if d.startswith("test_")):
            fn = getattr(TF, name)
            if not callable(fn):
                continue
            try:
                fn()
                rows.append({"check": name, "result": "PASS", "detail": ""})
            except Exception as exc:
                rows.append({"check": name, "result": "FAIL",
                             "detail": f"{type(exc).__name__}: {str(exc)[:160]}"})
        df = pd.DataFrame(rows)
        n = len(df)
        ok = int((df["result"] == "PASS").sum())
        self._log(f"· Self-test: {ok}/{n} checks passed")
        return df

    def diagnose(self, symbols: Sequence[str]) -> pd.DataFrame:
        """
        'Why is X not in my list?' — answers with the exact gate it failed, week by week.
        """
        rows = []
        for sym in symbols:
            daily = self.data.get(str(sym).upper())
            row: Dict[str, object] = {"Symbol": str(sym).upper()}
            if daily is None or len(daily) == 0:
                rep = self.report
                st = "not fetched"
                if rep is not None and "Symbol" in rep.columns:
                    got = rep.loc[rep["Symbol"].astype(str).str.upper() == str(sym).upper(),
                                  "status"]
                    if len(got):
                        st = str(got.iloc[0])
                why = {"no_data": "Yahoo has no bars for this ticker (delisted / renamed / too new)",
                       "throttled": "rate-limited (429) - re-run, the cache makes it cheap",
                       "insufficient_history": "listed too recently for a weekly screen",
                       "illiquid_history": "too many zero-volume days to trust a weekly candle",
                       "stale_data": "history stops well before today",
                       "not_run": "not part of the last fetch (see DataQuality sheet)"}.get(st, st)
                rows.append({**row, "verdict": f"no usable bars - {why}",
                             "weekly_bars": 0, "status": st})
                continue
            wk = SE.resample_weekly(daily)
            row["weekly_bars"] = int(len(wk))
            row["last_week"] = str(pd.Timestamp(wk.index[-1]).date()) if len(wk) else ""
            row["close"] = round(float(wk["Close"].iloc[-1]), 2) if len(wk) else np.nan
            row["turnover_lakh"] = round(SE.turnover_lakh(daily), 1)
            row["locked_weeks_26"] = SE.locked_weeks(wk)
            if len(wk) < self.p.min_weekly_bars:
                row["verdict"] = f"rejected: only {len(wk)} weekly bars (need {self.p.min_weekly_bars})"
                rows.append(row); continue
            i = len(wk) - 1
            lo, hi, cl, op = (float(wk['Low'].iloc[i]), float(wk['High'].iloc[i]),
                              float(wk['Close'].iloc[i]), float(wk['Open'].iloc[i]))
            rng = hi - lo
            if rng <= 0:
                row["verdict"] = "rejected: zero-range last week"
                rows.append(row); continue
            lw = min(op, cl) - lo
            levels = SE._levels_to_sweep(wk, i, self.p)
            row["last_low"] = round(lo, 2)
            row["last_close"] = round(cl, 2)
            row["swing_lows_in_range"] = ", ".join(
                f"{lev:g}@{str(pd.Timestamp(wk.index[j]).date())}" for lev, j, fr in levels[:4]
            ) or "none"
            row["wick_ratio"] = round(lw / rng, 3)
            row["close_in_range"] = round((cl - lo) / rng, 3)
            best = SE.evaluate_bar(wk, i, self.p)
            if best is not None:
                row["verdict"] = f"SWEEP: took {best.level:g}, closed {best.close_above_pct*100:.2f}% above, score {best.score:.0f}"
            else:
                nm = SE.near_miss(wk, self.p, i)
                if nm:
                    row["verdict"] = f"near miss: {nm['FailReason']}"
                else:
                    swept = [lev for lev, j, fr in levels if lo < lev and cl > lev]
                    if not levels:
                        row["verdict"] = "no weekly swing low in lookback window (raise swing_lookback)"
                    elif not swept:
                        row["verdict"] = ("last week did not undercut any swing low"
                                          if lo >= min([l for l, _, _ in levels], default=1e18)
                                          else "undercut the low but CLOSED BELOW it (breakdown, not sweep)")
                    else:
                        row["verdict"] = "swept & reclaimed but failed a quality gate"
            rows.append(row)
        return pd.DataFrame(rows)

    def backtest_rule(self, horizon_weeks: int = 8, r_mult: float = 2.0,
                      max_symbols: Optional[int] = None) -> Tuple[pd.DataFrame, dict]:
        self._log(f"· Backtesting the same rule: {horizon_weeks}w horizon, {r_mult}R target…")
        tr = SE.backtest(self.data, self.p, horizon_weeks=horizon_weeks, r_mult=r_mult,
                         max_symbols=max_symbols)
        summ = SE.summarize_backtest(tr)
        self._log("  → " + json.dumps(summ, default=str))
        if len(tr):
            tr.to_csv(os.path.join(self.out_dir, "backtest_trades.csv"), index=False)
        return tr, summ

    def grid_scan(self, grid: Optional[Dict[str, Sequence[float]]] = None,
                  sample: int = 250, horizon_weeks: int = 8) -> pd.DataFrame:
        """
        Small parameter grid (backtest-driven) so you can see how sensitive the rule is
        instead of trusting one arbitrary setting.
        """
        grid = grid or {"swing_strength": [1, 2, 3], "min_wick_ratio": [0.25, 0.34, 0.45],
                        "max_sweep_bars_ago": [1, 3]}
        keys = list(grid)
        combos = [{}]
        for k in keys:
            combos = [{**c, k: v} for c in combos for v in grid[k]]
        subset = dict(list(self.data.items())[:sample])
        rows = []
        for c in combos:
            p = Params(**{**self.p.__dict__, **c})
            tr = SE.backtest(subset, p, horizon_weeks=horizon_weeks)
            s = SE.summarize_backtest(tr)
            rows.append({**c, **{k: v for k, v in s.items() if k in
                                 ("trades", "win_rate_%", "avg_return_%", "avg_R", "profit_factor")}})
        df = pd.DataFrame(rows)
        if not df.empty:
            df = df.sort_values("avg_return_%", ascending=False).reset_index(drop=True)
            df.to_csv(os.path.join(self.out_dir, "param_grid.csv"), index=False)
        return df


# ======================================================================================
# 4. Presentation
# ======================================================================================
def _fmt(v, nd: int = 2) -> str:
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return ""
    if isinstance(v, (int, np.integer)):
        return f"{v:d}"
    if isinstance(v, (float, np.floating)):
        return f"{v:,.{nd}f}" if abs(v) >= 0.01 or v == 0 else f"{v:.4f}"
    return str(v)


def format_message(hits: pd.DataFrame, stamp: str, max_rows: int = 12) -> str:
    """Compact text summary (terminal or Telegram)."""
    head = f"NSE weekly sweep screen — {stamp}"
    if hits is None or hits.empty:
        return f"{head}\nNo completed weekly sweeps in the screened universe today."
    lines = [head, f"{len(hits)} setup(s); top {min(max_rows, len(hits))}:", ""]
    for _, r in hits.head(max_rows).iterrows():
        lines.append(
            f"{int(r.get('Rank', 0)):>2}. {r['Symbol']:<16} {r['Close']:>9}  "
            f"swept {r['SweptSwingLow']:>9} → low {r['SweepLow']:>9} "
            f"(+{r['ClosedAboveBy%']:.2f}%)  wick {r['WickRatio%']:.0f}%  "
            f"score {r['Score']:.0f}  stop {r['Stop']:>9}  2R {r['Target2R']:>9}")
    lines += ["", "Weekly candles only. Confirm on the chart; risk-manage every entry."]
    return "\n".join(lines)


def style_results(hits: pd.DataFrame, top: int = 40):
    """Colour-graded display frame (works in Colab; degrades to a plain table elsewhere)."""
    if hits is None or hits.empty:
        print("No setups to display.")
        return pd.DataFrame()
    show = hits.head(top).copy()
    num_cols = [c for c in show.columns if pd.api.types.is_numeric_dtype(show[c])]
    display_cols = [c for c in ["Rank", "Symbol", "Company", "Close", "SweepWeek", "BarsSinceSweep",
                                "SweepType", "SweptSwingLow", "SweepLow", "ClosedAboveBy%",
                                "SweepDepth%", "WickRatio%", "CloseInRange%", "Green",
                                "PoolsTaken", "VolRatio", "vsSMA20w%", "Risk%", "Stop",
                                "Target2R", "Target3R", "UpsideToRes%", "TurnoverLakh", "Score",
                                "Industry"]
                    if c in show.columns]
    out = show[display_cols]
    try:
        def _score_css(v):
            try:
                v = float(v)
            except Exception:
                return ""
            if v >= 75:
                return "background:#1b5e20;color:#fff;font-weight:600"
            if v >= 60:
                return "background:#2e7d32;color:#fff"
            if v >= 45:
                return "background:#f9a825;color:#000"
            return "background:#455a64;color:#fff"

        sty = (out.style
               .format(lambda x: "" if pd.isna(x) else (f"{x:.2f}" if isinstance(x, float) else str(x)))
               .map(_score_css, subset=["Score"] if "Score" in out.columns else None)
               .hide(axis="index"))
        if "WickRatio%" in out.columns:
            sty = sty.background_gradient(subset=["WickRatio%"], cmap="Greens")
        if "VolRatio" in out.columns:
            sty = sty.background_gradient(subset=["VolRatio"], cmap="Oranges")
        return sty
    except Exception as exc:
        print(f"(styling unavailable: {type(exc).__name__}) — showing raw table")
        return out


def plot_setup(screener: Screener, symbol: str, weeks: int = 60, engine: str = "plotly",
               height: int = 460):
    """
    Weekly candlestick with the swept liquidity line, the sweep candle highlighted, the
    swing-low markers and the suggested stop / 2R target. Returns a figure object.
    """
    daily = screener.data.get(symbol.upper())
    if daily is None or len(daily) == 0:
        raise KeyError(f"{symbol}: no cached bars (was it fetched?)")
    tail, info = SE.chart_frames(daily, screener.p, weeks_to_show=weeks)
    if tail.empty:
        raise RuntimeError(f"{symbol}: empty weekly tape after resampling")
    x = list(range(len(tail)))
    lbl = [str(pd.Timestamp(d).date()) for d in tail.index]

    if engine == "plotly":
        import plotly.graph_objects as go
        fig = go.Figure()
        fig.add_trace(go.Candlestick(x=x, open=tail["Open"], high=tail["High"],
                                     low=tail["Low"], close=tail["Close"], name="Weekly",
                                     increasing_line_color="#26a69a",
                                     decreasing_line_color="#ef5350"))
        hit = info.get("hit") or {}
        if hit:
            lev = float(hit["level"])
            fig.add_hline(y=lev, line=dict(color="#ffd54f", width=2, dash="dash"),
                          annotation_text=f"swept swing low {lev:g}", annotation_font_color="#ffd54f")
            p = int(hit.get("sweep_pos", -1))
            if 0 <= p < len(tail):
                fig.add_trace(go.Scatter(x=[p], y=[tail["Low"].iloc[p]], mode="markers",
                                         marker=dict(symbol="triangle-down", size=15,
                                                     color="#ffca28",
                                                     line=dict(width=1, color="#000")),
                                         name="sweep candle", showlegend=True))
        for sw in info.get("swings", []):
            if 0 <= sw["pos"] < len(tail):
                fig.add_trace(go.Scatter(x=[sw["pos"]], y=[sw["low"]], mode="markers",
                                         marker=dict(symbol="circle", size=6,
                                                     color="rgba(66,165,245,0.85)"),
                                         name="swing low", showlegend=False,
                                         hovertext=[f"{sw['date']} low {sw['low']:g}"]))
        close = float(tail["Close"].iloc[-1])
        if hit:
            stop = float(hit["low"]) * 0.995
            risk = close - stop
            if risk > 0:
                fig.add_hline(y=stop, line=dict(color="#ef5350", width=1, dash="dot"),
                              annotation_text="stop", annotation_font_color="#ef5350")
                fig.add_hline(y=close + 2 * risk, line=dict(color="#66bb6a", width=1, dash="dot"),
                              annotation_text="2R", annotation_font_color="#66bb6a")
        fig.update_layout(
            title=(f"{symbol} — weekly | sweep {hit.get('date_sweep','')[:10] if hit else '—'}"
                   f" | score {hit.get('score', 0):.0f}" if hit else f"{symbol} — weekly"),
            xaxis=dict(tickmode="array", tickvals=x, ticktext=lbl, nticks=12,
                       rangeslider=dict(thickness=0)),
            yaxis=dict(title="₹", side="right"), height=height,
            margin=dict(l=8, r=8, t=44, b=8), paper_bgcolor="#111827", plot_bgcolor="#111827",
            font=dict(color="#e5e7eb", size=11), showlegend=False,
            xaxis_rangeslider_visible=False)
        return fig

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(12, height / 40), facecolor="#111827")
    ax.set_facecolor("#111827")
    for i, (o, h, l, c) in enumerate(zip(tail["Open"], tail["High"], tail["Low"], tail["Close"])):
        col = "#26a69a" if c >= o else "#ef5350"
        ax.plot([i, i], [l, h], color=col, lw=1)
        ax.add_patch(plt.Rectangle((i - 0.32, min(o, c)), 0.64, max(abs(c - o), h * 1e-4),
                                   color=col, alpha=0.9))
    hit = info.get("hit") or {}
    if hit:
        ax.axhline(hit["level"], ls="--", color="#ffd54f", lw=1.5,
                   label=f"swept swing low {hit['level']:g}")
        p = int(hit.get("sweep_pos", -1))
        if 0 <= p < len(tail):
            ax.plot(p, tail["Low"].iloc[p], "v", ms=12, color="#ffca28")
    ax.set_xticks(range(0, len(tail), max(1, len(tail) // 10)))
    ax.set_xticklabels([lbl[i] for i in range(0, len(tail), max(1, len(tail) // 10))],
                       rotation=0, fontsize=8, color="#e5e7eb")
    ax.tick_params(colors="#e5e7eb")
    if ax.get_legend_handles_labels()[0]:          # avoid matplotlib's empty-legend warning
        ax.legend(facecolor="#1f2937", labelcolor="#e5e7eb", fontsize=8)
    ax.set_title(f"{symbol} weekly — liquidity sweep", color="#e5e7eb")
    ax.grid(alpha=0.15, color="#6b7280")
    fig.tight_layout()
    return fig


def render_charts(screener: Screener, symbols: Sequence[str], weeks: int = 60,
                  engine: str = "plotly"):
    """Yield (symbol, figure) for each symbol that has data — caller decides how to show."""
    for sym in symbols:
        try:
            yield str(sym), plot_setup(screener, str(sym), weeks=weeks, engine=engine)
        except Exception as exc:
            print(f"  ! {sym}: chart skipped ({type(exc).__name__}: {exc})")


def gallery_html(screener: Screener, symbols: Sequence[str], weeks: int = 55) -> str:
    """Static matplotlib PNGs embedded in one HTML page (nice for saving/sharing)."""
    import base64
    import io
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    blocks = []
    for sym in symbols:
        try:
            fig = plot_setup(screener, str(sym), weeks=weeks, engine="mpl")
        except Exception as exc:
            continue
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=100, bbox_inches="tight",
                    facecolor=fig.get_facecolor())
        plt.close(fig)
        b64 = base64.b64encode(buf.getvalue()).decode()
        blocks.append(f'<div style="margin:14px 0"><h3 style="color:#111;margin:4px 0">{sym}'
                      f'</h3><img src="data:image/png;base64,{b64}" style="max-width:100%"></div>')
    return ("<html><body style='font-family:system-ui;background:#fff'>"
            + "".join(blocks) + "</body></html>")


def save_html_gallery(screener: Screener, symbols: Sequence[str], path: str,
                      weeks: int = 55) -> str:
    html = gallery_html(screener, symbols, weeks=weeks)
    with open(path, "w") as f:
        f.write(html)
    return path


# ======================================================================================
# 5. Console helpers
# ======================================================================================
def print_table(df: pd.DataFrame, max_width: int = 200) -> None:
    try:
        with pd.option_context("display.max_columns", None, "display.width", max_width,
                               "display.max_rows", 200, "display.float_format",
                               lambda v: f"{v:,.2f}"):
            print(df.to_string(index=False))
    except Exception:
        print(df.head(40).to_string())


def describe_criteria(p: Params) -> str:
    return f"""BUY SETUP — completed weekly candle only (the live week is excluded, so nothing repaints)
  1. a prior weekly swing low exists  : fractal low with {p.swing_strength} lower/higher bars on each side,
                                        searched up to {p.swing_lookback} weeks back. Fresh lows and the 52-week
                                        low also qualify ({'yes' if p.allow_fresh_low else 'no'}).
  2. that low was TAKEN OUT            : weekly LOW < swing low by ≥ {p.min_depth_pct*100:.2f}% of the level
                                        (and ≤ {p.max_depth_pct*100:.0f}% / {p.max_depth_atr_mult:.1f}× ATR, so a
                                        collapse through it is NOT called a sweep).
  3. price REJECTED back above         : weekly CLOSE > swept level{' + cushion' if p.min_close_above_pct else ''}
                                        (≥ {p.min_close_above_pct*100:.1f}% above, if set).
  4. a PROPER WICK is visible          : lower wick ≥ {p.min_wick_ratio:.0%} of the weekly range,
                                        wick ≥ {p.min_wick_body_mult:.2f}× the body, close in the top
                                        {p.min_close_in_range:.0%} of the range, range ≤ {p.max_range_of_close:.0%} of price.
  5. tradability                       : price ≥ ₹{p.min_price:g}, median daily turnover ≥
                                        ₹{p.min_turnover_lakh:g}L (hard floor at
                                        ₹{p.min_turnover_lakh*p.hard_illiquid_mult:.0f}L), ≤ {p.max_locked_weeks_26}
                                        circuit-locked week in the last 26, ≥ {p.min_weekly_bars} weekly bars.
  6. recency                           : the sweep must be ≤ {p.max_sweep_bars_ago} completed week(s) old.
  Setup Score (0-100) = wick {p.weights['wick']:.0f} + close position {p.weights['close_position']:.0f} + depth {p.weights['depth']:.0f}
                        + volume {p.weights['volume']:.0f} + trend {p.weights['trend']:.0f} + risk-size {p.weights['proximity']:.0f}
                        + structure {p.weights['structure']:.0f} + freshness {p.weights['recency']:.0f}"""
'''
with open(os.path.join(PKG_DIR, 'screener.py'), 'w', encoding='utf-8') as _f:
    _f.write(_SRC_SCREENER)
print('screener.py           ', os.path.getsize(os.path.join(PKG_DIR, 'screener.py')), 'bytes  ·  Orchestration — universe, ranking, exports, charts, diagnostics')

screener.py            36027 bytes  ·  Orchestration — universe, ranking, exports, charts, diagnostics


In [ ]:
# Self-test suite — 49 checks on synthetic tapes   ·   embedded verbatim 2026-08-29, module starts at the next line
_SRC_TESTS_FIXTURES = r'''
"""
tests_fixtures.py — synthetic-data unit tests for the weekly liquidity-sweep screener.

No network needed. These pin down the exact contract the user asked for:

  1. a prior weekly swing low (old OR freshly made) must be taken out,
  2. the sweep candle must CLOSE back above that swing low,
  3. a genuine lower wick must be visible,
  4. the whole thing must be on a COMPLETED weekly candle (no mid-candle repaint),
  plus: the liquidity/universe plumbing, ranking, and the backtest harness.

Run with:  python -m pytest tests_fixtures.py -q     (or just run the notebook's test cell)
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from sweep_engine import (Params, _levels_to_sweep, _row_from_hit, atr, backtest, chart_frames,
                          find_sweeps, locked_weeks, near_miss, rank_top, resample_weekly,
                          screen_all, summarize_backtest, sweep_in_window, swing_lows)


# --------------------------------------------------------------------------------------
# Tape builders (deterministic — no RNG, so the assertions are exact)
# --------------------------------------------------------------------------------------
def weekly(bars, start="2023-01-02") -> pd.DataFrame:
    """bars: list of (o,h,l,c[,vol]) -> weekly frame stamped on consecutive Mondays."""
    rows = [(b[0], b[1], b[2], b[3], b[4] if len(b) > 4 else 1_000_000) for b in bars]
    idx = pd.date_range(start=start, periods=len(rows), freq="7D")
    df = pd.DataFrame(rows, columns=["Open", "High", "Low", "Close", "Volume"], index=idx)
    df.index.name = "WeekStart"
    df["Days"] = 5
    df["Partial"] = False
    df["LockedDays"] = 0.0
    return df


def daily_frame(days, o, h, l, c, v):
    idx = pd.DatetimeIndex(pd.to_datetime(list(days)))
    return pd.DataFrame({"Open": o, "High": h, "Low": l, "Close": c, "Volume": v}, index=idx)


def as_daily(wk: pd.DataFrame) -> pd.DataFrame:
    """Weekly frame -> synthetic 5-session-per-week daily frame (plumbing tests only)."""
    rows, idx = [], []
    for ts, r in wk.iterrows():
        for k in range(5):
            d = ts + pd.Timedelta(days=k)
            if d.weekday() > 4:
                continue
            rows.append((r["Open"], r["High"], r["Low"], r["Close"], float(r["Volume"]) / 5.0))
            idx.append(d)
    return pd.DataFrame(rows, columns=["Open", "High", "Low", "Close", "Volume"],
                        index=pd.DatetimeIndex(idx))


def drift(n=26, start=150.0, step=1.5, vol=900_000):
    """Gentle deterministic downtrend with 2.5-wide bars; never the tape's floor."""
    return [(round(start - k * step, 2), round(start - k * step + 2.0, 2),
             round(start - k * step - 0.5, 2), round(start - (k + 1) * step, 2), vol)
            for k in range(n)]


PRE = [(101.0, 103.0, 100.0, 102.5),   # idx F: the OLD SWING LOW = 100.0
       (102.5, 105.0, 102.0, 104.5),   # bounce
       (104.5, 106.0, 103.5, 105.0),    # hold
       (105.0, 106.0, 104.0, 104.5),    # hold
       (104.5, 105.5, 103.0, 104.0),    # hold
       (104.0, 105.0, 103.2, 104.8)]    # hold


def sweep_tape(n_filler=26):
    """
    Drift down -> swing low 100 -> bounce/hold -> textbook sweep candle:
        open 101.0, high 103.0, low 96.0, close 102.5, 4x volume
    lower wick 5.0 (range 7.0 -> wick ratio 0.714, close in range 0.929, green).
    """
    return weekly(drift(n_filler) + PRE + [(101.0, 103.0, 96.0, 102.5, 4_000_000)])


def final_bar_tape(o, h, l, c, v=1_000_000, n_filler=26, extra_pre=None):
    """Same structure as sweep_tape but with a custom last bar."""
    return weekly(drift(n_filler) + PRE + (extra_pre or []) + [(o, h, l, c, v)])


def fresh_low_tape(n=28):
    """
    Monotonic decline -> only *fresh* lows exist (no confirmed swing to sweep).
    Last bar undercuts the running low by 7.5 and closes back above it.
    """
    bars = drift(n, start=320.0, step=8.0)
    last_low = bars[-1][2]
    bars.append((round(last_low + 0.5, 2), round(last_low + 2.0, 2),
                 round(last_low - 7.5, 2), round(last_low + 1.5, 2), 2_500_000))
    return weekly(bars)


def stale_sweep_tape(n_filler=26, n_after=4):
    """The sweep candle sits `n_after` weeks back, followed by quiet higher chop."""
    bars = drift(n_filler) + PRE[:5] + [(101.0, 103.0, 96.0, 102.5, 4_000_000)]
    px = 103.0
    for _ in range(n_after):
        bars.append((round(px, 2), round(px + 2, 2), round(px + 1, 2), round(px + 1.5, 2)))
        px += 1.5
    return weekly(bars)


def _row(h, wk):
    return _row_from_hit("TEST", h, wk, None, {"TurnoverLakh": 100.0})


# --------------------------------------------------------------------------------------
# Weekly resampling (must not repaint mid-candle)
# --------------------------------------------------------------------------------------
def test_resample_last_completed_week_is_kept():
    """The signal lives on the newest completed week: it must NEVER be dropped."""
    days = list(pd.date_range("2024-01-15", "2024-01-19", freq="B"))
    df = pd.DataFrame({"Open": np.ones(5) * 10, "High": 11.0, "Low": 9.0, "Close": 10.5,
                       "Volume": 100.0}, index=pd.DatetimeIndex(days))
    for td in ["2024-01-19", "2024-01-20", "2024-01-21", "2024-02-02"]:
        wk = resample_weekly(df, keep_partial=False, today=pd.Timestamp(td))
        assert len(wk) == 1, (td, wk)
    wk_auto = resample_weekly(df, keep_partial=False)          # today defaults to last bar
    assert len(wk_auto) == 1
    # mid-week data (no Friday yet) is treated as an unfinished candle
    partial = resample_weekly(df.iloc[:3], keep_partial=False, today=pd.Timestamp("2024-01-17"))
    assert len(partial) == 0
    assert len(resample_weekly(df.iloc[:3], keep_partial=True,
                               today=pd.Timestamp("2024-01-17"))) == 1


def test_resample_groups_by_calendar_week():
    days = list(pd.date_range("2024-01-01", "2024-01-19", freq="B"))   # 3 x Mon..Fri
    n = len(days)
    assert n == 15
    o = np.arange(1, n + 1, dtype=float)
    df = daily_frame(days, o, o + 1.0, o - 0.5, o + 0.5, np.full(n, 10.0))
    wk = resample_weekly(df, keep_partial=True, today=pd.Timestamp("2024-01-19"))
    assert len(wk) == 3, len(wk)
    assert wk["Open"].iloc[0] == 1.0 and wk["High"].iloc[0] == 6.0     # Fri 5 Jan: O=5 H=6
    assert wk["Open"].iloc[1] == 6.0                                   # Mon 8 Jan opens week 2
    assert wk["Close"].iloc[-1] == 15.5                                # Fri 19 Jan closes week 3
    assert wk["Volume"].sum() == 150.0
    assert wk.index[0] == pd.Timestamp("2024-01-01")                   # stamped on Monday
    assert not bool(wk["Partial"].iloc[-1])  # a Friday session exists -> week is complete


def test_resample_drops_partial_current_week():
    days = list(pd.date_range("2024-01-01", "2024-01-12", freq="B"))    # 2 complete weeks
    days += list(pd.date_range("2024-01-15", "2024-01-16", freq="B"))    # 2-session week, no Friday
    n = len(days)
    o = np.arange(1, n + 1, dtype=float)
    df = pd.DataFrame({"Open": o, "High": o + 1, "Low": o - 1, "Close": o + 0.5,
                       "Volume": np.full(n, 10.0)}, index=pd.DatetimeIndex(days))
    part = resample_weekly(df, keep_partial=True, today=pd.Timestamp("2024-01-16"))
    full = resample_weekly(df, keep_partial=False, today=pd.Timestamp("2024-01-16"))
    assert list(part["Days"]) == [5, 5, 2], list(part["Days"])
    assert [bool(x) for x in part["Partial"]] == [False, False, True]
    assert len(full) == 2 and str(full.index[-1].date()) == "2024-01-08"
    # the same 2-day week becomes a *completed* week once data reaches Friday
    days5 = list(pd.date_range("2024-01-01", "2024-01-19", freq="B"))
    o5 = np.arange(1, len(days5) + 1, dtype=float)
    df5 = pd.DataFrame({"Open": o5, "High": o5 + 1, "Low": o5 - 1, "Close": o5 + 0.5,
                        "Volume": np.full(len(days5), 10.0)}, index=pd.DatetimeIndex(days5))
    nxt = resample_weekly(df5, keep_partial=False, today=pd.Timestamp("2024-01-22"))
    assert len(nxt) == 3 and str(nxt.index[-1].date()) == "2024-01-15"


def test_resample_keeps_completed_current_week():
    # week ending Fri 19 Jan is complete even if "today" is that same Friday
    days = list(pd.date_range("2024-01-15", "2024-01-19", freq="B"))
    df = daily_frame(days, np.ones(5), 2.0, 0.5, 1.5, np.ones(5) * 10)
    wk = resample_weekly(df, keep_partial=False, today=pd.Timestamp("2024-01-19"))
    assert len(wk) == 1


def test_resample_handles_tz_and_prebuilt_weekly_input():
    days = pd.date_range("2023-01-02", periods=52 * 3, freq="7D", tz="Asia/Kolkata")
    n = len(days)
    df = daily_frame(days, np.linspace(100, 80, n), np.linspace(102, 82, n),
                     np.linspace(98, 78, n), np.linspace(101, 81, n), np.full(n, 5e5))
    wk = resample_weekly(df, keep_partial=True, today=pd.Timestamp("2025-12-31"))
    assert len(wk) == n
    assert getattr(wk.index, "tz", None) is None


def test_locked_week_detection():
    """A lock means the price *could not trade*: a pinned limit day or a no-trade session.
    A merely large move is not a lock (that filter used to reject 998/1000 NSE stocks)."""
    days = list(pd.date_range("2024-01-01", "2024-01-05", freq="B"))
    o = np.full(5, 100.0)

    # (1) a +5.0% wide-range day with full volume -> ordinary volatility, NOT locked
    up = o.copy(); up[1] = 105.0
    wk = resample_weekly(daily_frame(days, o, np.maximum(o, up) + 0.1,
                                     np.minimum(o, up) - 0.1, up, np.full(5, 1e6)),
                         keep_partial=True, today=pd.Timestamp("2024-01-05"))
    assert float(wk["LockedDays"].iloc[0]) == 0.0, wk

    # (2) a day pinned at the 10% circuit limit (open=high=low=close, nothing traded) -> locked
    c2 = o.copy(); c2[1] = 110.0
    o2 = o.copy(); o2[1] = 110.0
    h2 = np.maximum(o, c2) + 0.0; h2[1] = 110.0
    l2 = np.minimum(o, c2) - 0.0; l2[1] = 110.0
    v2 = np.full(5, 1e6); v2[1] = 0.0
    wk2 = resample_weekly(daily_frame(days, o2, h2, l2, c2, v2),
                          keep_partial=True, today=pd.Timestamp("2024-01-05"))
    assert float(wk2["LockedDays"].iloc[0]) >= 1.0, wk2

    # (3) flat / no-trade sessions (a locked limit price) are also detected
    flat = daily_frame(days, o, o, o, o, np.array([1e6, 0.0, 0.0, 1e6, 1e6]))
    wk3 = resample_weekly(flat, keep_partial=True, today=pd.Timestamp("2024-01-05"))
    assert float(wk3["LockedDays"].iloc[0]) >= 2.0, wk3


# --------------------------------------------------------------------------------------
# Swing structure
# --------------------------------------------------------------------------------------
def test_swing_lows_fractal_rule():
    wk = weekly([(10, 11, 9, 10), (10, 11, 8, 9), (9, 12, 7, 11), (11, 13, 10, 12),
                 (12, 14, 11, 13)])
    assert bool(swing_lows(wk, 2)[2])
    # plateau of equal lows is booked once (the last equal bar); the level is de-duplicated
    # by the sweep stage so the pool is still counted exactly once
    wk2 = weekly([(10, 11, 9, 10), (9, 10, 5, 9), (9, 10, 5, 9), (9, 10, 6, 9), (9, 11, 8, 10)])
    m = swing_lows(wk2, 2)
    assert bool(m[2]) and int(m.sum()) == 1, m
    # the last `strength` bars can never be a *confirmed* swing low
    assert not m[-2:].any()


def test_swing_low_survives_a_later_lower_low_and_edge_rule_holds():
    wk = weekly([(10, 11, 9, 10), (10, 11, 8, 9), (9, 12, 7, 11), (11, 13, 10, 12),
                 (12, 14, 11, 13), (13, 15, 12, 14), (14, 15, 6, 13), (13, 16, 12, 15),
                 (15, 16, 14, 15)])
    m = swing_lows(wk, 2)
    assert bool(m[2]) and bool(m[6])          # both are local lows
    assert not m[-2:].any(), "un-confirmable right edge must never be booked"


def test_swing_strength_controls_confirmation_delay():
    """strength=N sets how many bars a low needs on each side to become a usable level."""
    bars = drift(27) + [(101.0, 103.0, 100.0, 102.5), (102.5, 105.0, 102.0, 104.5),
                        (104.0, 104.5, 96.2, 104.0, 1_500_000)]
    wk = weekly(bars)
    assert bool(swing_lows(wk, 1)[27]) and not bool(swing_lows(wk, 2)[27])
    common = dict(min_weekly_bars=20, swing_lookback=20, allow_fresh_low=False,
                  min_depth_pct=0.0, max_range_of_close=0.15, min_wick_ratio=0.0,
                  min_wick_body_mult=0.0, min_close_in_range=0.0)
    weak = find_sweeps(wk, Params(**{**common, "swing_strength": 1}))
    strong = find_sweeps(wk, Params(**{**common, "swing_strength": 2}))
    assert weak and abs(weak[0].level - 100.0) < 1e-6, [h.level for h in weak]
    assert not strong, "an unconfirmed swing must not be usable at strength=2"
    # strength also decides whether a low is confirmed at all: the sweep_tape swing needs
    # only 6 bars of right-side room, so strength=5 still sees it, but a short lookback does not
    wk_s = sweep_tape()
    pl_short = Params(min_weekly_bars=20, swing_strength=5, allow_fresh_low=False,
                      swing_lookback=3, min_depth_pct=0.0)
    pl_long = Params(**{**pl_short.__dict__, "swing_lookback": 10})
    assert _levels_to_sweep(wk_s, len(wk_s) - 1, pl_short) == []
    assert abs(_levels_to_sweep(wk_s, len(wk_s) - 1, pl_long)[0][0] - 100.0) < 1e-6
    assert bool(swing_lows(wk_s, 5)[26]) and not bool(swing_lows(wk_s, 26)[26])

# --------------------------------------------------------------------------------------
# The sweep rule itself
# --------------------------------------------------------------------------------------
def test_detects_classic_sweep_of_old_swing_low():
    wk = sweep_tape()
    hits = find_sweeps(wk, Params(min_weekly_bars=20))
    assert hits, "a textbook sweep must be detected"
    h = hits[0]
    assert h.sweep_idx == len(wk) - 1
    assert abs(h.level - 100.0) < 1e-6, h.level          # the OLD swing low was the pool
    assert h.low < h.level < h.close                      # pierced then closed back above
    assert abs(h.depth_abs - 4.0) < 1e-6
    assert abs(h.close_above_pct - 0.025) < 1e-6
    assert abs(h.wick_ratio - 0.7143) < 1e-3, h.wick_ratio   # lower wick 5.0 / range 7.0
    assert abs(h.lower_wick - 5.0) < 1e-9 and abs(h.body - 1.5) < 1e-9
    assert abs(h.close_in_range - 0.9286) < 1e-3
    assert h.fresh_low is False and h.green is True
    assert h.swing_age_weeks == 6 and h.bars_since_sweep == 0
    assert h.score > 60, h.score
    r = _row(h, wk)
    assert r["SweepType"] == "52-week-low sweep" and bool(r["Is52wLow"])
    assert r["SweptSwingLow"] == 100.0 and r["SweepLow"] == 96.0
    assert r["PoolsTaken"] == 1

def test_sweep_output_carries_tradeable_levels():
    wk = sweep_tape()
    h = find_sweeps(wk, Params(min_weekly_bars=20))[0]
    r = _row(h, wk)
    assert r["SweepLow"] == 96.0 and r["SweptSwingLow"] == 100.0
    assert abs(r["Stop"] - round(96.0 * 0.995, 2)) < 1e-6
    assert r["Risk%"] > 0
    assert r["Target2R"] > r["Close"] and r["Target3R"] > r["Target2R"]
    assert r["BarsSinceSweep"] == 0
    assert np.isfinite(r["ATR(14w)"]) and r["ATR(14w)"] > 0


def test_no_sweep_when_close_finishes_below_level():
    wk = final_bar_tape(o=101.0, h=103.0, l=95.5, c=95.5)   # closes on its low, below 100
    p = Params(min_weekly_bars=20)
    assert not find_sweeps(wk, p), "close below the swept level is a breakdown, not a buy setup"
    assert near_miss(wk, p) is None


def test_no_sweep_when_close_is_exactly_at_level():
    # close == swept level (equal, not above) and no tail at all
    wk = final_bar_tape(o=96.0, h=100.0, l=96.0, c=100.0)
    assert not find_sweeps(wk, Params(min_weekly_bars=20)), "close must be ABOVE the swept low"


def test_no_sweep_without_wick():
    """
    A bar that ends AT its low (close ~= low, near the bottom of the range) has no rejection
    body at all -> never a sweep, and never even a near miss. This is the bar a naive
    "low < swing_low and close > swing_low" screener wrongly flags.
    """
    wk = final_bar_tape(o=96.0, h=103.0, l=95.9, c=96.05)
    p = Params(min_weekly_bars=20)
    assert not find_sweeps(wk, p)
    assert near_miss(wk, p) is None, "close below the swept level -> nothing reclaimed"
    # same idea but the level IS reclaimed; with no tail the bar is only a watchlist item
    wk2 = final_bar_tape(o=96.6, h=103.0, l=96.5, c=101.0)
    assert not find_sweeps(wk2, p)
    m = near_miss(wk2, p)
    assert m is not None and "wick too small" in m["FailReason"], m
    assert m["SweptLevel"] == 100.0 and float(m["WickRatio%"]) < 5.0, m


def test_min_close_above_pct_cushion_is_enforced():
    # close 100.5 = only 0.5% above the swept 100 level, tight 4.5-wide range
    wk = final_bar_tape(o=100.6, h=100.9, l=96.4, c=100.5, v=2_000_000)
    p_loose = Params(min_weekly_bars=20, min_close_above_pct=0.0)
    p_tight = Params(min_weekly_bars=20, min_close_above_pct=0.02)
    assert find_sweeps(wk, p_loose), "0.6% above the level passes a 0% cushion"
    assert not find_sweeps(wk, p_tight), "0.6% above the level fails a 2% cushion"


def test_dragonfly_doji_is_a_hit_and_a_strict_gate_downgrades_it():
    # open == high == close, deep tail: the purest rejection candle (wick ratio ~1.0)
    wk = final_bar_tape(o=101.0, h=101.05, l=96.0, c=101.02)
    assert abs(float(wk["Low"].iloc[-1]) - 96.0) < 1e-9
    p_norm = Params(min_weekly_bars=20)
    hits = find_sweeps(wk, p_norm)
    assert hits and hits[0].wick_ratio > 0.98, hits
    strict = Params(min_weekly_bars=20, min_close_in_range=0.999)   # demands a close at the very top
    assert not find_sweeps(wk, strict)
    m = near_miss(wk, strict)
    assert m is not None and "close weak" in m["FailReason"], m


def test_body_dominating_the_wick_is_rejected():
    # a big-bodied red bar that dips to 96 and closes at 100.5: wick 0.5 vs body 4.5
    bars = drift(26) + PRE + [(105.0, 105.2, 96.0, 100.5)]
    wk = weekly(bars)
    p = Params(min_weekly_bars=20, min_wick_ratio=0.0)
    assert not find_sweeps(wk, p), "the wick must dominate the body"
    m = near_miss(wk, p)
    assert m and "body larger than wick" in m["FailReason"], m

def test_fresh_low_sweep_can_be_switched_off():
    wk = fresh_low_tape()
    on = Params(min_weekly_bars=5, allow_fresh_low=True)
    off = Params(min_weekly_bars=5, allow_fresh_low=False)
    hits = find_sweeps(wk, on)
    assert hits, "undercutting a fresh low and closing back above is a valid sweep"
    assert hits[0].fresh_low is True and hits[0].is_low_52w is True
    assert not find_sweeps(wk, off), "must vanish when fresh-low sweeps are disabled"


SELF_SWEEP_BARS = drift(26) + [(101.0, 103.0, 100.0, 102.5), (102.5, 105.0, 102.0, 104.5),
                              (104.5, 106.0, 103.5, 105.0), (105.0, 106.0, 104.0, 104.5),
                              (104.0, 105.0, 103.2, 104.8)]


def test_a_bar_must_not_sweep_the_low_it_is_printing():
    """On a monotonic tape the floor IS the newest bar's low; nothing was resting below it."""
    wk = weekly(drift(27, start=125.0, step=1.0))
    lows = wk["Low"].to_numpy(float)
    i = len(wk) - 1
    assert int(np.argmin(lows)) == i, "the last bar prints the tape's lowest low"
    assert list(np.flatnonzero(swing_lows(wk, 2))) == [], "no confirmed swing on this tape"
    loose = dict(min_weekly_bars=20, allow_fresh_low=True, min_wick_ratio=0.0,
                 min_wick_body_mult=0.0, min_close_in_range=0.0, max_range_of_close=0.9,
                 min_depth_pct=0.0, swing_lookback=26)
    off = Params(**{**loose, "include_sweep_on_swing_bar": False})
    on = Params(**{**loose, "include_sweep_on_swing_bar": True})
    # the engine's own floor is this bar's index -> it can never appear as a level in "off" mode
    lv_off = _levels_to_sweep(wk, i, off)
    assert i not in {j for _, j, _ in lv_off}, "a bar must not be listed as its own swept pool"
    lv_on = _levels_to_sweep(wk, i, on)
    assert i in {j for _, j, _ in lv_on}, "include_sweep_on_swing_bar is what unlocks that pool"
    assert not find_sweeps(wk, off)


def test_lowest_pool_taken_is_the_one_reported():
    # swing 100, then a deeper swing 97; one bar undercuts BOTH -> it must report 97 (the
    # pool actually taken out), and it must be booked as a 2-pool sweep.
    bars = drift(24) + PRE[:4] + [(104.0, 104.6, 97.0, 104.2), (104.2, 105.0, 103.5, 104.6),
                                  (103.0, 104.0, 96.0, 103.5, 2_000_000)]
    wk = weekly(bars)
    hits = find_sweeps(wk, Params(min_weekly_bars=20, swing_lookback=26, min_wick_ratio=0.30))
    assert hits
    assert abs(hits[0].level - 97.0) < 1e-6, [h.level for h in hits]
    assert hits[0].n_pools == 2


def test_52w_flag_only_when_really_the_52w_low():
    bars = (drift(20) + [(150.0, 151.0, 80.0, 145.0, 2_000_000)] + drift(39, start=145.0, step=1.0)
            + [(101.0, 103.0, 100.0, 102.5), (102.5, 105.0, 102.0, 104.5),
               (104.5, 106.0, 103.5, 105.0), (105.0, 106.0, 104.0, 104.5),
               (104.5, 105.5, 103.0, 104.0), (104.0, 105.0, 103.2, 104.8),
               (101.0, 103.0, 96.0, 102.5, 4_000_000)])
    wk = weekly(bars)
    hits = find_sweeps(wk, Params(min_weekly_bars=20, swing_lookback=13))
    assert hits, "100 must still be swept as an old swing low"
    assert hits[0].is_low_52w is False, "the 52-week low here is 80, not 100"
    assert hits[0].fresh_low is False


def test_sweep_of_a_52w_low_is_flagged():
    hits = find_sweeps(sweep_tape(), Params(min_weekly_bars=20))
    assert hits[0].is_low_52w is True      # on this tape 100 IS the lowest low of the year


def test_only_recent_sweeps_when_max_bars_ago_is_one():
    wk = stale_sweep_tape(n_after=4)
    assert not find_sweeps(wk, Params(min_weekly_bars=20, max_sweep_bars_ago=1))
    hits = find_sweeps(wk, Params(min_weekly_bars=20, max_sweep_bars_ago=6))
    assert hits and hits[0].sweep_idx == len(wk) - 5, [h.sweep_idx for h in hits]


def test_fresh_setup_ranks_first():
    bars = (drift(26) + [(101.0, 103.0, 100.0, 102.5), (102.5, 105.0, 102.0, 104.5),
                         (104.5, 106.0, 103.5, 105.0), (105.0, 106.0, 104.0, 104.5),
                         (104.0, 105.0, 103.0, 104.0),
                         (101.0, 103.0, 96.0, 102.5, 4_000_000),      # stale sweep of 100
                         (102.5, 105.0, 102.4, 104.5), (104.5, 106.0, 104.0, 105.5),
                         (105.5, 106.5, 104.5, 105.0), (105.0, 105.5, 104.2, 104.6),
                         (104.6, 105.0, 104.0, 104.4),
                         (104.2, 105.0, 95.0, 104.8, 1_500_000)])      # fresh sweep of 100
    wk = weekly(bars)
    hits = find_sweeps(wk, Params(min_weekly_bars=20, max_sweep_bars_ago=8))
    assert len(hits) >= 2, [(h.sweep_idx, h.level) for h in hits]
    assert hits[0].sweep_idx == len(wk) - 1, [(h.sweep_idx, h.score) for h in hits]
    assert hits[0].bars_since_sweep == 0


def test_monster_bar_rejected_as_corporate_action():
    wk = final_bar_tape(o=104.0, h=140.0, l=96.0, c=130.0)   # 34% weekly range
    p = Params(min_weekly_bars=20, max_range_of_close=0.12)
    assert not find_sweeps(wk, p)
    m = near_miss(wk, p)
    assert m and "range too wide" in m["FailReason"], m


def test_noise_pierce_below_min_depth_rejected():
    wk = final_bar_tape(o=104.0, h=105.0, l=99.98, c=104.8)   # 2 paise under the level
    assert not find_sweeps(wk, Params(min_weekly_bars=20, min_depth_pct=0.004))
    assert find_sweeps(wk, Params(min_weekly_bars=20, min_depth_pct=0.0)), \
        "with the floor removed the same bar becomes a (weak) sweep"


def test_deep_collapse_through_the_level_is_rejected():
    wk = final_bar_tape(o=104.0, h=104.6, l=70.0, c=104.2)   # 30% under the level
    p = Params(min_weekly_bars=20, max_depth_pct=0.16, max_range_of_close=0.9)
    assert not find_sweeps(wk, p)


def test_volume_drydown_penalised_not_fatal():
    p = Params(min_weekly_bars=20)
    wk1 = sweep_tape()
    wk2 = sweep_tape()
    wk2.iloc[-1, wk2.columns.get_loc("Volume")] = 20_000.0
    h1, h2 = find_sweeps(wk1, p)[0], find_sweeps(wk2, p)[0]
    assert h2.score < h1.score, (h1.score, h2.score)
    assert h2.sweep_idx == h1.sweep_idx


def test_stricter_gates_are_respected():
    base = Params(min_weekly_bars=20)
    assert find_sweeps(sweep_tape(), base)
    assert find_sweeps(sweep_tape(), Params(min_weekly_bars=20, require_green_close=True))
    red = final_bar_tape(o=103.0, h=103.2, l=96.0, c=100.5)     # red hammer (close < open)
    assert find_sweeps(red, base), "a red rejection candle is still a sweep by default"
    assert not find_sweeps(red, Params(min_weekly_bars=20, require_green_close=True))


def test_min_score_gate_filters():
    wk = sweep_tape()
    s0 = find_sweeps(wk, Params(min_weekly_bars=20))[0].score
    assert find_sweeps(wk, Params(min_weekly_bars=20, min_score=1.0))
    assert not find_sweeps(wk, Params(min_weekly_bars=20, min_score=min(99.0, s0 + 1.0)))


def test_swing_strength_changes_sensitivity():
    """A wider fractal needs a deeper V; a narrow one fires on minor kinks."""
    wk = sweep_tape()
    strong = find_sweeps(wk, Params(min_weekly_bars=20, swing_strength=5))
    weak = find_sweeps(wk, Params(min_weekly_bars=20, swing_strength=1))
    assert weak, "strength=1 must find the 100 swing"
    assert isinstance(strong, list)     # may be empty; must simply never crash


def test_lookback_window_limits_which_pool_can_be_swept():
    """swing_lookback decides which older pools are even in play for a sweep."""
    bars = [(101.0, 103.0, 80.0, 101.0), (101.0, 103.0, 101.5, 102.5)] + drift(24)
    bars += PRE[:5] + [(104.0, 104.6, 79.0, 104.4, 1_500_000)]
    wk = weekly(bars)
    common = dict(min_weekly_bars=20, min_depth_pct=0.0, max_range_of_close=0.30,
                  max_depth_pct=0.90, min_wick_ratio=0.0, min_wick_body_mult=0.0,
                  min_close_in_range=0.0)
    wide = find_sweeps(wk, Params(**{**common, "swing_lookback": 40}))
    narrow = find_sweeps(wk, Params(**{**common, "swing_lookback": 6}))
    assert wide and abs(wide[0].level - 80.0) < 1e-6, [h.level for h in wide]
    assert wide[0].n_pools == 2, wide[0].n_pools         # both 80 and 100 were undercut
    assert wide[0].swing_age_weeks == 31
    assert narrow and abs(narrow[0].level - 100.0) < 1e-6, [h.level for h in narrow]
    assert narrow[0].n_pools == 1, narrow[0].n_pools    # the deep 80 pool is out of range


# --------------------------------------------------------------------------------------
# Plumbing: screen_all / rank / backtest / charts
# --------------------------------------------------------------------------------------
def test_screen_all_end_to_end_on_synthetic_universe():
    p = Params(min_weekly_bars=20, min_turnover_lakh=0.0)
    wk_good = sweep_tape()
    data = {
        "GOODCO": as_daily(wk_good),
        "FLATCO": as_daily(weekly(drift(60))),
        "TINYCO": as_daily(weekly([(1.0, 1.02, 0.99, 1.01)] * 140)),
    }
    hits, miss, diag = screen_all(data, p)
    assert not hits.empty and list(hits["Symbol"]) == ["GOODCO"], hits
    assert hits.iloc[0]["SweptSwingLow"] == 100.0
    assert float(hits.iloc[0]["SweepLow"]) == 96.0
    assert hits.iloc[0]["Rank"] == 1 and hits.iloc[0]["Score"] > 0
    assert str(pd.Timestamp(wk_good.index[-1]).date()) == hits.iloc[0]["SweepWeek"]
    assert set(diag["Symbol"]) == set(data)
    assert "price_below_min" in set(diag["status"]), diag
    assert "FLATCO" in set(diag.loc[diag["status"] == "no_setup", "Symbol"]), diag
    for c in ("Close", "SweptSwingLow", "SweepLow", "WickRatio%", "Risk%", "Stop", "Target2R"):
        assert c in hits.columns, c


def test_screen_all_reports_near_miss_watchlist():
    weak = final_bar_tape(o=96.6, h=103.0, l=96.5, c=101.0)     # reclaimed 100, zero rejection tail
    strict = Params(min_weekly_bars=20, min_turnover_lakh=0.0)
    hits, miss, diag = screen_all({"WEAKCO": as_daily(weak)}, strict)
    assert hits.empty and not miss.empty, (hits, miss)
    assert miss.iloc[0]["Symbol"] == "WEAKCO"
    assert "wick too small" in miss.iloc[0]["FailReason"]
    assert diag.iloc[-1]["status"] == "near_miss", diag
    # dropping the close-quality bar is enough to turn the SAME candle into a hit
    loose = Params(min_weekly_bars=20, min_turnover_lakh=0.0, min_wick_ratio=0.0,
                   min_wick_body_mult=0.0, min_close_in_range=0.0, max_range_of_close=0.9)
    hits2, miss2, diag2 = screen_all({"WEAKCO": as_daily(weak)}, loose)
    assert not hits2.empty and diag2.iloc[-1]["status"] == "hit", (hits2, diag2)
    # keep_near_miss=False suppresses the watchlist but not the diagnostics
    h3, m3, d3 = screen_all({"WEAKCO": as_daily(weak)}, strict, keep_near_miss=False)
    assert m3.empty and len(d3) == 1


def test_screen_all_illiquidity_and_circuit_filters():
    wk = sweep_tape()
    daily = as_daily(wk)
    p_liquid = Params(min_weekly_bars=20, min_turnover_lakh=1e9)   # absurd threshold
    hits, miss, diag = screen_all({"GOODCO": daily}, p_liquid)
    assert hits.empty
    assert diag.iloc[-1]["status"] == "illiquid", diag
    # circuit / limit-locked weeks: 3 pinned no-trade sessions a week -> filtered
    days = list(pd.date_range("2023-06-05", periods=60, freq="B"))
    o = np.full(60, 100.0); c = o.copy()
    v = np.full(60, 1e6)
    for k in range(5, 60, 5):                 # three zero-range, zero-volume days each week
        for j in (k, k + 1, k + 2):
            c[j] = 100.0; v[j] = 0.0
    lock = daily_frame(days, o, o.copy(), o.copy(), c, v)
    hits3, miss3, diag3 = screen_all({"LOCKCO": lock}, Params(min_weekly_bars=5,
                                                              min_turnover_lakh=0.0))
    assert hits3.empty
    assert diag3.iloc[-1]["status"] == "circuit_locked", diag3.iloc[-1]
    # and the SAME tape without the locks must NOT be circuit-locked (no false positives)
    clean = daily_frame(days, o, o + 1.0, o - 1.0, o.copy(), np.full(60, 1e6))
    h4, m4, d4 = screen_all({"CLEANCO": clean}, Params(min_weekly_bars=5, min_turnover_lakh=0.0))
    assert (d4.iloc[-1]["status"] != "circuit_locked") or (d4.iloc[-1]["LockedWeeks26"] <= 1), d4


def test_screen_all_survives_garbage_input():
    p = Params(min_weekly_bars=20)
    junk = {
        "EMPTY": pd.DataFrame(columns=["Open", "High", "Low", "Close", "Volume"]),
        "NANS": pd.DataFrame({"Open": [np.nan], "High": [np.nan], "Low": [np.nan],
                              "Close": [np.nan], "Volume": [np.nan]},
                             index=pd.to_datetime(["2024-01-01"])),
        "SHORT": as_daily(weekly(drift(4))),
        "INV": as_daily(weekly([(10.0, 5.0, 20.0, 1.0)] * 130)),   # high < low
        "NOTIME": pd.DataFrame({"Open": [1.0, 2.0], "High": [3.0, 4.0], "Low": [0.5, 1.5],
                                "Close": [2.0, 3.0], "Volume": [1.0, 2.0]}, index=["x", "y"]),
    }
    hits, miss, diag = screen_all(junk, p)
    assert isinstance(hits, pd.DataFrame) and isinstance(diag, pd.DataFrame)
    assert hits.empty
    assert set(diag["Symbol"]) >= {"EMPTY", "NANS", "SHORT", "INV"}


def test_progress_callback_is_called():
    calls = []
    data = {f"S{i}": as_daily(weekly(drift(120))) for i in range(3)}
    screen_all(data, Params(min_weekly_bars=20, min_turnover_lakh=0.0),
               keep_near_miss=False, progress_cb=lambda a, b: calls.append((a, b)))
    assert calls and calls[0][1] == 3


def test_rank_top_prefers_market_cap_then_turnover():
    data = {s: as_daily(weekly(drift(60))) for s in ("A", "B", "C")}
    data["A"].iloc[-1, data["A"].columns.get_loc("Volume")] = 1e9
    quotes = pd.DataFrame({"Symbol": ["A", "B", "C"], "MarketCap": [1e11, 5e11, 2e11]})
    tab = rank_top(data, quotes, top_n=3)
    assert list(tab["Symbol"]) == ["B", "C", "A"], tab
    assert tab["rank_basis"].iloc[0] == "market_cap"
    # no quotes -> turnover/index ordering, and it must still be a clean 1..N ranking
    tab2 = rank_top(data, None, top_n=3)
    assert tab2["rank_basis"].iloc[0] == "index_tier+turnover"
    assert list(tab2["Rank"]) == [1, 2, 3]
    assert list(tab2["Symbol"])[0] == "A"          # A has the fattest turnover
    # patchy market cap must NOT hijack the ranking (that would silently drop most stocks)
    bad = quotes.copy(); bad.loc[1:, "MarketCap"] = np.nan
    assert rank_top(data, bad, top_n=3)["rank_basis"].iloc[0] == "index_tier+turnover"


def test_rank_top_honours_index_tiers_and_truncates():
    data = {s: as_daily(weekly(drift(60))) for s in ("BIG", "MID", "SMALL", "MICRO")}
    # make turnover strictly ordered SMALL > MICRO > MID > BIG so the index tier must decide
    for k, sym in enumerate(("SMALL", "MICRO", "MID", "BIG")):
        data[sym].iloc[:, data[sym].columns.get_loc("Volume")] = float(40 - k) * 1e7
    tiers = pd.DataFrame({"Symbol": ["BIG"], "IndexTier": ["Nifty50"], "Industry": ["Oil"]})
    tab = rank_top(data, None, top_n=4, index_tiers=tiers)
    assert list(tab["Symbol"])[0] == "SMALL", tab        # default: liquidity ordering wins
    assert list(tab["Symbol"]) == ["SMALL", "MICRO", "MID", "BIG"], list(tab["Symbol"])
    assert tab["IndexTier"].dropna().tolist() == ["Nifty50"]   # tier is carried, not enforced
    assert tab["Industry"].dropna().unique().tolist() == ["Oil"]
    # opt-in: official index tier first, turnover inside each tier
    tab2 = rank_top(data, None, top_n=4, index_tiers=tiers, prefer_index_tier=True)
    assert tab2["Symbol"].iloc[0] == "BIG", list(tab2["Symbol"])
    assert list(tab2["Rank"]) == [1, 2, 3, 4]
    # truncation is real
    assert len(rank_top(data, None, top_n=2)) == 2


def test_rank_top_empty():
    assert rank_top({}, None).empty


def _repetitive_tape(reps=6):
    """Alternating swing + sweep, so the backtest sees many repeatable occurrences."""
    bars = drift(10, start=170.0, step=1.5)
    for _ in range(reps):
        bars += [(151.0, 153.0, 150.0, 152.5), (152.5, 155.0, 152.0, 154.5),
                 (154.5, 156.0, 153.5, 155.0), (155.0, 156.0, 154.0, 155.5)]
        bars.append((151.5, 154.0, 145.5, 153.5, 3_000_000))   # sweep of the 150 swing low
        bars += [(153.5, 158.0, 153.0, 157.0), (157.0, 161.0, 156.0, 160.0),
                 (160.0, 163.0, 158.5, 162.0)]
    return weekly(bars)


def test_backtest_and_summary():
    wk = _repetitive_tape()
    daily = as_daily(wk)
    p = Params(min_weekly_bars=20, min_turnover_lakh=0.0)
    trades = backtest({"SYN": daily}, p, horizon_weeks=6, r_mult=2.0)
    assert not trades.empty, "repetitive tape must produce trades"
    assert np.isfinite(trades["Return%"].to_numpy(float)).all()
    assert (trades["SweptLevel"] - 150.0).abs().max() < 1e-6
    # entry = open of the week AFTER the sweep
    entries = trades["Entry"].to_numpy(float)
    opens = wk["Open"].to_numpy(float)
    for e in entries:
        assert float(np.abs(opens - e).min()) < 0.011, e
    s = summarize_backtest(trades)
    assert 0 <= s["win_rate_%"] <= 100 and s["trades"] == len(trades)
    assert abs(s["hit_target_%"] + s["hit_stop_%"] + s["expired_%"] - 100.0) < 0.15
    assert summarize_backtest(pd.DataFrame()) == {"trades": 0}


def test_backtest_exit_priority_is_stop_before_target_same_week():
    """If a week touches both, assume the stop (conservative)."""
    wk = weekly(_repetitive_tape(1).iloc[:15].to_dict("records") and
                drift(10) + [(151.0, 153.0, 150.0, 152.5), (152.5, 155.0, 152.0, 154.5),
                             (154.5, 156.0, 153.5, 155.0), (155.0, 156.0, 154.0, 155.5),
                             (151.5, 154.0, 145.5, 153.5, 3_000_000),
                             (153.5, 200.0, 100.0, 120.0)])   # both stop and target in one week
    trades = backtest({"SYN": as_daily(wk)}, Params(min_weekly_bars=20, min_turnover_lakh=0.0),
                      horizon_weeks=4, r_mult=2.0)
    if not trades.empty:
        assert set(trades["Outcome"]) <= {"stop", "target", "time"}
        assert (trades["Return%"] < 0).any(), "touching both must resolve as the stop"


def test_chart_frames_shapes():
    wk = sweep_tape()
    tail, info = chart_frames(as_daily(wk), Params(min_weekly_bars=20), weeks_to_show=20)
    assert 0 < len(tail) <= 20
    assert info["hit"] is not None
    assert 0 <= info["hit"]["sweep_pos"] < len(tail)
    assert all(0 <= s["pos"] < len(tail) for s in info["swings"])
    assert info["levels"] and all(l["level"] > 0 for l in info["levels"])
    empty = pd.DataFrame(columns=["Open", "High", "Low", "Close", "Volume"])
    t0, i0 = chart_frames(empty, Params(min_weekly_bars=20), weeks_to_show=20)
    assert t0.empty and i0["hit"] is None


def test_params_describe_is_json_safe():
    import json
    d = Params().describe()
    json.dumps({k: (None if isinstance(v, float) and not np.isfinite(v) else v) for k, v in d.items()})
    assert set(d) >= {"swing_strength", "min_wick_ratio", "max_sweep_bars_ago"}


def test_empty_inputs_never_raise():
    p = Params()
    assert find_sweeps(weekly([]), p) == []
    assert find_sweeps(weekly([(1, 2, 0.5, 1.5)] * 5), p) == []
    assert near_miss(weekly([(1.0, 2.0, 0.5, 1.5)] * 2), p) is None
    assert swing_lows(weekly([(1, 2, 0.5, 1.5)] * 3), 2).sum() == 0
    assert atr(weekly(drift(20)), 14).gt(0).all()
    assert len(resample_weekly(pd.DataFrame(columns=["Open", "High", "Low", "Close"]))) == 0


def test_resample_weekly_locked_days_is_silent_on_normal_volatility():
    """Guard for the bug that rejected 998/1000 symbols: a 5% day is NOT a circuit lock."""
    days = list(pd.bdate_range("2024-01-01", periods=25))
    o = np.full(25, 100.0); h = o + 2.0; l = o - 2.0; c = o.copy(); v = np.full(25, 1e6)
    c[3] = 105.0; h[3] = 106.0; l[3] = 99.0          # a wide, liquid +5% day
    c[9] = 94.0;  h[9] = 101.0; l[9] = 93.0           # and a -6% one
    wk = resample_weekly(pd.DataFrame({"Open": o, "High": h, "Low": l, "Close": c,
                                        "Volume": v}, index=pd.DatetimeIndex(days)),
                         keep_partial=True)
    assert float(wk["LockedDays"].sum()) == 0.0, wk["LockedDays"]
    assert locked_weeks(wk) == 0

    # a day pinned at +10% with no trade through it IS a lock
    c2 = c.copy(); o2 = o.copy(); h2 = h.copy(); l2 = l.copy(); v2 = v.copy()
    c2[9] = o2[9] = h2[9] = l2[9] = 110.0; v2[9] = 0.0
    wk2 = resample_weekly(pd.DataFrame({"Open": o2, "High": h2, "Low": l2, "Close": c2,
                                         "Volume": v2}, index=pd.DatetimeIndex(days)),
                          keep_partial=True)
    assert locked_weeks(wk2) >= 1, wk2["LockedDays"]


def test_assess_daily_repairs_inconsistent_bars_instead_of_deleting_them():
    """Yahoo hands back bars where High ignores Open (NORBTEAEXP 2021-06-25). The row must
    survive with a corrected range - deleting it silently loses sweep evidence."""
    from nse_data import assess_daily
    days = list(pd.bdate_range("2024-01-01", periods=5))
    o = np.array([6.35, 6.40, 6.45, 6.50, 6.55]); c = o.copy()
    h = np.array([6.30, 6.42, 6.47, 6.52, 6.57])          # first High < Open -> impossible
    l = np.array([6.25, 6.38, 6.43, 6.48, 6.53])
    df = pd.DataFrame({"Open": o, "High": h, "Low": l, "Close": c,
                       "Volume": np.full(5, 1e6)}, index=pd.DatetimeIndex(days))
    clean, issues, m = assess_daily(df, min_weekly_bars=1, max_fresh_lag_days=10_000)
    assert len(clean) == 5, (len(clean), issues)          # nothing dropped
    assert float(clean["High"].iloc[0]) == 6.35           # repaired to max(O,H,C)
    assert m["ohlc_repaired"] == 1
    assert any("repaired" in i for i in issues), issues


def test_max_sweep_bars_ago_actually_widens_the_window():
    """Regression: the screener once hard-filtered to the newest bar, which made
    max_sweep_bars_ago a silent no-op (a "5-week" screen returned the "1-week" answer)."""
    base = sweep_tape()                                    # sweep on the newest week
    p1 = Params(min_weekly_bars=20, min_turnover_lakh=0.0)
    p5 = Params(min_weekly_bars=20, min_turnover_lakh=0.0, max_sweep_bars_ago=5)
    h_now, _, _ = screen_all({"CO": as_daily(base)}, p1)
    assert len(h_now) == 1, h_now

    # three quiet weeks on top: nothing sweeps, the old sweep is now 3 weeks stale
    dead = pd.DataFrame({"Open": 130.0, "High": 133.0, "Low": 128.0, "Close": 131.0,
                         "Volume": 9e5},
                        index=pd.DatetimeIndex([base.index[-1] + pd.Timedelta(weeks=k)
                                                for k in (1, 2, 3)]))
    grown = as_daily(pd.concat([base, dead]))
    h_strict, _, d_strict = screen_all({"CO": grown}, p1)
    h_wide, _, _ = screen_all({"CO": grown}, p5)
    assert len(h_strict) == 0, h_strict          # newest week sweeps nothing
    assert len(h_wide) == 1, h_wide              # a window of 5 reaches back to the real sweep
    assert int(h_wide.iloc[0]["BarsSinceSweep"]) == 3, h_wide.iloc[0].to_dict()
    assert float(h_wide.iloc[0]["Score"]) < float(h_now.iloc[0]["Score"])   # recency penalised
    assert d_strict.iloc[-1]["status"] in ("no_setup", "near_miss"), d_strict.iloc[-1]["status"]


def test_cache_path_is_collision_free_for_special_symbols():
    """'M&M' and 'M_M' must not share a cache file, and read/write must always agree."""
    from nse_data import _cache_path, save_daily_cache, load_daily_cache
    import os, tempfile
    with tempfile.TemporaryDirectory() as d:
        a = _cache_path(d, "M&M", "1d", "csv")
        b = _cache_path(d, "M_M", "1d", "csv")
        c = _cache_path(d, " M&M ", "1d", "csv")
        assert a != b, (a, b)
        assert a == c, (a, c)                      # whitespace-only difference resolves the same
        df = daily_frame(list(pd.date_range("2024-01-01", periods=5, freq="B")),
                         np.full(5, 10.0), np.full(5, 11.0), np.full(5, 9.0),
                         np.full(5, 10.5), np.full(5, 1000.0))
        save_daily_cache(df, d, "M&M", "csv")
        got = load_daily_cache(d, "M&M", "csv")
        assert got is not None and len(got) == 5
        assert load_daily_cache(d, "M_M", "csv") is None      # no cross-contamination
        assert not os.path.basename(a).count("..")
'''
with open(os.path.join(PKG_DIR, 'tests_fixtures.py'), 'w', encoding='utf-8') as _f:
    _f.write(_SRC_TESTS_FIXTURES)
print('tests_fixtures.py     ', os.path.getsize(os.path.join(PKG_DIR, 'tests_fixtures.py')), 'bytes  ·  Self-test suite — 49 checks on synthetic tapes')

tests_fixtures.py      42042 bytes  ·  Self-test suite — 49 checks on synthetic tapes


In [ ]:
import importlib, sys
for _m in ("nse_data", "sweep_engine", "screener"):
    if _m in sys.modules:
        del sys.modules[_m]
import nse_data as ND
import sweep_engine as SE
import screener as SC
from nse_data import FetchConfig, YahooClient
from sweep_engine import Params
from screener import Screener, describe_criteria, print_table, style_results, plot_setup
print("engine imported ·", ND.__file__)

engine imported · /content/wlsweep/nse_data.py


---
## 2 · Configuration — the exact rule you asked for

Every criterion you specified is a **hard gate** (all four must pass):

1. **weekly** liquidity sweep of a **swing low** — the sweep must be *complete*, i.e. on the last **closed** weekly candle;
2. the swept low may be an **old** swing low **or a brand-new** one (`allow_fresh_low`);
3. the sweep candle **closes above that swing low** (`min_close_above_pct`);
4. a **proper rejection wick** is formed (`min_wick_ratio`, wick-vs-body multiple, close in the upper part of the range).

Then a liquidity/quality floor keeps the list tradable. Change anything here and re-run §5 —
re-screening from cache takes seconds.

In [ ]:
# @title 2 · Parameters  (Ctrl/Cmd+Enter = apply & re-screen later)
CONFIG = {
    # ---------- universe & data ----------
    "universe_size": 1000,        # screen the top-N by liquidity after ranking all of NSE
    "history_years": 6,           # 6y of daily bars -> ~310 weekly candles
    "max_workers": 3,             # parallel fetchers. 3 is the sweet spot vs. 429s
    "request_sleep": 0.12,        # GLOBAL min gap between any two HTTP calls (seconds)
    "throttle_pause": 20,         # park time when 429s cluster
    "final_pass": True,           # slow retry for throttled symbols on another transport
    "transport": "auto",          # auto | curl_cffi | requests | yfinance
    "use_quotes": False,          # True = also try Yahoo quotes for market cap (often 429)
    "prefer_index_tier": False,   # True = Nifty50/100/200... first, then liquidity
    "cache_dir": "cache_nse",
    "mount_drive": False,         # True = keep the cache + outputs on Google Drive

    # ---------- the sweep rule ----------
    "min_weekly_bars": 104,       # >= 2 years of weekly candles
    "min_price": 5.0,             # rupees; filters penny/illiquid noise
    "min_turnover_lakh": 75.0,    # median daily turnover, in lakh (75 L = 0.75 Cr)
    "swing_strength": 2,          # fractal: N bars lower on BOTH sides -> a confirmed swing
    "swing_lookback": 26,         # weeks back to hunt for swing lows
    "min_swing_age": 2,           # a swing needs this many closed weeks to be "confirmed"
    "allow_fresh_low": True,      # NEW lows count (window low / 52w low) - your requirement
    "require_old_swing": False,   # True = only pre-existing swing lows, no fresh lows
    "max_sweep_bars_ago": 1,      # 1 = sweep on the most recent CLOSED week. 3 = last 3 weeks
    "include_sweep_on_swing_bar": True,   # the sweep candle may BE the new swing low
    "min_close_above_pct": 0.002, # close must clear the swept level by 0.2%
    "min_wick_ratio": 0.34,       # lower wick >= 34% of the candle range
    "min_wick_body_mult": 1.15,   # lower wick >= 1.15x the body  (rejection, not a doji)
    "min_close_in_range": 0.50,   # close in the upper 50% of the range
    "min_depth_pct": 0.002,       # it must actually pierce the level by 0.2%...
    "max_depth_pct": 0.16,        # ...but not collapse 16% below it (that's a breakdown)
    "max_depth_atr_mult": 3.0,    # ...and not more than 3 weekly ATRs deep
    "max_range_of_close": 0.12,   # candle range <= 12% of close (no circuit-day lottery)
    "require_green_close": False, # True = only green sweep candles
    "exclude_locked_weeks": True, # drop weeks that gapped limit-up/down (untradeable)
    "max_locked_weeks_26": 1,
    "min_score": 0.0,             # 0 = show everything; 55 = only A-grade setups
}

CFG_PARAMS = {k: v for k, v in CONFIG.items()
              if k in Params.__dataclass_fields__}
CFG_FETCH  = {k: v for k, v in CONFIG.items()
              if k in FetchConfig.__dataclass_fields__}
fetch_cfg = FetchConfig(**CFG_FETCH)
params    = Params(**CFG_PARAMS)
print(describe_criteria(params))

BUY SETUP — completed weekly candle only (the live week is excluded, so nothing repaints)
  1. a prior weekly swing low exists  : fractal low with 2 lower/higher bars on each side,
                                        searched up to 26 weeks back. Fresh lows and the 52-week
                                        low also qualify (yes).
  2. that low was TAKEN OUT            : weekly LOW < swing low by ≥ 0.20% of the level
                                        (and ≤ 16% / 3.0× ATR, so a
                                        collapse through it is NOT called a sweep).
  3. price REJECTED back above         : weekly CLOSE > swept level + cushion
                                        (≥ 0.2% above, if set).
  4. a PROPER WICK is visible          : lower wick ≥ 34% of the weekly range,
                                        wick ≥ 1.15× the body, close in the top
                                        50% of the range, range ≤ 12% of price.
  5. tradability                     

## 3 · Universe & data health

This is the cell to watch on day one. It prints the **full** NSE equity list it found,
the index tiers it attached, then fetches every symbol and reports exactly what came back —
per-symbol status, row counts, date coverage and the HTTP telemetry. If a stock is missing,
§7 tells you why in one line.

In [ ]:
sc = Screener(params, fetch_cfg, out_dir=OUT_DIR,
              universe_size=CONFIG["universe_size"],
              prefer_index_tier=CONFIG["prefer_index_tier"],
              use_quotes=CONFIG["use_quotes"], verbose=True)
if CONFIG["mount_drive"]:
    SC.mount_drive_if_requested(True)
sc.prepare_universe()
uni = sc.universe
print(f"\nNSE equities in universe : {len(uni)}")
print("series mix             :", uni["Series"].value_counts().head(6).to_dict())
print("with an index tier     :", int(uni["IndexTier"].notna().sum()))
uni.head(8)[["Symbol", "Company", "Series", "DateOfListing", "IndexTier"]]

NSE WEEKLY LIQUIDITY-SWEEP SCREENER   2026-09-13 21:36 IST
· NSE full equity list (archives.nseindia.com, cached to disk)…
  universe: downloaded archives.nseindia.com (177 KB)
  → 2319 tradeable equities after excluding weird series
· Index membership (Nifty 50/100/200/500/Midcap150/Smallcap250)…
  → 501 symbols carry an index tier ({'Nifty500': 301, 'Nifty200': 100, 'Nifty50': 50, 'Nifty100': 50})

NSE equities in universe : 2319
series mix             : {'EQ': 2292, 'BZ': 27}
with an index tier     : 498


,Symbol,Company,Series,DateOfListing,IndexTier
0,20MICRONS,20 Microns Limited,EQ,06-OCT-2008,NaN
1,21STCENMGM,21st Century Management Services Limited,EQ,03-MAY-1995,NaN
2,360ONE,360 ONE WAM LIMITED,EQ,19-SEP-2019,Nifty200
3,3BBLACKBIO,3B Blackbio Dx Limited,EQ,20-APR-2026,NaN
4,3MINDIA,3M India Limited,EQ,13-AUG-2004,Nifty500
5,3PLAND,3P Land Holdings Limited,EQ,19-JUL-1995,NaN
6,5PAISA,5Paisa Capital Limited,EQ,16-NOV-2017,NaN
7,63MOONS,63 moons technologies limited,EQ,20-JUN-2005,NaN


In [ ]:
_t0 = time.time()
sc.fetch(progress=True)          # cache-first: re-runs skip the network entirely
print(f"fetch took {time.time()-_t0:.1f}s")

· Fetching 2319 symbols, 6y of daily bars (cache first, 3 workers, >=0.12s between calls)…


Fetching bars:   0%|                                                    | 0/2319 [00:00<?, ?stock/s]

  → usable 1763 / 2319 in 315s (each symbol accounted for: True)

Symbols attempted : 2319
Usable            : 1763  (76.0%)
   - ok: 1763
   - insufficient_history: 520
   - illiquid_history: 36
Sources: yahoo=1763, -=556
Rows/stock: median 1481, min 1, max 1485
Staleness warnings: 0
Minimum weekly bars required: 104 (~520 trading days)
fetch took 315.2s


### 3.1 · Did the fetch actually work? (read this table)

In [ ]:
rep = sc.report
print(ND.data_health(rep, sc.cfg))
print("\n--- every non-ok status, grouped ---")
bad = rep[rep["status"] != "ok"]
if len(bad):
    print(bad.groupby("status")["Symbol"].apply(lambda s: ", ".join(sorted(s)[:12])).to_string())
print("\n--- worst 10 by age of last bar ---")
rep2 = rep.copy()
rep2["last"] = pd.to_datetime(rep2["last"], errors="coerce")
rep2["stale_days"] = (pd.Timestamp.utcnow().tz_localize(None) - rep2["last"]).dt.days
print(rep2[["Symbol", "status", "rows", "last", "stale_days", "issues"]]
      .sort_values("stale_days", ascending=False).head(10).to_string(index=False))
_no_row = sorted(set(sc.universe["Symbol"].astype(str)) - set(rep["Symbol"].astype(str)))
print(f"\ncoverage: {len(rep)} report rows for {len(sc.universe)} universe symbols; "
      f"{len(_no_row)} with no row" + (f" -> {_no_row[:6]}..." if _no_row else " (complete)"))
if _no_row:
    print("   (only happens when a subset was fetched; run sc.fetch() with no argument for all of NSE)")
print(f"\nHTTP calls {sc.client.stats['http_calls']} for {len(rep)} symbols "
      f"= {sc.client.stats['http_calls']/max(1,len(rep)):.2f} per symbol · "
      f"429s {sc.client.stats['rate_limited_429']} · retries {sc.client.stats['retries']}")

Symbols attempted : 2319
Usable            : 1763  (76.0%)
   - ok: 1763
   - insufficient_history: 520
   - illiquid_history: 36
Sources: yahoo=1763, -=556
Rows/stock: median 1481, min 1, max 1485
Staleness warnings: 0
Minimum weekly bars required: 104 (~520 trading days)

--- every non-ok status, grouped ---
status
illiquid_history        ABCOTS, ACEINTEG, AQYLON, ATLASCYCLE, BLUECOAS...
insufficient_history    3BBLACKBIO, AASTHA, ABANSENT, ABLBL, ABMKNO, A...

--- worst 10 by age of last bar ---
   Symbol               status  rows       last  stale_days                         issues
VASUPRADA insufficient_history    12 2026-09-10           3 only ~2 weekly bars (need 104)
  WHBRADY insufficient_history    16 2026-09-10           3 only ~3 weekly bars (need 104)
      SAB insufficient_history    13 2026-09-10           3 only ~2 weekly bars (need 104)
 KANCOTEA insufficient_history    18 2026-09-10           3 only ~3 weekly bars (need 104)
 ASSAMENT insufficient_history    12 2026

## 4 · Ranking NSE → the top 1 000

`/v7/finance/quote` (market cap, the obvious "top 1000" source) is aggressively blocked and
needs cookie+crumb gymnastics, so ranking is built on data we can actually get **for free**:

* **median daily rupee turnover** (from the bars we already downloaded) — primary key;
* **NSE index tier** (Nifty 50 / 100 / 200 / 500 / Midcap150 / Smallcap250, straight from
  `archives.nseindia.com`) — tie-break, and it is the *real* "top 1000" anchor;
* market cap if — and only if — `use_quotes=True` returned a *dense* column (the code refuses
  to rank on a patchy column, because then whichever 400 symbols happened to answer would win).

The point: the ranking never *drops* a symbol silently. `sc.ranking` keeps every ranked name
with its `Rank`; only the `top_n` cut feeds the detector.

In [ ]:
sc.rank_universe()
rk = sc.ranking
print(f"ranked {len(rk)} symbols · basis = {rk['rank_basis'].iloc[0]}")
_disp = rk.head(20)[["Rank", "Symbol", "Close", "TurnoverLakh", "IndexTier", "Industry"]]
try:
    display_if(_disp.style.background_gradient(subset=["TurnoverLakh"], cmap="viridis"))
except Exception:
    print(_disp.to_string(index=False))

· Ranked universe: 1000 symbols kept (basis=index_tier+turnover)
ranked 1000 symbols · basis = index_tier+turnover


,Rank,Symbol,Close,TurnoverLakh,IndexTier,Industry
0,1,HDFCBANK,708.250000,202689.477256,Nifty50,Financial Services
1,2,ICICIBANK,1379.300049,155935.071462,Nifty50,Financial Services
2,3,RELIANCE,1257.500000,136051.242770,Nifty50,Oil Gas & Consumable Fuels
3,4,BSE,3384.000000,117813.271729,Nifty200,Financial Services
4,5,BHARTIARTL,1831.099976,108709.147018,Nifty50,Telecommunication
5,6,INFY,1037.699951,102962.367097,Nifty50,Information Technology
6,7,SBIN,995.700012,91473.389725,Nifty50,Financial Services
7,8,ETERNAL,323.500000,83470.216785,Nifty50,Consumer Services
8,9,TCS,2200.800049,72569.510648,Nifty50,Information Technology
9,10,BAJFINANCE,1034.500000,71744.086786,Nifty50,Financial Services


## 5 · Run the screener

In [ ]:
_t0 = time.time()
sc.screen()          # detector over the ranked top-N. Re-run after editing §2: seconds.
hits = sc.summary_frame()
print(f"setups {len(hits)} · near-misses {0 if sc.near is None else len(sc.near)} · "
      f"universe screened {len(sc.diag)}")
if len(hits):
    style_results(hits)
else:
    print("No completed weekly sweep in the top-N this week - that is a normal, honest answer.")
    print("Widen the window: CONFIG['max_sweep_bars_ago']=4 then re-run §2 and §5,")
    print("and look at the near-misses below.")
print(f"screen took {time.time()-_t0:.1f}s")

· Screening 1000 symbols for completed weekly sweeps (swing_strength=2, lookback=26w, wick≥34%, close in upper 50%+)…
  → 32 setups, 155 near-misses in 23.5s
setups 32 · near-misses 155 · universe screened 1000
screen took 23.5s


In [ ]:
# the near-misses: one gate away, so you can see what the filter is rejecting
if sc.near is not None and len(sc.near):
    cols = [c for c in ["Symbol", "Company", "Close", "SweepType", "SweptSwingLow", "SweepLow",
                       "WickRatio%", "CloseInRange%", "ClosedAboveBy%", "TurnoverLakh",
                       "FailReason"] if c in sc.near.columns]
    print(f"{len(sc.near)} near-miss(es):")
    display_if(sc.near[cols].head(30))
else:
    print("no near-misses")

155 near-miss(es):


,Symbol,Close,SweepLow,WickRatio%,CloseInRange%,FailReason
0,RELIANCE,"1,257.50","1,253.00",6.30,6.30,"wick too small, body larger than wick, close w..."
1,TATASTEEL,183.00,180.79,23.30,23.30,"wick too small, body larger than wick, close w..."
2,ADANIENT,"3,060.00","2,918.00",9.80,63.40,"wick too small, body larger than wick"
3,ADANIPOWER,209.97,204.12,24.90,48.80,"wick too small, body larger than wick, close w..."
4,BEL,404.35,399.25,30.40,30.40,"wick too small, close weak / not in upper half"
5,GVT&D,"4,306.30","4,183.20",47.00,47.00,close weak / not in upper half
6,VEDL,263.70,256.50,37.40,37.40,"body larger than wick, close weak / not in upp..."
7,CGPOWER,891.00,870.00,30.80,45.70,"wick too small, close weak / not in upper half"
8,MUTHOOTFIN,"2,913.80","2,840.00",35.40,35.40,"body larger than wick, close weak / not in upp..."
9,ZEEL,91.61,86.90,33.00,33.00,"wick too small, body larger than wick, close w..."


### 5.1 · Every column, defined

| column | meaning |
|---|---|
| `SweepType` | `Old swing low` / `New (fresh) low` / `52w low` — which kind of liquidity was taken |
| `SweptSwingLow` | the level that was raided — the line your stop goes below |
| `SweepLow` / `SwingLowMade` | the week's low, and the week that low was originally made |
| `PoolsTaken` | how many swing lows one candle cleared (2+ = a genuine liquidity run) |
| `ClosedAboveBy%` | how far the close recovered above the swept level |
| `SweepDepth%` | how deep the raid went, vs the level |
| `WickRatio%` | lower wick ÷ full candle range (≥ 34 % required) |
| `CloseInRange%` | where in the range the close sat (≥ 50 % = closed near the high) |
| `VolRatio` | sweep-candle volume ÷ 20-week median — the participation check |
| `Risk%` | entry (next week's open proxy: the sweep close) → `Stop` |
| `Target2R/3R` | mechanical exits at 2 and 3 times risk |
| `Resistance26w` | nearest supply above; if `UpsideToRes%` < 2 R, the trade is crowded |
| `Score` | 0–100 quality blend of the drivers above (see `S_*` columns for each part) |

## 6 · Charts — verify the setup with your own eyes

Each chart marks the swept level, the sweep candle, prior swing lows, the 20-week mean and
the suggested stop/target. If the picture does not match the numbers, trust the picture.

In [ ]:
def show(symbols=None, n=8, weeks=70, engine="plotly"):
    """Plot the top n setups (or the symbols you name). engine: plotly | matplotlib"""
    if symbols:
        syms = list(symbols)
    else:
        syms = list(hits["Symbol"].head(n)) if ("hits" in dir() and hits is not None and len(hits)) else []
        if not syms and sc.hits is not None and len(sc.hits):
            syms = list(sc.hits["Symbol"].head(n))
        if not syms and sc.ranking is not None and len(sc.ranking):
            syms = list(sc.ranking["Symbol"].head(3))     # show structure even with no setups
            print("(no setups yet - plotting ranked symbols for context)")
    if not syms:
        print("nothing to plot this week"); return
    for s in syms:
        print("\n" + "=" * 96); print(s)
        try:
            fig = plot_setup(sc, s, weeks=weeks, engine=engine)
            if engine == "plotly":
                fig.show()
            else:
                display_if(fig)
        except Exception as exc:
            print("  plot failed:", type(exc).__name__, exc)

show(n=6)


INDIGO


<!doctype html>


LLOYDSENGG


<!doctype html>


MASFIN


...

<!doctype html>


POWERINDIA


<!doctype html>


IBULLSLTD


<!doctype html>

In [ ]:
# a saved, self-contained HTML gallery (charts embedded) - great for sharing
_gal_syms = list(hits["Symbol"].head(12)) if ("hits" in dir() and hits is not None and len(hits)) else     (list(sc.ranking["Symbol"].head(6)) if sc.ranking is not None and len(sc.ranking) else [])
if _gal_syms:
    p = SC.save_html_gallery(sc, _gal_syms, os.path.join(OUT_DIR, "sweep_gallery.html"), weeks=70)
    print("gallery ->", p, "(charts for", len(_gal_syms), "symbol(s))")
else:
    print("nothing to gallery-ise yet")

gallery -> /content/out/sweep_gallery.html (charts for 12 symbol(s))


## 7 · “Why is *this stock* not on the list?”

The single most useful cell for trusting the output. It walks the gates for one specific
ticker and names the exact one it failed — including the difference between
*“Yahoo has no bars”*, *“429 throttled — re-run”* and *“history too short”*.

In [ ]:
CHECK = ["RELIANCE", "TCS", "HDFCBANK", "TATAMOTORS"]   # @param {type:"raw"}
pd.DataFrame(sc.diagnose([s.strip() for s in CHECK if s.strip()]))

,Symbol,weekly_bars,last_week,close,turnover_lakh,locked_weeks_26,last_low,last_close,swing_lows_in_range,wick_ratio,close_in_range,verdict,status
0,RELIANCE,314,2026-09-07,"1,257.50","136,051.20",0.00,"1,253.00","1,257.50","1290@2026-04-06, 1253.2@2026-06-08, 1249.8@202...",0.06,0.06,"near miss: wick too small, body larger than wi...",NaN
1,TCS,314,2026-09-07,"2,200.80","72,569.50",0.00,"2,185.50","2,200.80","2346.2@2026-03-30, 2206.4@2026-05-11, 1976.8@2...",0.13,0.13,last week did not undercut any swing low,NaN
2,HDFCBANK,314,2026-09-07,708.25,"202,689.50",0.00,681.90,708.25,"726.65@2026-03-30, 732.3@2026-06-08, 681.9@202...",0.85,0.85,"SWEEP: took 698.5, closed 1.40% above, score 75",NaN
3,TATAMOTORS,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no usable bars - not fetched,not fetched


In [ ]:
# same question, answered for the whole universe at once
d = sc.diag
print("verdict mix over", len(d), "screened symbols:\n")
print(d["status"].value_counts().to_string())
if "FailReason" in d.columns:
    print("\nmost common single reason a symbol had NO setup:")
    print(d["status"].value_counts().head(8).to_string())

verdict mix over 1000 screened symbols:

status
no_setup           810
near_miss          155
hit                 32
circuit_locked       2
price_below_min      1


## 8 · Save the output

CSV + a 6-sheet workbook (`Setups`, `NearMisses`, `ScreenedUniverse`, `DataQuality`, `Params`,
`UniverseRanking`), plus the parameter snapshot as JSON so a result is always reproducible.

In [ ]:
paths = sc.export("nse")
for k, v in paths.items():
    print(f"{k:16s} {v}  ({os.path.getsize(v):,} B)")
if CONFIG["mount_drive"]:
    SC.snapshot_workspace(OUT_DIR, os.path.expanduser("~/../content/Drive/NSE_Screener/"
                                                     + _dt.date.today().isoformat()))
# download straight from the notebook (no Drive needed):
try:
    from google.colab import files
    files.download(paths["results_xlsx"])
except Exception as e:
    print("(auto-download unavailable here:", type(e).__name__ + ") — grab the file from the left panel)")

· Wrote nse_results_2026-09-13.csv, nse_results_2026-09-13.xlsx, nse_near_misses_2026-09-13.csv, nse_diagnostics_2026-09-13.csv, nse_params_2026-09-13.json
results_csv      /content/out/nse_results_2026-09-13.csv  (7,272 B)
results_xlsx     /content/out/nse_results_2026-09-13.xlsx  (279,006 B)
near_miss_csv    /content/out/nse_near_misses_2026-09-13.csv  (16,263 B)
diagnostics_csv  /content/out/nse_diagnostics_2026-09-13.csv  (48,664 B)
params_json      /content/out/nse_params_2026-09-13.json  (1,635 B)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 9 · Optional: prove the rule, backtest it, tune it

In [ ]:
# 9.1 · the detector is unit-tested on synthetic tapes - no network, " + str(_NT) + " checks
st = sc.selftest()
print(f"{(st['result']=='PASS').sum()}/{len(st)} passed")
fails = st[st["result"] != "PASS"]
if len(fails):
    print("FAILURES:\n", fails.to_string(index=False))
st.head(50)

· Self-test: 49/49 checks passed
49/49 passed


,check,result,detail
0,test_52w_flag_only_when_really_the_52w_low,PASS,
1,test_a_bar_must_not_sweep_the_low_it_is_printing,PASS,
2,test_assess_daily_repairs_inconsistent_bars_in...,PASS,
3,test_backtest_and_summary,PASS,
4,test_backtest_exit_priority_is_stop_before_tar...,PASS,
5,test_body_dominating_the_wick_is_rejected,PASS,
6,test_cache_path_is_collision_free_for_special_...,PASS,
7,test_chart_frames_shapes,PASS,
8,test_deep_collapse_through_the_level_is_rejected,PASS,
9,test_detects_classic_sweep_of_old_swing_low,PASS,


In [ ]:
# 9.2 · backtest the identical rule on the cached history (entry = next week's open)
trades, summary = sc.backtest_rule(horizon_weeks=8, r_mult=2.0, max_symbols=None)
print(json.dumps(summary, indent=1, default=str))
if trades is not None and len(trades):
    display_if(trades.head(15))
print("\nThis is an *un-optimised, single-parameter* read of the rule, on 6y of bars,")
print("with 0.15% round-trip cost and no slippage model. Read the shape, not the decimal.")

· Backtesting the same rule: 8w horizon, 2.0R target…
  → {"trades": 19427, "win_rate_%": 36.5, "avg_return_%": 0.3, "median_return_%": -3.83, "avg_R": 0.06, "hit_target_%": 32.7, "hit_stop_%": 61.0, "expired_%": 6.2, "profit_factor": 1.08, "p5_%": -9.56, "p95_%": 17.07}
{
 "trades": 19427,
 "win_rate_%": 36.5,
 "avg_return_%": 0.3,
 "median_return_%": -3.83,
 "avg_R": 0.06,
 "hit_target_%": 32.7,
 "hit_stop_%": 61.0,
 "expired_%": 6.2,
 "profit_factor": 1.08,
 "p5_%": -9.56,
 "p95_%": 17.07
}


,Symbol,SweepWeek,SweptLevel,Entry,Stop,Exit,Outcome,Return%,R,BarsToExit
0,20MICRONS,2021-10-25,58.15,59.45,55.22,67.91,target,14.07,2.00,8
1,20MICRONS,2021-11-01,58.15,60.95,57.31,57.31,stop,-6.12,-1.00,8
2,20MICRONS,2021-11-29,58.15,58.75,55.72,64.81,target,10.16,2.00,8
3,20MICRONS,2023-04-17,80.80,83.00,80.05,80.05,stop,-3.71,-1.00,8
4,20MICRONS,2023-04-24,80.80,83.65,78.80,93.34,target,11.44,2.00,8
5,20MICRONS,2025-06-02,225.70,229.90,213.97,261.75,target,13.70,2.00,8
6,20MICRONS,2025-09-29,215.05,217.18,203.48,203.48,stop,-6.46,-1.00,8
7,20MICRONS,2025-11-24,189.56,192.04,182.69,182.69,stop,-5.02,-1.00,8
8,20MICRONS,2025-12-08,183.61,190.02,181.69,206.69,target,8.62,2.00,8
9,20MICRONS,2026-01-26,172.11,175.00,166.21,166.21,stop,-5.17,-1.00,8



This is an *un-optimised, single-parameter* read of the rule, on 6y of bars,
with 0.15% round-trip cost and no slippage model. Read the shape, not the decimal.


In [ ]:
# 9.3 · parameter sensitivity - does a result survive small changes, or is it a curve-fit?
grid = {
    "min_wick_ratio":    [0.25, 0.34, 0.45],
    "min_close_above_pct": [0.0, 0.002, 0.008],
    "max_sweep_bars_ago": [1, 2],
}
try:
    gs = sc.grid_scan(grid)
    display_if(gs.head(30))
except Exception as exc:
    print("grid scan skipped:", type(exc).__name__, exc)

,min_wick_ratio,min_close_above_pct,max_sweep_bars_ago,trades,win_rate_%,avg_return_%,avg_R,profit_factor
0,0.45,0.01,1,2156,37.50,0.52,0.09,1.14
1,0.45,0.01,2,2156,37.50,0.52,0.09,1.14
2,0.45,0.00,2,2419,37.20,0.49,0.08,1.13
3,0.45,0.00,1,2419,37.20,0.49,0.08,1.13
4,0.34,0.01,1,2426,37.20,0.48,0.08,1.13
5,0.34,0.01,2,2426,37.20,0.48,0.08,1.13
6,0.45,0.00,1,2486,37.10,0.48,0.08,1.13
7,0.45,0.00,2,2486,37.10,0.48,0.08,1.13
8,0.25,0.01,1,2467,37.00,0.46,0.08,1.12
9,0.25,0.01,2,2467,37.00,0.46,0.08,1.12


---
## 10 · Data-troubleshooting playbook (the part everyone gets wrong)

| symptom | what is actually happening | what to do |
|---|---|---|
| `429 Too Many Requests`, mass `throttled` | you (or another notebook on this IP) hammered Yahoo | re-run §3 — the cache means only the missing symbols are fetched. If it persists: `CONFIG["max_workers"]=1`, `request_sleep=0.35`, `transport="curl_cffi"` |
| every symbol `no_data` | Colab egress or a Yahoo-side block on your IP | `transport="yfinance"` (it has its own cookie/crumb handling), or run §3 on your laptop and upload `cache_nse/` |
| `HTTP 401` on quotes | crumb/cookie expired | leave `use_quotes=False` — ranking does not need it |
| `universe: ... failed` | NSE blocks the sandbox UA | download `EQUITY_L.csv` from `archives.nseindia.com/content/equities/` and upload it as `cache_nse/universe_equity_l.csv` |
| `TATAMOTORS` (and similar) missing | post-demerger the ticker has **no bars on Yahoo**; it is not a bug and not silently dropped — §7 says so | search the successor tickers in your own universe |
| “only ~N weekly bars” | listing is younger than `min_weekly_bars` | lower it (e.g. 52) *and* accept that swing structure is thin on such names |
| results change every re-run | you are screening a live (in-progress) week | the code drops the partial week by design; run after Friday 15:30 IST for a stable answer |

**Self-diagnosis you can run any time** — this answers “is my *data* sound?”, independent of the strategy:

In [ ]:
def audit(sc):
    """Structural integrity checks on every cached series. Cheap; run it whenever results look odd."""
    rows = []
    for sym, df in sc.data.items():
        d = pd.to_datetime(df.index)
        span = (d.max() - d.min()).days
        rows.append({
            "Symbol": sym, "rows": len(df),
            "dup_dates": int(d.duplicated().sum()),
            "unsorted": int((d.to_numpy()[1:] < d.to_numpy()[:-1]).sum()),
            "ohlc_violations": int(((df["High"] < df[["Open", "Close"]].max(axis=1)) |
                                    (df["Low"]  > df[["Open", "Close"]].min(axis=1))).sum()),
            "nonpositive": int((df[["Open", "High", "Low", "Close"]] <= 0).any(axis=1).sum()),
            "zero_vol_days": int((df["Volume"] == 0).sum()),
            "big_gaps_gt_10d": int((d.to_series().diff().dt.days > 10).sum()),
            "span_days": int(span),
            "implied_weeks": int(round(span / 7.0)),
        })
    a = pd.DataFrame(rows)
    if a.empty:
        print("nothing cached yet"); return a
    print(f"audited {len(a)} series | dup dates {int(a.dup_dates.sum())} | unsorted {int(a.unsorted.sum())} "
          f"| OHLC violations {int(a.ohlc_violations.sum())} | non-positive {int(a.nonpositive.sum())}")
    worst = a.nlargest(8, "ohlc_violations")
    if worst["ohlc_violations"].max() > 0:
        print("\nsymbols with OHLC inconsistencies (suspicious feeds - inspect before trusting):")
        print(worst.to_string(index=False))
    return a

audit(sc)

audited 1763 series | dup dates 0 | unsorted 0 | OHLC violations 0 | non-positive 0


,Symbol,rows,dup_dates,unsorted,ohlc_violations,nonpositive,zero_vol_days,big_gaps_gt_10d,span_days,implied_weeks
0,20MICRONS,1483,0,0,0,0,0,0,2191,313
1,21STCENMGM,1480,0,0,0,0,0,0,2191,313
2,360ONE,1482,0,0,0,0,0,0,2191,313
3,3MINDIA,1483,0,0,0,0,1,0,2191,313
4,3PLAND,1465,0,0,0,0,0,1,2191,313
...,...,...,...,...,...,...,...,...,...,...
1758,ZOTA,1483,0,0,0,0,0,0,2191,313
1759,ZYDUSLIFE,1480,0,0,0,0,0,0,2191,313
1760,ZUARIIND,1482,0,0,0,0,0,0,2191,313
1761,ZUARI,1484,0,0,0,0,0,0,2191,313


## 11 · Weekly automation (Colab-only niceties)

* **Drive persistence** — turn `CONFIG["mount_drive"]=True` in §2. The cache survives runtime
  resets, so you never re-download 2 300 symbols (that is what gets you rate-limited).
* **Headless re-runs** — `File → Save a copy in GitHub` then use a scheduled Colab; or run
  `jupyter nbconvert --to script --execute` on a box.
* **Telegram / webhook alert** — paste token + chat id below; only the top-N setups are sent.

In [ ]:
# optional push notification (leave blank to skip)
BOT_TOKEN = ""   # @param {type:"string"}
CHAT_ID   = ""   # @param {type:"string"}
print(sc.telegram_alert(BOT_TOKEN, CHAT_ID, top_n=10) if BOT_TOKEN else "telegram skipped")
print("\n" + SC.format_message(hits, sc.finished_at or ""))

telegram skipped

NSE weekly sweep screen — 2026-09-13 21:41 IST
32 setup(s); top 12:

 1. INDIGO              4973.0  swept    4886.0 → low    4822.0 (+1.78%)  wick 74%  score 81  stop   4797.89  2R   5323.22
 2. LLOYDSENGG           82.76  swept      81.0 → low      78.5 (+2.17%)  wick 84%  score 77  stop     78.11  2R     92.07
 3. MASFIN              295.95  swept     290.1 → low     282.0 (+2.02%)  wick 67%  score 76  stop    280.59  2R    326.67
 4. HDFCBANK            708.25  swept     698.5 → low     681.9 (+1.40%)  wick 85%  score 75  stop    678.49  2R    767.77
 5. POWERINDIA         31435.0  swept   31195.0 → low   30550.0 (+0.77%)  wick 85%  score 73  stop  30397.25  2R   33510.5
 6. IBULLSLTD            26.14  swept      25.4 → low     24.29 (+2.91%)  wick 47%  score 72  stop     24.17  2R     30.08
 7. ANGELONE             305.0  swept     290.8 → low    289.05 (+4.88%)  wick 80%  score 71  stop     287.6  2R    339.79
 8. INDUSTOWER           388.0  swept    375.05 → lo

---
## 12 · Method — what “complete weekly sweep” means here, precisely

**Weekly candle.** Daily bars are resampled to ISO weeks (Mon–Fri), stamped with the Monday.
The currently running week is *excluded* unless it is genuinely closed (holiday-shortened
weeks still count; the code checks for a Friday or a sufficiently elapsed date, so a
3-day week is not silently treated as a fresh sweep).

**Swing low (confirmed).** A bar whose low is the lowest of a ±`swing_strength` window, aged at
least `min_swing_age` closed weeks. Only *confirmed* swings are used as levels — a low that has
not yet been defended is not liquidity, it is just the current price.

**Liquidity pools available to sweep** = confirmed swings within `swing_lookback` weeks
∪ the rolling window low ∪ the 52-week low. The latter two are what make **new** swing lows
sweepable (your requirement); they can be switched off with `allow_fresh_low=False` /
`require_old_swing=True`.

**The sweep itself.** On the last closed weekly candle:
`Low < level·(1−min_depth_pct)` (it actually raided the stops below the level) **and**
`Close > level·(1+min_close_above_pct)` (it *reclaimed* — the raid failed). Depth is capped at
`max_depth_pct` and `max_depth_atr_mult·ATR(14w)` so that a genuine breakdown or a circuit-day
collapse is not mistaken for a sweep.

**The wick.** `lower_wick = min(Open, Close) − Low`, required to be ≥ `min_wick_ratio` of the
candle range **and** ≥ `min_wick_body_mult` × the body, with the close in the upper
`min_close_in_range` of the range and the total range ≤ `max_range_of_close` × close.
That combination is what separates “banks bought the low” from “a doji happened to print below
a level”.

**Which level gets reported.** If one candle clears several pools, the **lowest** undercut level
is reported (`PoolsTaken` counts how many) — that is the true extent of the raid.

**Trading plan attached to each row** — stop `= SweepLow × 0.995` (below the raid, not below the
level), risk from the sweep close, targets at 2 R and 3 R, plus the nearest 26-week resistance
so you can see whether there is room.

---
### Footer

Generated as a single self-contained Colab notebook · engines embedded above are the exact
code that ran, unit-tested with the 49 checks in §9.1 · data: Yahoo Finance (bars) +
NSE archives (universe, index tiers). **Educational tooling only — no advice, no warranty.**
If you change a parameter, note it in the JSON snapshot §8 writes, so tomorrow’s list is
comparable to today’s.